# DeMine-VN — Bản đồ nguy cơ và thứ tự ưu tiên rà phá bom mìn

**Cuộc thi Sáng tạo trẻ Quốc gia trong lĩnh vực Trí tuệ nhân tạo năm 2026 — Bảng C**
Chủ đề: AI cho Phát triển kinh tế – xã hội

---

## Vấn đề

Gần **6,1 triệu héc ta đất Việt Nam bị ô nhiễm hoặc nghi ngờ ô nhiễm bom mìn, chiếm
18,71% diện tích cả nước**. Cả 63 trên 63 tỉnh thành đều có; 9.116 xã phường vẫn còn
bom mìn trong lòng đất. Từ sau năm 1975 đến nay, **hơn 40.000 người đã chết và
60.000 người bị thương**. Với năng lực rà phá hiện tại, việc làm sạch cần thêm
**hàng trăm năm**.

Gần một phần năm đất đai quốc gia không dám canh tác, không dám xây, không dám mở
đường. Đó là rào cản phát triển kinh tế – xã hội ở quy mô quốc gia, và những huyện ô
nhiễm nặng nhất cũng chính là những huyện nghèo nhất.

## Vai trò của trí tuệ nhân tạo trong đề tài

Hệ thống **không có nhiệm vụ tìm ra vật nổ** — đó là việc của thiết bị dò và của con
người tại thực địa. Nhiệm vụ của hệ thống là **xếp thứ tự ưu tiên**: chỉ ra trong
hàng triệu héc ta nghi ngờ, những khoảnh đất nào nên rà phá trước. Nếu xác định được
phần diện tích nhỏ chứa phần lớn nguy cơ thì cùng một ngân sách và cùng số đội rà
phá, số vật nổ được xử lý sớm sẽ tăng lên nhiều lần.

## Kiến trúc bốn tầng

| Tầng | Chức năng | Đầu ra |
|---|---|---|
| Một | Chuẩn hoá hồ sơ không kích giải mật | Lưới ô mang tải trọng bom, mật độ phi vụ |
| Hai | Phát hiện hố bom trên ảnh vệ tinh lịch sử | Toạ độ và đường kính từng hố bom |
| Ba | Mô hình nguy cơ hợp nhất | Xác suất còn vật nổ theo ô, đã hiệu chỉnh |
| Bốn | Ưu tiên và trình bày | Danh mục rà phá và bản đồ nguy cơ |

## Về dữ liệu dùng trong notebook này

Notebook chạy trên **bộ dữ liệu mô phỏng do đội thi tự xây dựng**. Lý do không phải
vì thiếu dữ liệu thật — hồ sơ không kích giải mật và ảnh vệ tinh giải mật đều công
khai — mà vì một lý do sâu hơn: **dữ liệu thật không có nhãn đối chứng**. Ngoài thực
địa không ai biết chắc dưới một thửa ruộng chưa rà phá có bom hay không, nên không
thể đo được mô hình đúng bao nhiêu phần trăm.

Bộ mô phỏng tái hiện đúng chuỗi nhân quả vật lý và cung cấp nhãn đối chứng chính xác,
nhờ đó toàn bộ khung kiểm chứng bốn tầng chạy và đo được ngay. Khi có dữ liệu thật,
chỉ cần thay tầng dữ liệu theo hướng dẫn ở `docs/ADAPT_NEW_DATA.md`; các tầng còn lại
không phải sửa.

> **Nguyên tắc an toàn bắt buộc.** Hệ thống chỉ xếp thứ tự ưu tiên rà phá. Hệ thống
> **không bao giờ** tuyên bố một khu đất là an toàn. Mọi khu đất, kể cả khi được chấm
> mức nguy cơ thấp nhất, vẫn phải rà phá đầy đủ theo quy trình kỹ thuật hiện hành
> trước khi đưa vào sử dụng.

---

## Cách chạy

Chọn **Run All**. Các ô chạy tuần tự từ đầu đến cuối.

**Thiết lập cần chọn trong bảng Session options ở bên phải:**

| Thiết lập | Giá trị |
|---|---|
| Accelerator | **GPU T4 × 2** |
| Internet | Bật nếu được (không bắt buộc) |
| Persistence | Không cần |

Nếu tắt Internet, hệ thống tự chuyển sang phương án phát hiện dự phòng bằng
Torchvision, vốn luôn có sẵn. Toàn bộ quy trình vẫn chạy trọn vẹn.


## Bước 0 — Ghi mã nguồn ra đĩa

In [ ]:
import json, os, sys, pathlib

# Toàn bộ mã nguồn của dự án được nhúng ngay trong notebook này và ghi ra
# đĩa khi ô lệnh chạy. Nhờ vậy chỉ cần tải lên một tệp duy nhất.
_FILES = json.loads(r'''{"src/demine/__init__.py": "\"\"\"DeMine-VN — Hệ thống lập bản đồ nguy cơ và xếp thứ tự ưu tiên rà phá bom mìn.\n\nHợp nhất hồ sơ không kích giải mật, dấu vết hố bom trên ảnh vệ tinh lịch sử và dữ\nliệu rà phá thực địa để ước lượng xác suất còn tồn tại vật nổ theo từng ô lưới,\nphục vụ công tác xếp thứ tự ưu tiên rà phá.\n\nHệ thống chỉ xếp thứ tự ưu tiên. Hệ thống không bao giờ tuyên bố một khu đất là an\ntoàn. Mọi khu đất, kể cả khi được chấm mức nguy cơ thấp nhất, vẫn phải rà phá đầy\nđủ theo đúng quy trình kỹ thuật hiện hành trước khi đưa vào sử dụng.\n\"\"\"\n\n__version__ = \"1.0.0\"\n\nSAFETY_NOTICE = (\n    \"Hệ thống chỉ xếp thứ tự ưu tiên rà phá. Hệ thống KHÔNG xác nhận bất kỳ khu \"\n    \"đất nào là an toàn. Mọi khu đất vẫn phải được rà phá đầy đủ theo quy trình \"\n    \"kỹ thuật hiện hành trước khi đưa vào sử dụng.\"\n)\n", "src/demine/config.py": "\"\"\"Tham số cấu hình toàn hệ thống.\n\nMọi hằng số có ý nghĩa vật lý đều được nêu rõ căn cứ lựa chọn ngay tại chỗ khai\nbáo, để người đọc kiểm chứng được thay vì phải tin.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple\n\n\n@dataclass\nclass GridConfig:\n    \"\"\"Lưới ô vuông dùng chung cho toàn bộ hệ thống.\"\"\"\n\n    # Cạnh ô lưới, mét. Chọn 100 m vì đây là bậc kích thước của một khoảnh rà phá\n    # trong thực tế, đồng thời đủ lớn để thống kê ổn định trên từng ô.\n    cell_size_m: float = 100.0\n\n    # Kích thước vùng nghiên cứu theo số ô.\n    n_cells_x: int = 220\n    n_cells_y: int = 180\n\n    # Toạ độ gốc quy ước của vùng nghiên cứu (kinh độ, vĩ độ).\n    origin_lon: float = 106.85\n    origin_lat: float = 16.72\n\n\n@dataclass\nclass SortieConfig:\n    \"\"\"Mô phỏng hồ sơ không kích theo cấu trúc của bộ dữ liệu THOR.\"\"\"\n\n    n_missions: int = 1800\n\n    # Số quả bom mỗi phi vụ. Dải giá trị phản ánh tải trọng thực tế của các loại\n    # máy bay cường kích sử dụng trong giai đoạn được mô phỏng.\n    bombs_per_mission: Tuple[int, int] = (6, 18)\n\n    # Tỉ trọng của thành phần rải đều trong phân bố điểm ngắm. Phần còn lại bám\n    # theo tuyến giao thông, sông ngòi và khu dân cư. Giá trị nhỏ vì không kích\n    # trong chiến tranh tập trung rất mạnh vào các mục tiêu vận tải, và chính sự\n    # tập trung đó tạo ra những hành lang ô nhiễm đậm đặc ngoài thực địa.\n    uniform_target_fraction: float = 0.06\n\n    # Độ tản mát của điểm rơi quanh điểm ngắm, mét. Đây là sai số kỹ thuật của\n    # phương thức ném bom không dẫn đường.\n    impact_dispersion_m: float = 95.0\n\n    # Sai số định vị của chính hồ sơ ghi chép, mét. Toạ độ trong hồ sơ thời chiến\n    # được ghi theo lưới bản đồ quân sự và làm tròn, nên lệch đáng kể so với điểm\n    # rơi thật. Đây là nguồn bất định lớn nhất của tầng một.\n    record_position_error_m: float = 180.0\n\n    # Tỉ lệ phi vụ hoàn toàn không có trong hồ sơ, do mất mát tư liệu.\n    missing_record_fraction: float = 0.08\n\n\n@dataclass\nclass OrdnanceConfig:\n    \"\"\"Đặc tính vật lý của bom đạn, chi phối việc còn sót lại vật nổ.\"\"\"\n\n    # Tỉ lệ bom không nổ ở điều kiện nền chuẩn. Các tài liệu kỹ thuật quân sự ghi\n    # nhận tỉ lệ này ở mức khoảng một phần mười đối với bom thông thường.\n    base_dud_rate: float = 0.10\n\n    # Hệ số nhân tỉ lệ không nổ trên nền đất mềm và ngập nước. Bom cắm sâu vào nền\n    # mềm thường không kích nổ, đây là cơ chế vật lý then chốt của bài toán.\n    soft_soil_dud_multiplier: float = 2.4\n\n    # Xác suất một quả bom đã nổ để lại hố quan sát được trên ảnh, theo nền cứng.\n    crater_visible_rate_hard: float = 0.92\n\n    # Trên nền mềm, hố bom bị bồi lấp nhanh nên khó quan sát hơn nhiều.\n    crater_visible_rate_soft: float = 0.45\n\n    # Đường kính hố bom, mét.\n    crater_diameter_m: Tuple[float, float] = (9.0, 22.0)\n\n\n@dataclass\nclass ImageryConfig:\n    \"\"\"Ảnh vệ tinh trinh sát lịch sử mô phỏng.\"\"\"\n\n    # Kích thước điểm ảnh trên mặt đất, mét. Tương ứng độ phân giải của máy chụp\n    # toàn cảnh trên vệ tinh trinh sát thế hệ được sử dụng, sau khi quét phim.\n    ground_sample_distance_m: float = 1.2\n\n    tile_size_px: int = 640\n\n    # Số ảnh con. Con số này quyết định tỉ lệ diện tích vùng nghiên cứu thực sự có\n    # ảnh vệ tinh phủ tới. Mỗi ảnh con phủ khoảng 0,59 km2, nên để phủ được phần lớn\n    # một vùng rộng vài trăm km2 thì cần vài trăm ảnh. Độ phủ quá thấp sẽ khiến nhóm\n    # đặc trưng quan sát khuyết ở hầu hết các ô và mất tác dụng.\n    n_tiles: int = 420\n\n    # Cường độ nhiễu hạt phim. Ảnh trinh sát là ảnh phim quét lại nên có hạt rõ.\n    film_grain_sigma: float = 0.055\n\n    # Độ lệch chuẩn của trường độ sáng nền, mô phỏng chiếu sáng không đều và\n    # chênh lệch mật độ phim giữa các vùng của tấm ảnh.\n    illumination_sigma: float = 0.13\n\n    train_fraction: float = 0.70\n    val_fraction: float = 0.15\n\n\n@dataclass\nclass TerrainConfig:\n    \"\"\"Địa hình, thổ nhưỡng và hiện trạng sử dụng đất.\"\"\"\n\n    n_villages: int = 14\n    n_roads: int = 5\n    n_rivers: int = 2\n\n    # Tỉ lệ diện tích là đất canh tác. Đây là nơi con người tiếp xúc nhiều nhất\n    # với lòng đất, nên cũng là nơi nguy cơ tai nạn cao nhất.\n    farmland_fraction: float = 0.42\n\n\n@dataclass\nclass ClearanceConfig:\n    \"\"\"Mô phỏng hoạt động khảo sát và rà phá đã thực hiện.\"\"\"\n\n    # Tỉ lệ diện tích vùng nghiên cứu đã được rà phá. Con số nhỏ phản ánh đúng\n    # thực tế: phần đã làm sạch chỉ chiếm phần rất nhỏ của diện tích ô nhiễm.\n    cleared_area_fraction: float = 0.11\n\n    # Trọng số của các yếu tố chi phối việc chọn khoảnh đất đưa vào rà phá. Đây\n    # chính là nguồn thiên lệch chọn mẫu mà tầng ba phải hiệu chỉnh.\n    selection_weight_tonnage: float = 1.0\n    selection_weight_near_village: float = 1.6\n    selection_weight_near_road: float = 0.9\n\n    # Hiệu suất phát hiện của công tác rà phá thực địa. Không tuyệt đối, nhưng rất\n    # cao, nên vẫn dùng làm nhãn đối chứng được.\n    detection_efficiency: float = 0.96\n\n\n@dataclass\nclass AccidentConfig:\n    \"\"\"Hồ sơ tai nạn bom mìn, dùng làm tập kiểm chứng độc lập.\"\"\"\n\n    n_years: int = 24\n\n    # Xác suất một vật nổ còn sót gây tai nạn trong một năm, tính trên mỗi đơn vị\n    # mức độ phơi nhiễm. Giá trị nhỏ vì phần lớn vật nổ không bao giờ bị chạm tới.\n    annual_incident_rate: float = 0.0115\n\n    # Mốc thời gian chia tập kiểm chứng theo thời gian. Mô hình chỉ học dữ liệu\n    # trước mốc này và được đánh giá bằng tai nạn xảy ra sau mốc.\n    temporal_split_year: int = 17\n\n\n@dataclass\nclass DetectConfig:\n    \"\"\"Tầng hai — phát hiện hố bom trên ảnh.\"\"\"\n\n    backend: str = \"auto\"\n    model_spec: str = \"yolo11n.pt\"\n    epochs: int = 40\n    batch_size: int = 16\n    image_size: int = 640\n    confidence_threshold: float = 0.25\n    iou_threshold: float = 0.50\n    devices: List[int] = field(default_factory=lambda: [0, 1])\n    workers: int = 2\n    seed: int = 20260914\n\n\n@dataclass\nclass RiskConfig:\n    \"\"\"Tầng ba — mô hình nguy cơ.\"\"\"\n\n    # Bán kính lan toả của hạt nhân quy chiếu hồ sơ không kích về lưới, mét. Bằng\n    # đúng sai số định vị của hồ sơ; đây là cách mô hình hoá tường minh bất định\n    # thay vì coi mỗi bản ghi là một điểm chính xác.\n    record_kernel_radius_m: float = 180.0\n\n    # Số khối không gian dùng để chia tập. Chia theo khối chứ không theo điểm là\n    # bắt buộc, vì phân bố vật nổ có tương quan không gian rất mạnh.\n    n_spatial_blocks: int = 24\n    n_cv_folds: int = 4\n\n    learning_rate: float = 0.06\n    max_iter: int = 400\n    max_leaf_nodes: int = 31\n    min_samples_leaf: int = 25\n    l2_regularization: float = 1.0\n\n    # Bật hiệu chỉnh thiên lệch chọn mẫu bằng trọng số nghịch đảo xác suất được\n    # chọn đưa vào rà phá.\n    use_propensity_weighting: bool = True\n\n    # Bật hiệu chỉnh theo khung học từ dữ liệu chỉ có quan sát dương.\n    use_pu_correction: bool = True\n\n    # Ba hằng số dưới đây phải khớp với OrdnanceConfig. Chúng được lặp lại ở đây vì\n    # tầng ba dùng chúng để dựng đặc trưng vật lý, và trong triển khai thật thì\n    # chúng đến từ tài liệu kỹ thuật quân sự chứ không đến từ bộ mô phỏng.\n    dud_softness_gain: float = 2.4\n    crater_visibility_hard: float = 0.92\n    crater_visibility_soft: float = 0.45\n\n    n_calibration_bins: int = 12\n    seed: int = 20260914\n\n\n@dataclass\nclass RunConfig:\n    \"\"\"Cấu hình tổng của một lần chạy.\"\"\"\n\n    grid: GridConfig = field(default_factory=GridConfig)\n    sortie: SortieConfig = field(default_factory=SortieConfig)\n    ordnance: OrdnanceConfig = field(default_factory=OrdnanceConfig)\n    imagery: ImageryConfig = field(default_factory=ImageryConfig)\n    terrain: TerrainConfig = field(default_factory=TerrainConfig)\n    clearance: ClearanceConfig = field(default_factory=ClearanceConfig)\n    accident: AccidentConfig = field(default_factory=AccidentConfig)\n    detect: DetectConfig = field(default_factory=DetectConfig)\n    risk: RiskConfig = field(default_factory=RiskConfig)\n\n    output_dir: str = \"outputs\"\n    data_dir: str = \"data\"\n    seed: int = 20260914\n    quick: bool = False\n    multi_gpu: bool = True\n\n    def apply_quick_mode(self) -> \"RunConfig\":\n        \"\"\"Rút gọn khối lượng tính toán để chạy thử nhanh.\"\"\"\n        self.sortie.n_missions = 420\n        self.imagery.n_tiles = 120\n        self.grid.n_cells_x = 120\n        self.grid.n_cells_y = 100\n        self.detect.epochs = 6\n        self.risk.max_iter = 120\n        self.risk.n_spatial_blocks = 12\n        self.risk.n_cv_folds = 3\n        self.quick = True\n        return self\n", "src/demine/pipeline.py": "\"\"\"Điều phối toàn bộ quy trình bốn tầng, từ dữ liệu đến bản đồ ưu tiên.\n\nTrình tự chạy:\n\n  1. Dựng vùng nghiên cứu: địa hình, thổ nhưỡng, hiện trạng sử dụng đất.\n  2. Mô phỏng phi vụ không kích, điểm rơi, vật nổ còn sót và hồ sơ ghi chép.\n  3. Kết xuất ảnh vệ tinh lịch sử và ghi bộ dữ liệu theo quy ước YOLO.\n  4. Huấn luyện và đánh giá mô hình phát hiện hố bom  (tầng hai).\n  5. Quy hố bom phát hiện được về lưới, dựng bảng đặc trưng bốn nhóm.\n  6. Mô phỏng hoạt động rà phá đã thực hiện và hồ sơ tai nạn.\n  7. Huấn luyện mô hình nguy cơ trên phần đất đã rà phá  (tầng ba).\n  8. Chạy đủ bốn tầng kiểm chứng.\n  9. Sinh bản đồ, hình minh hoạ, danh mục ưu tiên và báo cáo tổng hợp  (tầng bốn).\n\nMọi bước đều ghi nhật ký và mọi kết quả trung gian đều được lưu ra đĩa, để có thể\nchạy lại từng phần mà không phải chạy lại toàn bộ.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport numpy as np\n\nfrom .config import RunConfig\nfrom .data.clearance import simulate_accidents, simulate_clearance\nfrom .data.dataset import (\n    detections_to_grid,\n    read_ground_truth,\n    read_tile_metadata,\n    write_dataset,\n)\nfrom .data.geo import Grid\nfrom .data.imagery import build_tiles\nfrom .data.sorties import BOMB_MASS_KG, simulate_scene\nfrom .data.terrain import build_terrain\nfrom .evaluation.ablation import run_ablation\nfrom .evaluation.calibration import calibration_report\nfrom .evaluation.consistency import check_consistency\nfrom .evaluation.detection_metrics import evaluate_detections\nfrom .evaluation.prioritisation import (\n    accident_coverage,\n    clearance_efficiency_curve,\n    priority_index,\n)\nfrom .evaluation.spatial_cv import (\n    block_kfold,\n    make_spatial_blocks,\n    spatial_holdout,\n    transfer_split,\n)\nfrom .risk.features import FEATURE_NAMES, build_features, gaussian_spread\nfrom .risk.model import RiskModel, permutation_importance\nfrom .utils import describe_environment, ensure_dir, get_logger, seed_everything\n\nlogger = get_logger(__name__)\n\n\nclass Pipeline:\n    \"\"\"Quy trình đầu cuối của DeMine-VN.\"\"\"\n\n    def __init__(self, cfg: RunConfig) -> None:\n        self.cfg = cfg\n        self.out = ensure_dir(cfg.output_dir)\n        self.data_dir = ensure_dir(cfg.data_dir)\n        self.results: Dict[str, object] = {}\n        self.rng = np.random.default_rng(cfg.seed)\n        seed_everything(cfg.seed)\n\n    # ------------------------------------------------------------------\n    def step_1_build_area(self):\n        logger.info(\"Bước 1 — dựng vùng nghiên cứu\")\n        self.grid = Grid(self.cfg.grid)\n        self.terrain = build_terrain(self.grid, self.cfg.terrain, self.rng)\n        logger.info(\n            \"  Vùng nghiên cứu %.1f × %.1f km | %d ô lưới cạnh %.0f m\",\n            self.grid.width_m / 1000.0,\n            self.grid.height_m / 1000.0,\n            self.grid.n_cells,\n            self.grid.cell,\n        )\n        return self\n\n    def step_2_simulate(self):\n        logger.info(\"Bước 2 — mô phỏng phi vụ và vật nổ còn sót\")\n        self.scene = simulate_scene(\n            self.grid, self.terrain, self.cfg.sortie, self.cfg.ordnance, self.rng\n        )\n        return self\n\n    def step_3_imagery(self):\n        logger.info(\"Bước 3 — kết xuất ảnh vệ tinh lịch sử\")\n        self.tiles = build_tiles(self.scene, self.cfg.imagery, self.rng)\n        self.data_yaml = write_dataset(self.tiles, self.data_dir / \"imagery\")\n        self.tile_meta = read_tile_metadata(self.data_dir / \"imagery\")\n        return self\n\n    def step_4_detect(self, detector=None, train: bool = True):\n        logger.info(\"Bước 4 — phát hiện hố bom trên ảnh (tầng hai)\")\n        from .detect import build_detector\n\n        detector = detector or build_detector(self.cfg)\n        self.detector = detector\n\n        if train:\n            info = detector.train(self.data_yaml, self.cfg)\n            logger.info(\"  Đã huấn luyện: %s\", info.get(\"weights\", \"(không rõ)\"))\n        else:\n            # Không huấn luyện thì phải nạp trọng số đã có, nếu không tầng hai sẽ\n            # không có mô hình nào để suy luận.\n            runs = Path(self.cfg.output_dir).resolve() / \"runs\"\n            candidates = [\n                runs / \"yolo_crater\" / \"weights\" / \"best.pt\",\n                runs / \"frcnn_crater\" / \"weights\" / \"best.pt\",\n            ]\n            weights = next((c for c in candidates if c.exists()), None)\n            if weights is None:\n                found = sorted(\n                    runs.rglob(\"weights/best.pt\"),\n                    key=lambda q: q.stat().st_mtime,\n                    reverse=True,\n                )\n                weights = found[0] if found else None\n            if weights is None:\n                raise FileNotFoundError(\n                    \"Không tìm thấy trọng số đã huấn luyện trong \"\n                    f\"{Path(self.cfg.output_dir) / 'runs'}. Hãy bỏ tuỳ chọn \"\n                    \"--no-train-detector, hoặc trỏ --output-dir tới thư mục đã có \"\n                    \"trọng số.\"\n                )\n            logger.info(\"  Nạp trọng số đã có: %s\", weights)\n            detector.load(weights)\n\n        root = self.data_dir / \"imagery\"\n        test_images = sorted((root / \"images\" / \"test\").glob(\"*.png\"))\n        preds = detector.predict_sharded(\n            test_images, conf=self.cfg.detect.confidence_threshold\n        )\n        gt = read_ground_truth(root, \"test\")\n        metrics = evaluate_detections(\n            preds, gt, score_threshold=self.cfg.detect.confidence_threshold\n        )\n        self.results[\"tang_hai_phat_hien_ho_bom\"] = metrics.as_dict()\n        logger.info(\n            \"  mAP@0.5 = %.3f | Recall = %.3f | %d hố bom thật, %d dự báo\",\n            metrics.ap50,\n            metrics.recall,\n            metrics.n_true,\n            metrics.n_pred,\n        )\n\n        # Suy luận trên toàn bộ ảnh để dựng lớp hố bom phủ khắp vùng nghiên cứu.\n        all_images = sorted(root.glob(\"images/*/*.png\"))\n        all_preds = detector.predict_sharded(\n            all_images, conf=self.cfg.detect.confidence_threshold\n        )\n        self.crater_map, self.crater_diameter_map, self.crater_coverage = (\n            detections_to_grid(all_preds, self.tile_meta, self.grid)\n        )\n        n_craters = int(self.crater_map.sum())\n        logger.info(\"  Đã quy %d hố bom phát hiện được về lưới\", n_craters)\n\n        # Nếu tầng hai gần như không phát hiện được gì thì nhóm đặc trưng quan sát\n        # sẽ toàn số không, và mọi kết quả liên quan tới nguồn ảnh vệ tinh sẽ bằng\n        # không theo. Đó là hệ quả của việc huấn luyện chưa đủ, không phải kết luận\n        # khoa học, nên phải được nêu rõ thay vì để người đọc tự suy diễn.\n        expected = max(1, int(0.05 * self.scene.impact_crater_visible.sum()))\n        self.detector_underfitted = n_craters < expected\n        if self.detector_underfitted:\n            logger.warning(\n                \"Tầng hai chỉ phát hiện %d hố bom, quá thấp so với mức kỳ vọng. \"\n                \"Nhóm đặc trưng ảnh vệ tinh sẽ gần như không mang thông tin. \"\n                \"Hãy tăng số chu kỳ huấn luyện hoặc chạy trên GPU.\",\n                n_craters,\n            )\n        coverage_fraction = float(self.crater_coverage.mean())\n        logger.info(\n            \"  Ảnh vệ tinh phủ %.1f%% diện tích vùng nghiên cứu; phần còn lại được \"\n            \"đánh dấu là khuyết dữ liệu chứ không phải bằng không.\",\n            100.0 * coverage_fraction,\n        )\n        self.results[\"canh_bao_tang_hai\"] = {\n            \"so_ho_bom_phat_hien_toan_vung\": n_craters,\n            \"ty_le_dien_tich_co_anh_ve_tinh\": round(coverage_fraction, 4),\n            \"huan_luyen_chua_du\": bool(self.detector_underfitted),\n        }\n        return self\n\n    def step_5_features(self):\n        logger.info(\"Bước 5 — dựng bảng đặc trưng bốn nhóm (tầng ba, phần một)\")\n        self.features = build_features(\n            self.scene,\n            self.crater_map,\n            self.crater_diameter_map,\n            self.cfg.risk,\n            crater_coverage=self.crater_coverage,\n        )\n        self.tonnage_spread = gaussian_spread(\n            self.scene.recorded_tonnage,\n            self.cfg.risk.record_kernel_radius_m / self.grid.cell,\n        )\n        logger.info(\n            \"  %d ô × %d đặc trưng\", self.features.n_rows, len(FEATURE_NAMES)\n        )\n        return self\n\n    def step_6_clearance(self):\n        logger.info(\"Bước 6 — mô phỏng rà phá đã thực hiện và hồ sơ tai nạn\")\n        self.clearance = simulate_clearance(\n            self.scene, self.tonnage_spread, self.cfg.clearance, self.rng\n        )\n        self.accidents = simulate_accidents(\n            self.scene, self.clearance, self.cfg.accident, self.rng\n        )\n        return self\n\n    # ------------------------------------------------------------------\n    def _prepare_matrices(self):\n        \"\"\"Chuẩn bị các mảng phẳng dùng chung cho tầng ba và phần kiểm chứng.\"\"\"\n        self.y_true = (self.scene.uxo_count.ravel() > 0).astype(int)\n        self.items = self.scene.uxo_count.ravel().astype(float)\n        self.is_cleared = self.clearance.is_cleared.ravel()\n        self.items_found = self.clearance.items_found.ravel().astype(float)\n        self.y_observed = (self.items_found > 0).astype(int)\n        self.exposure = self.terrain.exposure.ravel()\n\n        self.blocks = make_spatial_blocks(\n            self.features.col,\n            self.features.row,\n            self.cfg.risk.n_spatial_blocks,\n            self.grid.shape,\n        )\n        self.accident_flat = (\n            self.grid.flat_index(self.accidents.col, self.accidents.row)\n            if self.accidents.n\n            else np.array([], dtype=int)\n        )\n\n    def step_7_risk_model(self):\n        logger.info(\"Bước 7 — huấn luyện mô hình nguy cơ (tầng ba, phần hai)\")\n        self._prepare_matrices()\n\n        X = self.features.X\n        labelled = self.is_cleared\n\n        # Tập hiệu chỉnh xác suất được tách riêng theo khối, không dùng để học.\n        fit_mask, cal_mask = spatial_holdout(\n            self.blocks[labelled], test_fraction=0.25, seed=self.cfg.risk.seed\n        )\n\n        self.model = RiskModel(cfg=self.cfg.risk)\n        self.model.fit(\n            X[labelled],\n            self.y_observed[labelled],\n            was_selected=labelled,\n            X_all=X,\n            feature_names=list(FEATURE_NAMES),\n            calibration_mask=cal_mask,\n        )\n\n        self.probability = self.model.predict_probability(X)\n        self.priority = priority_index(self.probability, self.exposure)\n        logger.info(\n            \"  Xác suất trung bình %.4f | kỳ vọng tổng số vật nổ còn lại %.0f\",\n            float(self.probability.mean()),\n            float(self.probability.sum()),\n        )\n        return self\n\n    # ------------------------------------------------------------------\n    def step_8_validate(self):\n        logger.info(\"Bước 8 — kiểm chứng bốn tầng\")\n        X = self.features.X\n\n        # -- Tầng một: kiểm chứng chéo theo khối trên đất đã rà phá -------\n        logger.info(\"  Tầng 1 — đối chứng trên đất đã rà phá, chia theo khối không gian\")\n        labelled_idx = np.flatnonzero(self.is_cleared)\n        fold_scores, fold_gains = [], []\n\n        for train_mask, test_mask in block_kfold(\n            self.blocks[labelled_idx], self.cfg.risk.n_cv_folds, seed=self.cfg.risk.seed\n        ):\n            tr = labelled_idx[train_mask]\n            te = labelled_idx[test_mask]\n            if te.size < 20 or self.y_observed[tr].sum() < 5:\n                continue\n            m = RiskModel(cfg=self.cfg.risk)\n            m.fit(\n                X[tr],\n                self.y_observed[tr],\n                was_selected=self.is_cleared,\n                X_all=X,\n                feature_names=list(FEATURE_NAMES),\n            )\n            p = m.predict_probability(X[te])\n            curve = clearance_efficiency_curve(p, self.items_found[te])\n            fold_scores.append(curve.recovered_at_20pct)\n            fold_gains.append(curve.gain_over_uniform)\n\n        self.results[\"tang_1_doi_chung_dat_da_ra_pha\"] = {\n            \"so_lan_chia\": len(fold_scores),\n            \"thu_hoi_tai_20pct_trung_binh\": round(float(np.mean(fold_scores)), 4)\n            if fold_scores\n            else 0.0,\n            \"do_lech_chuan\": round(float(np.std(fold_scores)), 4) if fold_scores else 0.0,\n            \"loi_the_so_voi_quet_deu\": round(float(np.mean(fold_gains)), 4)\n            if fold_gains\n            else 0.0,\n        }\n        logger.info(\n            \"    thu hồi tại 20%% diện tích: %.3f ± %.3f qua %d lần chia\",\n            float(np.mean(fold_scores)) if fold_scores else 0.0,\n            float(np.std(fold_scores)) if fold_scores else 0.0,\n            len(fold_scores),\n        )\n\n        # Đường cong hiệu quả trên toàn vùng, dùng nhãn thật để báo cáo.\n        self.curve = clearance_efficiency_curve(self.priority, self.items)\n        self.results[\"duong_cong_hieu_qua_ra_pha\"] = self.curve.as_dict()\n\n        # -- Tầng hai: kiểm chứng độc lập bằng hồ sơ tai nạn --------------\n        logger.info(\"  Tầng 2 — đối chứng độc lập bằng hồ sơ tai nạn\")\n        cov_all = accident_coverage(self.priority, self.accident_flat, \"toàn bộ\")\n\n        late = self.accidents.year > self.cfg.accident.temporal_split_year\n        cov_late = accident_coverage(\n            self.priority,\n            self.grid.flat_index(self.accidents.col[late], self.accidents.row[late])\n            if late.any()\n            else np.array([], dtype=int),\n            f\"sau năm thứ {self.cfg.accident.temporal_split_year}\",\n        )\n\n        self.results[\"tang_2_doi_chung_ho_so_tai_nan\"] = [\n            cov_all.as_dict(),\n            cov_late.as_dict(),\n        ]\n        logger.info(\n            \"    bao phủ tại 20%% diện tích: %.3f (toàn bộ) | %.3f (chia theo thời gian)\",\n            cov_all.capture_at_20pct,\n            cov_late.capture_at_20pct,\n        )\n\n        # -- Tầng ba: nhất quán giữa hai nguồn độc lập --------------------\n        logger.info(\"  Tầng 3 — nhất quán giữa hồ sơ không kích và ảnh vệ tinh\")\n        n_recorded_bombs = int(self.scene.record_n_bombs.sum())\n        # Chỉ so sánh trên phần có ảnh vệ tinh. So trên vùng không có ảnh là so mật\n        # độ hố bom bằng không với tải trọng bom khác không, và sẽ cho hệ số gần bằng\n        # không vì lý do hoàn toàn nhân tạo.\n        cov = self.crater_coverage\n        consistency = check_consistency(\n            self.crater_map[cov],\n            self.tonnage_spread[cov],\n            n_recorded_bombs,\n            self.cfg.ordnance.base_dud_rate,\n            self.probability,\n        )\n        self.results[\"tang_3_nhat_quan_hai_nguon\"] = consistency.as_dict()\n        logger.info(\n            \"    Spearman = %.3f trên %d ô | tỉ số bậc độ lớn = %.2f\",\n            consistency.spearman_crater_vs_tonnage,\n            consistency.n_cells_compared,\n            consistency.order_of_magnitude_ratio,\n        )\n\n        # -- Tầng bốn: chuyển vùng và hiệu chỉnh --------------------------\n        logger.info(\"  Tầng 4 — chuyển vùng địa lý và chất lượng hiệu chỉnh\")\n        west, east = transfer_split(self.features.col, self.grid.shape)\n\n        base = self.results[\"tang_1_doi_chung_dat_da_ra_pha\"][\n            \"thu_hoi_tai_20pct_trung_binh\"\n        ]\n        transfer_value = 0.0\n        train_sel = west & self.is_cleared\n        test_sel = east & self.is_cleared\n        if train_sel.sum() > 50 and test_sel.sum() > 20 and self.y_observed[train_sel].sum() >= 5:\n            m = RiskModel(cfg=self.cfg.risk)\n            m.fit(\n                X[train_sel],\n                self.y_observed[train_sel],\n                was_selected=self.is_cleared,\n                X_all=X,\n                feature_names=list(FEATURE_NAMES),\n            )\n            p = m.predict_probability(X[test_sel])\n            transfer_value = clearance_efficiency_curve(\n                p, self.items_found[test_sel]\n            ).recovered_at_20pct\n\n        drop = (base - transfer_value) / base if base > 1e-9 else 0.0\n        self.results[\"tang_4a_chuyen_vung_dia_ly\"] = {\n            \"thu_hoi_tai_20pct_cung_vung\": round(base, 4),\n            \"thu_hoi_tai_20pct_vung_moi\": round(transfer_value, 4),\n            \"muc_suy_giam_tuong_doi\": round(drop, 4),\n        }\n        logger.info(\n            \"    huấn luyện nửa tây, áp dụng nửa đông: %.3f → %.3f (suy giảm %.1f%%)\",\n            base,\n            transfer_value,\n            100.0 * drop,\n        )\n\n        cal = calibration_report(\n            self.probability[self.is_cleared],\n            self.y_observed[self.is_cleared],\n            self.cfg.risk.n_calibration_bins,\n        )\n        self.results[\"tang_4b_hieu_chinh_xac_suat\"] = cal\n        logger.info(\n            \"    sai số hiệu chỉnh kỳ vọng = %.4f | điểm Brier = %.4f\",\n            cal[\"sai_so_hieu_chinh_ky_vong\"],\n            cal[\"diem_brier\"],\n        )\n        return self\n\n    def step_9_ablation(self):\n        logger.info(\"Bước 9 — phân tích đóng góp thành phần\")\n        X = self.features.X\n        train_mask, test_mask = spatial_holdout(\n            self.blocks, test_fraction=0.3, seed=self.cfg.risk.seed + 1\n        )\n        tr = np.flatnonzero(train_mask & self.is_cleared)\n        te = np.flatnonzero(test_mask)\n\n        acc_test = (\n            np.array(\n                [i for i in self.accident_flat if test_mask[i]], dtype=int\n            )\n            if self.accident_flat.size\n            else np.array([], dtype=int)\n        )\n        # Chỉ số tai nạn phải được quy về vị trí trong tập kiểm tra.\n        remap = -np.ones(test_mask.size, dtype=int)\n        remap[te] = np.arange(te.size)\n        acc_test_local = remap[acc_test]\n        acc_test_local = acc_test_local[acc_test_local >= 0]\n\n        rows = run_ablation(\n            X[tr],\n            self.y_observed[tr],\n            self.is_cleared,\n            X,\n            X[te],\n            self.items[te],\n            acc_test_local,\n            self.cfg.risk,\n        )\n        self.results[\"phan_tich_dong_gop_thanh_phan\"] = [r.as_dict() for r in rows]\n\n        self.results[\"muc_dong_gop_dac_trung\"] = self.model.feature_importance()\n        self.results[\"muc_dong_gop_theo_hoan_vi\"] = permutation_importance(\n            self.model, X[te], self.items[te], n_repeats=2, seed=self.cfg.risk.seed\n        )\n        return self\n\n    def step_10_outputs(self):\n        logger.info(\"Bước 10 — kết xuất bản đồ, hình minh hoạ và báo cáo\")\n        from .reporting import write_priority_list, write_report\n        from .viz.figures import make_all_figures\n        from .viz.maps import make_priority_map\n\n        fig_dir = ensure_dir(self.out / \"figures\")\n        make_all_figures(self, fig_dir)\n        make_priority_map(self, self.out / \"ban_do_uu_tien.html\")\n        write_priority_list(self, self.out / \"danh_muc_uu_tien_ra_pha.csv\")\n\n        self.results[\"moi_truong_chay\"] = describe_environment()\n        (self.out / \"ket_qua.json\").write_text(\n            json.dumps(self.results, ensure_ascii=False, indent=2), encoding=\"utf-8\"\n        )\n        write_report(self, self.out / \"bao_cao_tong_hop.md\")\n        logger.info(\"  Đã ghi toàn bộ kết quả vào %s\", self.out)\n        return self\n\n    # ------------------------------------------------------------------\n    def run(self, detector=None, train_detector: bool = True):\n        (\n            self.step_1_build_area()\n            .step_2_simulate()\n            .step_3_imagery()\n            .step_4_detect(detector=detector, train=train_detector)\n            .step_5_features()\n            .step_6_clearance()\n            .step_7_risk_model()\n            .step_8_validate()\n            .step_9_ablation()\n            .step_10_outputs()\n        )\n        return self.results\n", "src/demine/reporting.py": "\"\"\"Kết xuất danh mục ưu tiên rà phá và báo cáo tổng hợp.\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport numpy as np\n\nfrom . import SAFETY_NOTICE, __version__\nfrom .utils import format_table, get_logger\n\nlogger = get_logger(__name__)\n\n\ndef write_priority_list(pipeline, path: Path, n_top: int = 500) -> Path:\n    \"\"\"Danh mục khoảnh đất xếp theo thứ tự rà phá đề xuất.\n\n    Đây là sản phẩm đầu ra mà đơn vị lập kế hoạch sử dụng trực tiếp. Mỗi dòng có\n    đầy đủ toạ độ, xác suất đã hiệu chỉnh, chỉ số ưu tiên và các yếu tố giải thích,\n    để người đọc kiểm tra được vì sao một khoảnh được xếp cao.\n    \"\"\"\n    path = Path(path)\n    order = np.argsort(-pipeline.priority)[:n_top]\n    lon, lat = pipeline.grid.cell_centers_lonlat()\n\n    terrain = pipeline.terrain\n    dist_village = terrain.dist_village_m.ravel()\n    farmland = terrain.is_farmland.ravel()\n    tonnage = pipeline.tonnage_spread.ravel()\n    craters = pipeline.crater_map.ravel()\n\n    with path.open(\"w\", encoding=\"utf-8-sig\", newline=\"\") as fh:\n        writer = csv.writer(fh)\n        writer.writerow(\n            [\n                \"thu_tu_uu_tien\",\n                \"chi_so_o_luoi\",\n                \"kinh_do\",\n                \"vi_do\",\n                \"xac_suat_con_vat_no\",\n                \"chi_so_uu_tien\",\n                \"tai_trong_bom_ghi_nhan_tan\",\n                \"so_ho_bom_phat_hien\",\n                \"khoang_cach_khu_dan_cu_m\",\n                \"la_dat_canh_tac\",\n                \"da_ra_pha\",\n                \"ghi_chu\",\n            ]\n        )\n        for rank, i in enumerate(order, start=1):\n            writer.writerow(\n                [\n                    rank,\n                    int(i),\n                    f\"{lon[i]:.6f}\",\n                    f\"{lat[i]:.6f}\",\n                    f\"{pipeline.probability[i]:.4f}\",\n                    f\"{pipeline.priority[i]:.4f}\",\n                    f\"{tonnage[i]:.2f}\",\n                    int(craters[i]),\n                    f\"{dist_village[i]:.0f}\",\n                    \"co\" if farmland[i] else \"khong\",\n                    \"co\" if pipeline.is_cleared[i] else \"khong\",\n                    \"Thu tu uu tien ra pha. Khong phai xac nhan an toan.\",\n                ]\n            )\n\n    logger.info(\"  Đã ghi danh mục ưu tiên: %s (%d khoảnh)\", path.name, len(order))\n    return path\n\n\ndef _fmt_pct(value: float) -> str:\n    return f\"{100.0 * float(value):.1f}%\"\n\n\ndef write_report(pipeline, path: Path) -> Path:\n    \"\"\"Báo cáo tổng hợp dạng văn bản, đọc được trực tiếp trong notebook.\"\"\"\n    path = Path(path)\n    r: Dict = pipeline.results\n    cfg = pipeline.cfg\n\n    lines: List[str] = []\n    add = lines.append\n\n    add(f\"# DeMine-VN — Báo cáo tổng hợp kết quả (phiên bản {__version__})\")\n    add(\"\")\n    add(\n        \"Hệ thống lập bản đồ nguy cơ và xếp thứ tự ưu tiên rà phá bom mìn, vật nổ \"\n        \"còn sót lại sau chiến tranh, trên cơ sở hợp nhất hồ sơ không kích giải mật, \"\n        \"ảnh vệ tinh lịch sử và dữ liệu rà phá thực địa.\"\n    )\n    add(\"\")\n    add(\"> **Nguyên tắc an toàn bắt buộc.** \" + SAFETY_NOTICE)\n    add(\"\")\n    add(\"---\")\n    add(\"\")\n\n    # -- Vùng nghiên cứu ------------------------------------------------\n    add(\"## 1. Vùng nghiên cứu và dữ liệu\")\n    add(\"\")\n    add(\n        f\"- Diện tích: {pipeline.grid.width_m/1000:.1f} × \"\n        f\"{pipeline.grid.height_m/1000:.1f} km, chia thành \"\n        f\"{pipeline.grid.n_cells:,} ô lưới cạnh {pipeline.grid.cell:.0f} m\".replace(\",\", \".\")\n    )\n    add(f\"- Phi vụ mô phỏng: {cfg.sortie.n_missions}, trong đó \"\n        f\"{pipeline.scene.record_x.size} phi vụ còn hồ sơ\")\n    add(f\"- Điểm rơi: {pipeline.scene.n_impacts:,}\".replace(\",\", \".\"))\n    add(f\"- Vật nổ còn sót (nhãn đối chứng): {pipeline.scene.n_uxo:,}\".replace(\",\", \".\"))\n    add(f\"- Hố bom phát hiện được trên ảnh: {int(pipeline.crater_map.sum()):,}\".replace(\",\", \".\"))\n    add(f\"- Diện tích đã rà phá: {_fmt_pct(pipeline.clearance.is_cleared.mean())}\")\n    add(f\"- Tai nạn đã ghi nhận: {pipeline.accidents.n} vụ trong {cfg.accident.n_years} năm\")\n    add(\"\")\n\n    # -- Tầng hai --------------------------------------------------------\n    add(\"## 2. Tầng hai — phát hiện hố bom trên ảnh vệ tinh lịch sử\")\n    add(\"\")\n    det = r.get(\"tang_hai_phat_hien_ho_bom\", {})\n    if det:\n        add(format_table([det]))\n    add(\"\")\n\n    # -- Kiểm chứng ------------------------------------------------------\n    add(\"## 3. Kiểm chứng bốn tầng\")\n    add(\"\")\n\n    add(\"### Tầng 1 — đối chứng trên đất đã rà phá\")\n    add(\"\")\n    add(\n        \"Chia tập **theo khối không gian**, không chia ngẫu nhiên theo điểm. \"\n        \"Chi tiết phương pháp xem `docs/VALIDATION.md`.\"\n    )\n    add(\"\")\n    t1 = r.get(\"tang_1_doi_chung_dat_da_ra_pha\", {})\n    if t1:\n        add(format_table([t1]))\n    add(\"\")\n    curve = r.get(\"duong_cong_hieu_qua_ra_pha\", {})\n    if curve:\n        add(\"Đường cong hiệu quả rà phá trên toàn vùng:\")\n        add(\"\")\n        add(format_table([curve]))\n    add(\"\")\n\n    add(\"### Tầng 2 — đối chứng độc lập bằng hồ sơ tai nạn\")\n    add(\"\")\n    add(\n        \"Vị trí tai nạn không do mô hình chọn cũng không do cơ quan chuyên môn chọn, \"\n        \"nên đây là phép lấy mẫu độc lập với mọi phán đoán đã có trước. Dòng thứ hai \"\n        \"là phép thử nghiêm hơn: mô hình chỉ học dữ liệu trước một mốc thời gian và \"\n        \"được đánh giá bằng các vụ tai nạn xảy ra sau mốc đó.\"\n    )\n    add(\"\")\n    t2 = r.get(\"tang_2_doi_chung_ho_so_tai_nan\", [])\n    if t2:\n        add(format_table(t2))\n    add(\"\")\n\n    add(\"### Tầng 3 — nhất quán giữa hai nguồn độc lập\")\n    add(\"\")\n    add(\n        \"Hồ sơ không kích và ảnh vệ tinh không liên quan về xuất xứ. Nếu chúng khớp \"\n        \"nhau về mặt không gian thì độ tin cậy của cả hai cùng được củng cố mà không \"\n        \"cần viện đến bất kỳ nhãn đối chứng nào.\"\n    )\n    add(\"\")\n    t3 = r.get(\"tang_3_nhat_quan_hai_nguon\", {})\n    if t3:\n        add(format_table([t3]))\n    add(\"\")\n\n    add(\"### Tầng 4 — chuyển vùng địa lý và chất lượng hiệu chỉnh\")\n    add(\"\")\n    t4a = r.get(\"tang_4a_chuyen_vung_dia_ly\", {})\n    if t4a:\n        add(format_table([t4a]))\n    add(\"\")\n    t4b = r.get(\"tang_4b_hieu_chinh_xac_suat\", {})\n    if t4b:\n        add(format_table([t4b]))\n    add(\"\")\n\n    # -- Đóng góp thành phần ---------------------------------------------\n    add(\"## 4. Phân tích đóng góp thành phần\")\n    add(\"\")\n    add(\n        \"Hồ sơ không kích có sai số định vị lớn nhưng phủ khắp; hố bom thì chính xác \"\n        \"về vị trí nhưng thiếu hụt có hệ thống đúng ở nơi nền đất mềm — tức là đúng nơi \"\n        \"nhiều vật nổ còn sót nhất. Hai nguồn sai theo hai kiểu khác nhau, nên việc hợp \"\n        \"nhất có giá trị thật chứ không phải cộng thêm cho đủ.\"\n    )\n    add(\"\")\n    abl = r.get(\"phan_tich_dong_gop_thanh_phan\", [])\n    if abl:\n        add(format_table(abl))\n    add(\"\")\n\n    imp = r.get(\"muc_dong_gop_theo_hoan_vi\", {})\n    if imp:\n        add(\"### Đóng góp của từng đặc trưng, đo bằng phép hoán vị\")\n        add(\"\")\n        top = list(imp.items())[:10]\n        add(format_table([{\"dac_trung\": k, \"muc_sut_giam\": v} for k, v in top]))\n        add(\"\")\n\n    # -- Kết luận ---------------------------------------------------------\n    add(\"## 5. Phạm vi cam kết\")\n    add(\"\")\n    add(\n        \"Đề tài không cam kết rằng hệ thống phát hiện được vật nổ. Đề tài cam kết bốn \"\n        \"điều, và cả bốn đều được kiểm chứng bằng dữ liệu trong chính báo cáo này:\"\n    )\n    add(\"\")\n    add(\"1. Xếp hạng đúng các khoảnh đất đã rà phá, đánh giá trên khối không gian giữ lại.\")\n    add(\"2. Bao phủ phần lớn các vụ tai nạn đã xảy ra bằng một phần nhỏ diện tích.\")\n    add(\"3. Duy trì hiệu năng khi chuyển sang địa bàn chưa từng xuất hiện trong huấn luyện.\")\n    add(\"4. Cung cấp ước lượng xác suất đã hiệu chỉnh, kèm biểu đồ tin cậy và điểm Brier.\")\n    add(\"\")\n    add(\"> **Nguyên tắc an toàn bắt buộc.** \" + SAFETY_NOTICE)\n    add(\"\")\n\n    env = r.get(\"moi_truong_chay\", {})\n    if env:\n        add(\"---\")\n        add(\"\")\n        add(\"## Môi trường chạy\")\n        add(\"\")\n        for k, v in env.items():\n            add(f\"- {k}: {v}\")\n        add(\"\")\n\n    path.write_text(\"\\n\".join(lines), encoding=\"utf-8\")\n    logger.info(\"  Đã ghi báo cáo tổng hợp: %s\", path.name)\n    return path\n", "src/demine/utils.py": "\"\"\"Tiện ích dùng chung: nhật ký, gieo hạt ngẫu nhiên, thư mục.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport random\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Optional\n\nimport numpy as np\n\n_LOG_FORMAT = \"%(asctime)s | %(levelname)-7s | %(name)-26s | %(message)s\"\n_DATE_FORMAT = \"%H:%M:%S\"\n\n\ndef setup_logging(level: int = logging.INFO) -> None:\n    root = logging.getLogger()\n    if root.handlers:\n        return\n    handler = logging.StreamHandler(sys.stdout)\n    handler.setFormatter(logging.Formatter(_LOG_FORMAT, datefmt=_DATE_FORMAT))\n    root.addHandler(handler)\n    root.setLevel(level)\n\n\ndef get_logger(name: str) -> logging.Logger:\n    return logging.getLogger(name)\n\n\ndef seed_everything(seed: int) -> None:\n    \"\"\"Cố định mọi nguồn ngẫu nhiên để kết quả tái lập được.\"\"\"\n    random.seed(seed)\n    np.random.seed(seed % (2 ** 32 - 1))\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        if torch.cuda.is_available():\n            torch.cuda.manual_seed_all(seed)\n    except Exception:\n        pass\n\n\ndef ensure_dir(path) -> Path:\n    p = Path(path)\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef describe_environment() -> dict:\n    \"\"\"Ghi nhận môi trường chạy để đưa vào báo cáo.\"\"\"\n    info = {\"python\": sys.version.split()[0], \"numpy\": np.__version__}\n    try:\n        import torch\n\n        info[\"torch\"] = torch.__version__\n        info[\"cuda_available\"] = bool(torch.cuda.is_available())\n        info[\"gpu_count\"] = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0\n        if info[\"gpu_count\"]:\n            info[\"gpu_names\"] = [\n                torch.cuda.get_device_name(i) for i in range(info[\"gpu_count\"])\n            ]\n    except Exception:\n        info[\"torch\"] = None\n        info[\"cuda_available\"] = False\n        info[\"gpu_count\"] = 0\n    return info\n\n\ndef format_int(value: float) -> str:\n    return f\"{int(round(value)):,}\".replace(\",\", \".\")\n\n\ndef package_available(name: str) -> bool:\n    \"\"\"Kiểm tra một gói Python có nạp được hay không.\"\"\"\n    import importlib.util\n\n    try:\n        return importlib.util.find_spec(name) is not None\n    except Exception:\n        return False\n\n\n@dataclass\nclass DeviceInfo:\n    n_gpu: int\n    names: list\n    kind: str = \"cpu\"\n\n\ndef probe_devices() -> DeviceInfo:\n    \"\"\"Thăm dò thiết bị tăng tốc hiện có.\n\n    Hỗ trợ ba trường hợp: nhiều card NVIDIA (môi trường Kaggle), bộ tăng tốc Metal\n    trên máy Apple dùng chip dòng M, và chỉ có CPU. Việc nhận biết Metal là cần thiết\n    để đội thi phát triển và thử nghiệm được ngay trên máy cá nhân, thay vì phải chờ\n    tới lúc lên Kaggle mới chạy được.\n    \"\"\"\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            n = int(torch.cuda.device_count())\n            return DeviceInfo(\n                n, [torch.cuda.get_device_name(i) for i in range(n)], \"cuda\"\n            )\n        if getattr(torch.backends, \"mps\", None) is not None and torch.backends.mps.is_available():\n            return DeviceInfo(1, [\"Apple Metal (MPS)\"], \"mps\")\n    except Exception:\n        pass\n    return DeviceInfo(0, [], \"cpu\")\n\n\ndef torch_device(index: int = 0):\n    \"\"\"Trả về thiết bị torch phù hợp với môi trường hiện tại.\"\"\"\n    import torch\n\n    info = probe_devices()\n    if info.kind == \"cuda\":\n        return torch.device(f\"cuda:{index}\")\n    if info.kind == \"mps\":\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef resolve_devices(requested, allow_multi: bool = True):\n    \"\"\"Lọc danh sách GPU yêu cầu theo số thiết bị thực có.\"\"\"\n    info = probe_devices()\n    if info.n_gpu == 0:\n        return []\n    if info.kind == \"mps\":\n        # Metal chỉ phơi bày một thiết bị duy nhất, không có khái niệm chỉ số card.\n        return [0]\n    usable = [d for d in requested if d < info.n_gpu]\n    if not usable:\n        usable = [0]\n    if not allow_multi:\n        usable = usable[:1]\n    return usable\n\n\ndef gpu_utilisation() -> Optional[str]:\n    \"\"\"Đọc mức sử dụng GPU qua nvidia-smi, trả về None nếu không có.\"\"\"\n    import subprocess\n\n    try:\n        out = subprocess.run(\n            [\n                \"nvidia-smi\",\n                \"--query-gpu=index,name,utilization.gpu,memory.used,memory.total\",\n                \"--format=csv,noheader,nounits\",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=10,\n        )\n        if out.returncode != 0:\n            return None\n        return out.stdout.strip()\n    except Exception:\n        return None\n\n\ndef format_table(rows, headers=None) -> str:\n    \"\"\"Định dạng danh sách bản ghi thành bảng văn bản căn cột.\"\"\"\n    if not rows:\n        return \"(không có dữ liệu)\"\n    headers = headers or list(rows[0].keys())\n    widths = {\n        h: max(len(str(h)), max(len(str(r.get(h, \"\"))) for r in rows)) for h in headers\n    }\n    line = \" | \".join(str(h).ljust(widths[h]) for h in headers)\n    sep = \"-+-\".join(\"-\" * widths[h] for h in headers)\n    body = [\n        \" | \".join(str(r.get(h, \"\")).ljust(widths[h]) for h in headers) for r in rows\n    ]\n    return \"\\n\".join([line, sep] + body)\n", "src/demine/data/__init__.py": "\"\"\"Tầng một — dữ liệu, địa hình và mô phỏng.\"\"\"\n\nfrom .clearance import (\n    AccidentData,\n    ClearanceData,\n    simulate_accidents,\n    simulate_clearance,\n)\nfrom .dataset import (\n    detections_to_grid,\n    read_ground_truth,\n    read_tile_metadata,\n    write_dataset,\n)\nfrom .geo import Grid\nfrom .imagery import Tile, build_tiles\nfrom .sorties import Scene, simulate_scene\nfrom .terrain import Terrain, build_terrain\n\n__all__ = [\n    \"AccidentData\",\n    \"ClearanceData\",\n    \"simulate_accidents\",\n    \"simulate_clearance\",\n    \"detections_to_grid\",\n    \"read_ground_truth\",\n    \"read_tile_metadata\",\n    \"write_dataset\",\n    \"Grid\",\n    \"Tile\",\n    \"build_tiles\",\n    \"Scene\",\n    \"simulate_scene\",\n    \"Terrain\",\n    \"build_terrain\",\n]\n", "src/demine/data/clearance.py": "\"\"\"Mô phỏng hoạt động rà phá đã thực hiện và hồ sơ tai nạn.\n\nHai nguồn nhãn này có tính chất thống kê hoàn toàn khác nhau, và chính sự khác\nnhau đó là nền tảng của phương pháp kiểm chứng bốn tầng.\n\nĐất đã rà phá **không phải mẫu ngẫu nhiên**. Các cơ quan chuyên môn chọn khoảnh\nđất để rà phá dựa trên hồ sơ không kích, mức độ gần khu dân cư và nhu cầu sử dụng\nđất. Nếu huấn luyện và đánh giá chỉ trên nguồn này, mô hình sẽ học lại chính phán\nđoán của những người đi trước, và mọi chỉ tiêu sẽ bị thổi phồng. Mô-đun này tái\nhiện đúng cơ chế chọn mẫu đó để tầng ba có cái mà hiệu chỉnh.\n\nHồ sơ tai nạn thì ngược lại. Vị trí tai nạn không do ai chọn — đó là nơi người dân\nvô tình chạm phải vật nổ trong sinh hoạt. Nó phụ thuộc vào vật nổ có thật ở đó hay\nkhông và mức độ lui tới của con người, chứ không phụ thuộc phán đoán chuyên môn.\nVì vậy đây là tập kiểm chứng độc lập, và là bằng chứng mạnh nhất của đề tài.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom ..config import AccidentConfig, ClearanceConfig\nfrom ..utils import get_logger\nfrom .sorties import Scene\n\nlogger = get_logger(__name__)\n\n# Cạnh một khoảnh rà phá, tính bằng số ô lưới. Năm ô tương ứng 500 mét, là bậc\n# kích thước của một khoảnh được giao cho một đội trong thực tế.\nPARCEL_CELLS = 5\n\n\ndef _zscore(a: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float)\n    s = a.std()\n    return (a - a.mean()) / (s if s > 1e-12 else 1.0)\n\n\n@dataclass\nclass ClearanceData:\n    \"\"\"Kết quả rà phá đã thực hiện, theo từng ô lưới.\"\"\"\n\n    is_cleared: np.ndarray\n    items_found: np.ndarray\n    parcel_id: np.ndarray\n    propensity: np.ndarray\n\n    @property\n    def n_cleared_cells(self) -> int:\n        return int(self.is_cleared.sum())\n\n\n@dataclass\nclass AccidentData:\n    \"\"\"Hồ sơ tai nạn bom mìn đã ghi nhận.\"\"\"\n\n    col: np.ndarray\n    row: np.ndarray\n    year: np.ndarray\n\n    @property\n    def n(self) -> int:\n        return int(self.col.size)\n\n\ndef simulate_clearance(\n    scene: Scene,\n    tonnage_spread: np.ndarray,\n    cfg: ClearanceConfig,\n    rng: np.random.Generator,\n) -> ClearanceData:\n    \"\"\"Chọn các khoảnh đất đưa vào rà phá theo cơ chế có thiên lệch, rồi rà phá.\"\"\"\n    grid = scene.grid\n    terrain = scene.terrain\n    ny, nx = grid.shape\n\n    ny_p = int(np.ceil(ny / PARCEL_CELLS))\n    nx_p = int(np.ceil(nx / PARCEL_CELLS))\n\n    cols, rows = np.meshgrid(np.arange(nx), np.arange(ny))\n    parcel_id = (rows // PARCEL_CELLS) * nx_p + (cols // PARCEL_CELLS)\n\n    # Điểm ưu tiên của cơ quan chuyên môn khi chọn khoảnh đất.\n    score = (\n        cfg.selection_weight_tonnage * _zscore(tonnage_spread)\n        + cfg.selection_weight_near_village * _zscore(np.exp(-terrain.dist_village_m / 1200.0))\n        + cfg.selection_weight_near_road * _zscore(np.exp(-terrain.dist_road_m / 900.0))\n    )\n    propensity = 1.0 / (1.0 + np.exp(-score))\n\n    n_parcels = ny_p * nx_p\n    parcel_score = np.zeros(n_parcels, dtype=float)\n    np.add.at(parcel_score, parcel_id.ravel(), score.ravel())\n    parcel_count = np.bincount(parcel_id.ravel(), minlength=n_parcels).astype(float)\n    parcel_score = parcel_score / np.maximum(parcel_count, 1.0)\n\n    # Quyết định chọn không hoàn toàn theo điểm: thêm nhiễu để phản ánh các yếu\n    # tố ngoài mô hình như ngân sách, đề nghị của địa phương, khả năng tiếp cận.\n    parcel_score = parcel_score + rng.normal(0.0, 0.85, size=n_parcels)\n\n    n_select = max(1, int(round(n_parcels * cfg.cleared_area_fraction)))\n    chosen = np.argsort(-parcel_score)[:n_select]\n    chosen_mask = np.zeros(n_parcels, dtype=bool)\n    chosen_mask[chosen] = True\n\n    is_cleared = chosen_mask[parcel_id]\n\n    # Rà phá: thu hồi được phần lớn nhưng không phải toàn bộ vật nổ trong khoảnh.\n    true_counts = scene.uxo_count.astype(int)\n    found = np.zeros_like(true_counts)\n    sel = is_cleared & (true_counts > 0)\n    if sel.any():\n        found[sel] = rng.binomial(true_counts[sel], cfg.detection_efficiency)\n\n    logger.info(\n        \"Rà phá đã thực hiện: %d/%d khoảnh (%.1f%% diện tích) | thu hồi %d vật nổ trên tổng %d nằm trong vùng đã rà\",\n        n_select,\n        n_parcels,\n        100.0 * is_cleared.mean(),\n        int(found.sum()),\n        int(true_counts[is_cleared].sum()),\n    )\n    return ClearanceData(\n        is_cleared=is_cleared,\n        items_found=found,\n        parcel_id=parcel_id,\n        propensity=propensity,\n    )\n\n\ndef simulate_accidents(\n    scene: Scene,\n    clearance: ClearanceData,\n    cfg: AccidentConfig,\n    rng: np.random.Generator,\n) -> AccidentData:\n    \"\"\"Sinh hồ sơ tai nạn từ vật nổ còn sót và mức độ phơi nhiễm của con người.\n\n    Vật nổ nằm trong khoảnh đã rà phá và đã được thu hồi thì không còn gây tai nạn\n    nữa, nên được loại khỏi nguồn rủi ro.\n    \"\"\"\n    grid = scene.grid\n    terrain = scene.terrain\n\n    dud = scene.impact_is_dud\n    x = scene.impact_x[dud]\n    y = scene.impact_y[dud]\n    col, row = grid.xy_to_index(x, y)\n\n    # Vật nổ đã bị thu hồi thì loại khỏi nguồn rủi ro.\n    removed_fraction = np.zeros(scene.grid.shape, dtype=float)\n    with np.errstate(divide=\"ignore\", invalid=\"ignore\"):\n        removed_fraction = np.where(\n            scene.uxo_count > 0, clearance.items_found / np.maximum(scene.uxo_count, 1), 0.0\n        )\n    still_present = rng.random(x.size) >= removed_fraction[row, col]\n\n    exposure = terrain.exposure[row, col]\n    hazard = cfg.annual_incident_rate * exposure * still_present\n\n    acc_col, acc_row, acc_year = [], [], []\n    for year in range(1, cfg.n_years + 1):\n        hit = rng.random(x.size) < hazard\n        if not hit.any():\n            continue\n        acc_col.append(col[hit])\n        acc_row.append(row[hit])\n        acc_year.append(np.full(int(hit.sum()), year, dtype=int))\n        # Vật nổ đã gây tai nạn thì được xử lý ngay sau đó.\n        hazard = np.where(hit, 0.0, hazard)\n\n    if acc_col:\n        acc_col = np.concatenate(acc_col)\n        acc_row = np.concatenate(acc_row)\n        acc_year = np.concatenate(acc_year)\n    else:\n        acc_col = np.array([], dtype=int)\n        acc_row = np.array([], dtype=int)\n        acc_year = np.array([], dtype=int)\n\n    logger.info(\n        \"Hồ sơ tai nạn: %d vụ trong %d năm | %d vụ trước mốc chia thời gian, %d vụ sau\",\n        acc_col.size,\n        cfg.n_years,\n        int((acc_year <= cfg.temporal_split_year).sum()),\n        int((acc_year > cfg.temporal_split_year).sum()),\n    )\n    return AccidentData(col=acc_col, row=acc_row, year=acc_year)\n", "src/demine/data/dataset.py": "\"\"\"Ghi bộ dữ liệu ảnh ra đĩa theo quy ước YOLO và đọc lại.\n\nQuy ước thư mục là hợp đồng dữ liệu giữa tầng một và tầng hai. Khi thay dữ liệu\nmô phỏng bằng ảnh vệ tinh giải mật thật, chỉ cần tạo ra đúng cấu trúc thư mục này\nlà toàn bộ phần còn lại của hệ thống chạy không cần sửa. Xem ``docs/ADAPT_NEW_DATA.md``.\n\n    <root>/images/{train,val,test}/<tile_id>.png\n    <root>/labels/{train,val,test}/<tile_id>.txt\n    <root>/tiles.jsonl\n    <root>/data.yaml\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Dict, List\n\nimport numpy as np\n\nfrom ..utils import ensure_dir, get_logger\nfrom .imagery import Tile\n\nlogger = get_logger(__name__)\n\nSPLITS = (\"train\", \"val\", \"test\")\n\n\ndef save_image(path: Path, array: np.ndarray) -> None:\n    from PIL import Image\n\n    Image.fromarray(array).save(path)\n\n\ndef load_image(path: Path) -> np.ndarray:\n    from PIL import Image\n\n    return np.asarray(Image.open(path).convert(\"L\"))\n\n\ndef _to_yolo_line(box: np.ndarray, size: int) -> str:\n    x1, y1, x2, y2 = box\n    xc = (x1 + x2) / 2.0 / size\n    yc = (y1 + y2) / 2.0 / size\n    w = (x2 - x1) / size\n    h = (y2 - y1) / size\n    return f\"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\"\n\n\ndef write_dataset(tiles: List[Tile], root: Path) -> Path:\n    \"\"\"Ghi toàn bộ ảnh con, nhãn, siêu dữ liệu và tệp mô tả bộ dữ liệu.\"\"\"\n    root = ensure_dir(root)\n    for split in SPLITS:\n        ensure_dir(root / \"images\" / split)\n        ensure_dir(root / \"labels\" / split)\n\n    meta_path = root / \"tiles.jsonl\"\n    counts: Dict[str, int] = {s: 0 for s in SPLITS}\n\n    with meta_path.open(\"w\", encoding=\"utf-8\") as fh:\n        for tile in tiles:\n            name = f\"tile_{tile.tile_id:05d}\"\n            img_path = root / \"images\" / tile.split / f\"{name}.png\"\n            lbl_path = root / \"labels\" / tile.split / f\"{name}.txt\"\n\n            save_image(img_path, tile.image)\n            lines = [_to_yolo_line(b, tile.size_px) for b in tile.boxes]\n            lbl_path.write_text(\"\\n\".join(lines) + (\"\\n\" if lines else \"\"), encoding=\"utf-8\")\n\n            counts[tile.split] += 1\n            fh.write(\n                json.dumps(\n                    {\n                        \"tile_id\": tile.tile_id,\n                        \"name\": name,\n                        \"split\": tile.split,\n                        \"origin_x_m\": tile.origin_x_m,\n                        \"origin_y_m\": tile.origin_y_m,\n                        \"gsd_m\": tile.gsd_m,\n                        \"size_px\": tile.size_px,\n                        \"n_craters\": int(tile.boxes.shape[0]),\n                    },\n                    ensure_ascii=False,\n                )\n                + \"\\n\"\n            )\n\n    yaml_path = root / \"data.yaml\"\n    yaml_path.write_text(\n        \"\\n\".join(\n            [\n                f\"path: {root.resolve()}\",\n                \"train: images/train\",\n                \"val: images/val\",\n                \"test: images/test\",\n                \"names:\",\n                \"  0: ho_bom\",\n                \"\",\n            ]\n        ),\n        encoding=\"utf-8\",\n    )\n\n    logger.info(\n        \"Đã ghi bộ dữ liệu ảnh: %d huấn luyện | %d thẩm định | %d kiểm tra → %s\",\n        counts[\"train\"],\n        counts[\"val\"],\n        counts[\"test\"],\n        root,\n    )\n    return yaml_path\n\n\ndef read_tile_metadata(root: Path) -> List[dict]:\n    path = Path(root) / \"tiles.jsonl\"\n    if not path.exists():\n        return []\n    return [json.loads(line) for line in path.read_text(encoding=\"utf-8\").splitlines() if line.strip()]\n\n\ndef read_ground_truth(root: Path, split: str) -> Dict[str, np.ndarray]:\n    \"\"\"Đọc nhãn thật của một tập, trả về khung bao theo đơn vị điểm ảnh.\"\"\"\n    root = Path(root)\n    lbl_dir = root / \"labels\" / split\n    img_dir = root / \"images\" / split\n    out: Dict[str, np.ndarray] = {}\n\n    for lbl in sorted(lbl_dir.glob(\"*.txt\")):\n        img = img_dir / f\"{lbl.stem}.png\"\n        if not img.exists():\n            continue\n        size = load_image(img).shape[0]\n        boxes = []\n        for line in lbl.read_text(encoding=\"utf-8\").splitlines():\n            parts = line.split()\n            if len(parts) != 5:\n                continue\n            _, xc, yc, w, h = map(float, parts)\n            boxes.append(\n                [\n                    (xc - w / 2) * size,\n                    (yc - h / 2) * size,\n                    (xc + w / 2) * size,\n                    (yc + h / 2) * size,\n                ]\n            )\n        out[lbl.stem] = np.asarray(boxes, dtype=float).reshape(-1, 4)\n    return out\n\n\ndef coverage_mask(tile_meta: List[dict], grid) -> np.ndarray:\n    \"\"\"Đánh dấu những ô lưới thực sự nằm trong phạm vi có ảnh vệ tinh.\n\n    Đây là một phân biệt bắt buộc, không phải chi tiết phụ. Ảnh vệ tinh lịch sử\n    không bao giờ phủ kín toàn bộ vùng nghiên cứu — kho ảnh giải mật là những dải\n    chụp rời rạc. Nếu gán số không cho cả những ô không có ảnh, mô hình sẽ đọc\n    thành ``nơi này đã được nhìn và không thấy hố bom nào``, trong khi sự thật là\n    ``nơi này chưa từng được nhìn``. Hai điều đó khác nhau hoàn toàn, và nhầm lẫn\n    giữa chúng khiến mô hình đánh giá thấp toàn bộ phần lãnh thổ chưa có ảnh.\n    \"\"\"\n    covered = np.zeros(grid.shape, dtype=bool)\n    for m in tile_meta:\n        extent = float(m[\"size_px\"]) * float(m[\"gsd_m\"])\n        x0, y0 = float(m[\"origin_x_m\"]), float(m[\"origin_y_m\"])\n        c0, r0 = grid.xy_to_index(np.array([x0]), np.array([y0]))\n        c1, r1 = grid.xy_to_index(np.array([x0 + extent]), np.array([y0 + extent]))\n        c0 = int(np.clip(c0[0], 0, grid.cfg.n_cells_x - 1))\n        c1 = int(np.clip(c1[0], 0, grid.cfg.n_cells_x - 1))\n        r0 = int(np.clip(r0[0], 0, grid.cfg.n_cells_y - 1))\n        r1 = int(np.clip(r1[0], 0, grid.cfg.n_cells_y - 1))\n        covered[r0 : r1 + 1, c0 : c1 + 1] = True\n    return covered\n\n\ndef detections_to_grid(\n    detections, tile_meta: List[dict], grid\n) -> \"tuple[np.ndarray, np.ndarray, np.ndarray]\":\n    \"\"\"Quy các hố bom phát hiện được về lưới ô của vùng nghiên cứu.\n\n    Trả về ba lớp: số lượng hố bom trên mỗi ô, đường kính trung bình của hố bom\n    trên ô đó, và mặt nạ cho biết ô nào thực sự có ảnh vệ tinh phủ tới.\n    \"\"\"\n    by_name = {m[\"name\"]: m for m in tile_meta}\n    count = np.zeros(grid.shape, dtype=float)\n    diameter_sum = np.zeros(grid.shape, dtype=float)\n\n    for det in detections:\n        meta = by_name.get(det.image_id)\n        if meta is None or det.boxes.shape[0] == 0:\n            continue\n        gsd = float(meta[\"gsd_m\"])\n        size = int(meta[\"size_px\"])\n        ox, oy = float(meta[\"origin_x_m\"]), float(meta[\"origin_y_m\"])\n\n        cx_px = (det.boxes[:, 0] + det.boxes[:, 2]) / 2.0\n        cy_px = (det.boxes[:, 1] + det.boxes[:, 3]) / 2.0\n        width_px = det.boxes[:, 2] - det.boxes[:, 0]\n\n        x_m = ox + cx_px * gsd\n        y_m = oy + (size - 1 - cy_px) * gsd\n        diameter_m = width_px * gsd / 0.62 / 2.0 * 2.0\n\n        col, row = grid.xy_to_index(x_m, y_m)\n        keep = grid.inside(col, row)\n        np.add.at(count, (row[keep], col[keep]), 1.0)\n        np.add.at(diameter_sum, (row[keep], col[keep]), diameter_m[keep])\n\n    mean_diameter = np.divide(\n        diameter_sum, count, out=np.zeros_like(diameter_sum), where=count > 0\n    )\n    return count, mean_diameter, coverage_mask(tile_meta, grid)\n", "src/demine/data/geo.py": "\"\"\"Hệ quy chiếu của vùng nghiên cứu.\n\nToàn hệ thống làm việc trên một lưới ô vuông phẳng có gốc quy ước. Mô-đun này\nchuyển đổi hai chiều giữa toạ độ mét trong hệ phẳng, chỉ số ô lưới và toạ độ địa\nlý dùng để hiển thị bản đồ.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Tuple\n\nimport numpy as np\n\nfrom ..config import GridConfig\n\n# Bán kính Trái Đất trung bình, mét.\nEARTH_RADIUS_M = 6_371_000.0\n\n\n@dataclass\nclass Grid:\n    \"\"\"Lưới ô vuông của vùng nghiên cứu.\"\"\"\n\n    cfg: GridConfig\n\n    @property\n    def cell(self) -> float:\n        return self.cfg.cell_size_m\n\n    @property\n    def shape(self) -> Tuple[int, int]:\n        return (self.cfg.n_cells_y, self.cfg.n_cells_x)\n\n    @property\n    def n_cells(self) -> int:\n        return self.cfg.n_cells_x * self.cfg.n_cells_y\n\n    @property\n    def width_m(self) -> float:\n        return self.cfg.n_cells_x * self.cell\n\n    @property\n    def height_m(self) -> float:\n        return self.cfg.n_cells_y * self.cell\n\n    def xy_to_index(self, x_m, y_m):\n        \"\"\"Chuyển toạ độ mét sang chỉ số cột và hàng của ô lưới.\"\"\"\n        col = np.floor(np.asarray(x_m, dtype=float) / self.cell).astype(int)\n        row = np.floor(np.asarray(y_m, dtype=float) / self.cell).astype(int)\n        return col, row\n\n    def inside(self, col, row):\n        col = np.asarray(col)\n        row = np.asarray(row)\n        return (col >= 0) & (col < self.cfg.n_cells_x) & (row >= 0) & (row < self.cfg.n_cells_y)\n\n    def flat_index(self, col, row):\n        return np.asarray(row) * self.cfg.n_cells_x + np.asarray(col)\n\n    def index_to_center_xy(self, col, row):\n        x = (np.asarray(col, dtype=float) + 0.5) * self.cell\n        y = (np.asarray(row, dtype=float) + 0.5) * self.cell\n        return x, y\n\n    def xy_to_lonlat(self, x_m, y_m):\n        \"\"\"Chuyển toạ độ mét sang kinh độ và vĩ độ để hiển thị bản đồ.\"\"\"\n        lat0 = np.deg2rad(self.cfg.origin_lat)\n        dlat = np.asarray(y_m, dtype=float) / EARTH_RADIUS_M\n        dlon = np.asarray(x_m, dtype=float) / (EARTH_RADIUS_M * np.cos(lat0))\n        return (\n            self.cfg.origin_lon + np.rad2deg(dlon),\n            self.cfg.origin_lat + np.rad2deg(dlat),\n        )\n\n    def cell_centers_lonlat(self):\n        cols, rows = np.meshgrid(\n            np.arange(self.cfg.n_cells_x), np.arange(self.cfg.n_cells_y)\n        )\n        x, y = self.index_to_center_xy(cols.ravel(), rows.ravel())\n        return self.xy_to_lonlat(x, y)\n\n    def accumulate(self, x_m, y_m, weights=None):\n        \"\"\"Dồn các điểm về lưới, trả về mảng hai chiều.\"\"\"\n        col, row = self.xy_to_index(x_m, y_m)\n        keep = self.inside(col, row)\n        col, row = col[keep], row[keep]\n        w = None if weights is None else np.asarray(weights, dtype=float)[keep]\n        out = np.zeros(self.shape, dtype=float)\n        np.add.at(out, (row, col), 1.0 if w is None else w)\n        return out\n", "src/demine/data/imagery.py": "\"\"\"Dựng ảnh vệ tinh trinh sát lịch sử mô phỏng.\n\nẢnh trinh sát giải mật của thời kỳ được nghiên cứu là ảnh phim đơn sắc, quét lại\nở độ phân giải vài mét. Ba đặc điểm của loại ảnh này gây khó cho mô hình thị giác\nvà đều được tái hiện ở đây:\n\n  hạt phim        — nhiễu hạt thô, không phải nhiễu Gauss mịn như ảnh số\n  chiếu sáng lệch — độ sáng nền thay đổi chậm trên khắp tấm ảnh\n  tương phản thấp — hố bom sau nhiều năm chỉ còn là vệt tròn mờ\n\nHố bom được dựng theo hình thái thật: lòng hố tối hơn nền do đọng bóng và ẩm,\nviền hố sáng hơn do đất bị hất lên. Hình thái viền sáng lòng tối này chính là dấu\nhiệu mà người giải đoán ảnh dùng để nhận ra hố bom, nên mô hình cũng phải học\nđúng dấu hiệu đó.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import List, Tuple\n\nimport numpy as np\n\nfrom ..config import ImageryConfig\nfrom ..utils import get_logger\nfrom .geo import Grid\nfrom .sorties import Scene\n\nlogger = get_logger(__name__)\n\n\n@dataclass\nclass Tile:\n    \"\"\"Một tấm ảnh con cùng nhãn hố bom nằm trong nó.\"\"\"\n\n    image: np.ndarray\n    boxes: np.ndarray\n    origin_x_m: float\n    origin_y_m: float\n    gsd_m: float\n    tile_id: int\n    split: str\n\n    @property\n    def size_px(self) -> int:\n        return int(self.image.shape[0])\n\n\ndef _smooth2d(field: np.ndarray, sigma_px: float) -> np.ndarray:\n    if sigma_px <= 0:\n        return field\n    radius = max(1, int(3.0 * sigma_px))\n    offsets = np.arange(-radius, radius + 1, dtype=float)\n    kernel = np.exp(-0.5 * (offsets / sigma_px) ** 2)\n    kernel /= kernel.sum()\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 0, field)\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 1, out)\n    return out\n\n\nclass HistoricalImageryRenderer:\n    \"\"\"Kết xuất ảnh trinh sát lịch sử mô phỏng cho một vùng.\"\"\"\n\n    def __init__(self, cfg: ImageryConfig, rng: np.random.Generator):\n        self.cfg = cfg\n        self.rng = rng\n\n    def _background(self, size: int) -> np.ndarray:\n        \"\"\"Nền ảnh: kết cấu ruộng và thảm thực vật, cộng chiếu sáng không đều.\"\"\"\n        rng = self.rng\n\n        texture = _smooth2d(rng.normal(size=(size, size)), sigma_px=2.2)\n        texture = texture / (np.abs(texture).max() + 1e-9)\n\n        # Bờ thửa ruộng: các dải song song, đặc trưng của vùng canh tác.\n        angle = rng.uniform(0.0, np.pi)\n        yy, xx = np.mgrid[0:size, 0:size]\n        proj = xx * np.cos(angle) + yy * np.sin(angle)\n        period = rng.uniform(34.0, 110.0)\n        amplitude = rng.uniform(0.035, 0.105)\n        fields = amplitude * np.sin(2.0 * np.pi * proj / period)\n\n        # Mảng ruộng và vạt cây: các vùng sáng tối không đều ở bậc kích thước lớn.\n        patches = _smooth2d(rng.normal(size=(size, size)), sigma_px=size / 16.0)\n        patches = 0.11 * patches / (np.abs(patches).max() + 1e-9)\n\n        illumination = _smooth2d(rng.normal(size=(size, size)), sigma_px=size / 7.0)\n        illumination = illumination / (np.abs(illumination).max() + 1e-9)\n\n        img = (0.52 + 0.17 * texture + fields + patches\n               + self.cfg.illumination_sigma * illumination)\n        return img\n\n    def _draw_crater(\n        self, img: np.ndarray, cx: float, cy: float, diameter_px: float\n    ) -> None:\n        \"\"\"Vẽ một hố bom: lòng tối, viền sáng.\"\"\"\n        r = diameter_px / 2.0\n        pad = int(np.ceil(r * 1.9)) + 2\n        x0, x1 = int(max(0, cx - pad)), int(min(img.shape[1], cx + pad + 1))\n        y0, y1 = int(max(0, cy - pad)), int(min(img.shape[0], cy + pad + 1))\n        if x1 <= x0 or y1 <= y0:\n            return\n\n        yy, xx = np.mgrid[y0:y1, x0:x1]\n        dist = np.hypot(xx - cx, yy - cy)\n\n        # Lòng hố: tối dần vào tâm.\n        bowl = -0.30 * np.exp(-(dist ** 2) / (2.0 * (r * 0.55) ** 2))\n        # Viền hố: vành sáng quanh mép.\n        rim = 0.22 * np.exp(-((dist - r) ** 2) / (2.0 * (r * 0.30) ** 2))\n\n        # Biến dạng nhẹ để hố không tròn hoàn hảo, đúng như ngoài thực địa.\n        wobble = 1.0 + 0.12 * np.sin(4.0 * np.arctan2(yy - cy, xx - cx) + self.rng.uniform(0, 6.28))\n        img[y0:y1, x0:x1] += (bowl + rim) * wobble\n\n    def render_tile(\n        self,\n        scene: Scene,\n        origin_x_m: float,\n        origin_y_m: float,\n        tile_id: int,\n        split: str,\n    ) -> Tile:\n        size = self.cfg.tile_size_px\n        gsd = self.cfg.ground_sample_distance_m\n        extent = size * gsd\n\n        img = self._background(size)\n\n        sel = (\n            scene.impact_crater_visible\n            & (scene.impact_x >= origin_x_m)\n            & (scene.impact_x < origin_x_m + extent)\n            & (scene.impact_y >= origin_y_m)\n            & (scene.impact_y < origin_y_m + extent)\n        )\n\n        boxes = []\n        for x, y, d in zip(\n            scene.impact_x[sel],\n            scene.impact_y[sel],\n            scene.impact_crater_diameter_m[sel],\n        ):\n            px = (x - origin_x_m) / gsd\n            # Trục dọc của ảnh hướng xuống, ngược với trục bắc của mặt đất.\n            py = size - 1 - (y - origin_y_m) / gsd\n            dia_px = d / gsd\n            self._draw_crater(img, px, py, dia_px)\n            half = dia_px * 0.62\n            boxes.append([px - half, py - half, px + half, py + half])\n\n        # Nhiễu hạt phim, thêm sau khi đã vẽ đối tượng.\n        img = img + self.rng.normal(0.0, self.cfg.film_grain_sigma, size=img.shape)\n        img = np.clip(img, 0.0, 1.0)\n        img_u8 = (img * 255.0).astype(np.uint8)\n\n        box_arr = np.asarray(boxes, dtype=float).reshape(-1, 4)\n        if box_arr.size:\n            box_arr[:, [0, 2]] = np.clip(box_arr[:, [0, 2]], 0, size - 1)\n            box_arr[:, [1, 3]] = np.clip(box_arr[:, [1, 3]], 0, size - 1)\n            wide = (box_arr[:, 2] - box_arr[:, 0]) > 3.0\n            tall = (box_arr[:, 3] - box_arr[:, 1]) > 3.0\n            box_arr = box_arr[wide & tall]\n\n        return Tile(\n            image=img_u8,\n            boxes=box_arr,\n            origin_x_m=origin_x_m,\n            origin_y_m=origin_y_m,\n            gsd_m=gsd,\n            tile_id=tile_id,\n            split=split,\n        )\n\n\ndef build_tiles(scene: Scene, cfg: ImageryConfig, rng: np.random.Generator) -> List[Tile]:\n    \"\"\"Sinh toàn bộ tập ảnh con và chia tập theo khối không gian.\n\n    Việc chia tập cũng theo khối không gian chứ không ngẫu nhiên: các ảnh con của\n    cùng một vùng không được phép nằm ở cả tập huấn luyện lẫn tập kiểm tra.\n    \"\"\"\n    grid: Grid = scene.grid\n    extent = cfg.tile_size_px * cfg.ground_sample_distance_m\n    renderer = HistoricalImageryRenderer(cfg, rng)\n\n    max_x = max(1.0, grid.width_m - extent)\n    max_y = max(1.0, grid.height_m - extent)\n\n    # Chia vùng nghiên cứu thành ba dải dọc: huấn luyện, thẩm định, kiểm tra.\n    n_train = int(cfg.n_tiles * cfg.train_fraction)\n    n_val = int(cfg.n_tiles * cfg.val_fraction)\n    n_test = cfg.n_tiles - n_train - n_val\n\n    bounds = {\n        \"train\": (0.0, max_x * cfg.train_fraction),\n        \"val\": (max_x * cfg.train_fraction, max_x * (cfg.train_fraction + cfg.val_fraction)),\n        \"test\": (max_x * (cfg.train_fraction + cfg.val_fraction), max_x),\n    }\n    counts = {\"train\": n_train, \"val\": n_val, \"test\": n_test}\n\n    tiles: List[Tile] = []\n    tid = 0\n    for split, n in counts.items():\n        lo, hi = bounds[split]\n        if hi <= lo:\n            lo, hi = 0.0, max_x\n        for _ in range(max(0, n)):\n            ox = float(rng.uniform(lo, hi))\n            oy = float(rng.uniform(0.0, max_y))\n            tiles.append(renderer.render_tile(scene, ox, oy, tid, split))\n            tid += 1\n\n    n_boxes = sum(t.boxes.shape[0] for t in tiles)\n    logger.info(\n        \"Ảnh lịch sử: %d ảnh con (%d huấn luyện, %d thẩm định, %d kiểm tra) | %d hố bom có nhãn\",\n        len(tiles),\n        counts[\"train\"],\n        counts[\"val\"],\n        counts[\"test\"],\n        n_boxes,\n    )\n    return tiles\n", "src/demine/data/sorties.py": "\"\"\"Mô phỏng hồ sơ không kích và vật nổ còn sót lại.\n\nĐây là tầng nền của toàn bộ bài toán. Mô phỏng tái hiện đúng quan hệ nhân quả có\nthật ngoài thực địa, và chính quan hệ đó tạo ra bài toán học máy có ý nghĩa:\n\n  hồ sơ ghi chép  →  điểm rơi thật  →  nổ hoặc không nổ  →  hố bom hoặc vật nổ sót\n\nBa điểm quan trọng về mặt vật lý được mô hình hoá tường minh:\n\n1. Hồ sơ ghi chép lệch so với điểm rơi thật một khoảng đáng kể, và một phần hồ sơ\n   đã thất lạc hoàn toàn. Vì vậy hồ sơ là chỉ báo có nhiễu, không phải chân lý.\n\n2. Bom rơi xuống nền đất mềm có xác suất không nổ cao hơn nhiều, vì đầu nổ chạm\n   nổ cần lực cản đủ lớn mới kích hoạt.\n\n3. Hố bom chỉ hình thành ở nơi bom đã nổ. Nghĩa là hố bom nhìn thấy trên ảnh\n   là bằng chứng cho biết bom đã rơi ở đó, nhưng đồng thời cho biết chính quả bom\n   đó đã nổ rồi. Trên nền mềm, nơi nhiều bom không nổ nhất, lại chính là nơi ít\n   hố bom quan sát được nhất vì hố bị bồi lấp nhanh.\n\nĐiểm thứ ba là mấu chốt. Nó khiến bài toán không thể giải bằng một nguồn dữ liệu\nduy nhất, và làm cho việc hợp nhất ba nguồn trở thành đóng góp thực chất.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nfrom ..config import OrdnanceConfig, SortieConfig\nfrom ..utils import get_logger\nfrom .geo import Grid\nfrom .terrain import Terrain\n\nlogger = get_logger(__name__)\n\n# Khối lượng quy ước của một quả bom, ki-lô-gam. Dùng để quy đổi ra tải trọng,\n# đại lượng mà hồ sơ không kích ghi nhận.\nBOMB_MASS_KG = 340.0\n\n\n@dataclass\nclass Scene:\n    \"\"\"Toàn bộ trạng thái mô phỏng của vùng nghiên cứu.\"\"\"\n\n    grid: Grid\n    terrain: Terrain\n\n    impact_x: np.ndarray\n    impact_y: np.ndarray\n    impact_is_dud: np.ndarray\n    impact_crater_visible: np.ndarray\n    impact_crater_diameter_m: np.ndarray\n    impact_mission_id: np.ndarray\n\n    record_x: np.ndarray\n    record_y: np.ndarray\n    record_n_bombs: np.ndarray\n    record_mission_id: np.ndarray\n\n    uxo_count: np.ndarray = field(default=None)\n    crater_count: np.ndarray = field(default=None)\n    recorded_tonnage: np.ndarray = field(default=None)\n    true_tonnage: np.ndarray = field(default=None)\n\n    @property\n    def n_uxo(self) -> int:\n        return int(self.impact_is_dud.sum())\n\n    @property\n    def n_impacts(self) -> int:\n        return int(self.impact_x.size)\n\n\ndef _dud_probability(softness: np.ndarray, cfg: OrdnanceConfig) -> np.ndarray:\n    \"\"\"Xác suất một quả bom không nổ, theo độ mềm của nền đất tại điểm rơi.\"\"\"\n    multiplier = 1.0 + (cfg.soft_soil_dud_multiplier - 1.0) * softness\n    return np.clip(cfg.base_dud_rate * multiplier, 0.0, 0.85)\n\n\ndef _crater_visibility(softness: np.ndarray, cfg: OrdnanceConfig) -> np.ndarray:\n    \"\"\"Xác suất hố bom còn quan sát được trên ảnh, theo độ mềm của nền đất.\"\"\"\n    return (\n        cfg.crater_visible_rate_hard\n        + (cfg.crater_visible_rate_soft - cfg.crater_visible_rate_hard) * softness\n    )\n\n\ndef simulate_scene(\n    grid: Grid,\n    terrain: Terrain,\n    sortie_cfg: SortieConfig,\n    ordnance_cfg: OrdnanceConfig,\n    rng: np.random.Generator,\n) -> Scene:\n    \"\"\"Sinh toàn bộ phi vụ, điểm rơi, vật nổ còn sót và hồ sơ ghi chép.\"\"\"\n\n    xs, ys, mids = [], [], []\n\n    # Phân bố điểm ngắm không đồng đều trên lãnh thổ. Không kích trong chiến tranh\n    # bám theo mục tiêu quân sự — tuyến vận tải, bến vượt sông, nút giao thông —\n    # chứ không rải đều. Hệ quả là ô nhiễm bom mìn ngoài thực địa tập trung thành\n    # những hành lang rất đậm xen với những vùng gần như sạch, và chính cấu trúc\n    # tập trung đó là thứ làm cho bài toán xếp thứ tự ưu tiên có ý nghĩa.\n    target_field = (\n        np.exp(-terrain.dist_road_m / 700.0)\n        + 0.75 * np.exp(-terrain.dist_river_m / 900.0)\n        + 0.35 * np.exp(-terrain.dist_village_m / 1100.0)\n        + sortie_cfg.uniform_target_fraction\n    )\n    target_prob = (target_field / target_field.sum()).ravel()\n    ny_grid, nx_grid = grid.shape\n\n    for mission_id in range(sortie_cfg.n_missions):\n        # Mỗi phi vụ là một đường bay ngắn với loạt bom rải dọc theo hướng bay.\n        pick = int(rng.choice(target_prob.size, p=target_prob))\n        ty, tx = divmod(pick, nx_grid)\n        cx = (tx + rng.random()) * grid.cell\n        cy = (ty + rng.random()) * grid.cell\n        heading = rng.uniform(0.0, 2.0 * np.pi)\n        n_bombs = int(rng.integers(*sortie_cfg.bombs_per_mission))\n\n        # Khoảng cách giữa các quả trong loạt, mét.\n        spacing = rng.uniform(35.0, 75.0)\n        along = (np.arange(n_bombs) - (n_bombs - 1) / 2.0) * spacing\n\n        bx = cx + along * np.cos(heading)\n        by = cy + along * np.sin(heading)\n\n        bx = bx + rng.normal(0.0, sortie_cfg.impact_dispersion_m, size=n_bombs)\n        by = by + rng.normal(0.0, sortie_cfg.impact_dispersion_m, size=n_bombs)\n\n        xs.append(bx)\n        ys.append(by)\n        mids.append(np.full(n_bombs, mission_id, dtype=int))\n\n    impact_x = np.concatenate(xs)\n    impact_y = np.concatenate(ys)\n    mission_id = np.concatenate(mids)\n\n    keep = (\n        (impact_x >= 0.0)\n        & (impact_x < grid.width_m)\n        & (impact_y >= 0.0)\n        & (impact_y < grid.height_m)\n    )\n    impact_x, impact_y, mission_id = impact_x[keep], impact_y[keep], mission_id[keep]\n\n    col, row = grid.xy_to_index(impact_x, impact_y)\n    softness = terrain.soil_softness[row, col]\n\n    is_dud = rng.random(impact_x.size) < _dud_probability(softness, ordnance_cfg)\n\n    visible_prob = _crater_visibility(softness, ordnance_cfg)\n    exploded = ~is_dud\n    crater_visible = exploded & (rng.random(impact_x.size) < visible_prob)\n\n    lo, hi = ordnance_cfg.crater_diameter_m\n    crater_diameter = np.where(\n        crater_visible, rng.uniform(lo, hi, size=impact_x.size), 0.0\n    )\n\n    # Hồ sơ ghi chép: một điểm cho mỗi phi vụ, kèm sai số định vị, và một phần\n    # phi vụ bị thất lạc hoàn toàn khỏi hồ sơ.\n    rec_x, rec_y, rec_n, rec_id = [], [], [], []\n    for mid in np.unique(mission_id):\n        if rng.random() < sortie_cfg.missing_record_fraction:\n            continue\n        sel = mission_id == mid\n        cx = float(impact_x[sel].mean())\n        cy = float(impact_y[sel].mean())\n        rec_x.append(cx + rng.normal(0.0, sortie_cfg.record_position_error_m))\n        rec_y.append(cy + rng.normal(0.0, sortie_cfg.record_position_error_m))\n        rec_n.append(int(sel.sum()))\n        rec_id.append(int(mid))\n\n    scene = Scene(\n        grid=grid,\n        terrain=terrain,\n        impact_x=impact_x,\n        impact_y=impact_y,\n        impact_is_dud=is_dud,\n        impact_crater_visible=crater_visible,\n        impact_crater_diameter_m=crater_diameter,\n        impact_mission_id=mission_id,\n        record_x=np.asarray(rec_x, dtype=float),\n        record_y=np.asarray(rec_y, dtype=float),\n        record_n_bombs=np.asarray(rec_n, dtype=int),\n        record_mission_id=np.asarray(rec_id, dtype=int),\n    )\n\n    scene.uxo_count = grid.accumulate(impact_x[is_dud], impact_y[is_dud])\n    scene.crater_count = grid.accumulate(\n        impact_x[crater_visible], impact_y[crater_visible]\n    )\n    scene.true_tonnage = grid.accumulate(\n        impact_x, impact_y, np.full(impact_x.size, BOMB_MASS_KG / 1000.0)\n    )\n    scene.recorded_tonnage = grid.accumulate(\n        scene.record_x,\n        scene.record_y,\n        scene.record_n_bombs * BOMB_MASS_KG / 1000.0,\n    )\n\n    logger.info(\n        \"Mô phỏng: %d điểm rơi | %d vật nổ còn sót (%.1f%%) | %d hố bom quan sát được | \"\n        \"%d phi vụ có hồ sơ trên tổng %d\",\n        scene.n_impacts,\n        scene.n_uxo,\n        100.0 * scene.n_uxo / max(1, scene.n_impacts),\n        int(crater_visible.sum()),\n        scene.record_x.size,\n        sortie_cfg.n_missions,\n    )\n    return scene\n", "src/demine/data/terrain.py": "\"\"\"Sinh địa hình, thổ nhưỡng và hiện trạng sử dụng đất của vùng nghiên cứu.\n\nCác lớp này đóng hai vai trò trong bài toán. Thứ nhất, chúng chi phối cơ chế vật\nlý: nền đất mềm làm tăng tỉ lệ bom không nổ và làm hố bom bị bồi lấp nhanh hơn.\nThứ hai, chúng quyết định mức độ phơi nhiễm của con người: người dân chỉ gặp tai\nnạn ở nơi họ thực sự lui tới.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom ..config import GridConfig, TerrainConfig\nfrom .geo import Grid\n\n\ndef _smooth(field: np.ndarray, sigma_cells: float) -> np.ndarray:\n    \"\"\"Làm trơn trường ngẫu nhiên bằng bộ lọc Gauss tách được.\"\"\"\n    if sigma_cells <= 0:\n        return field\n    radius = max(1, int(3.0 * sigma_cells))\n    offsets = np.arange(-radius, radius + 1, dtype=float)\n    kernel = np.exp(-0.5 * (offsets / sigma_cells) ** 2)\n    kernel /= kernel.sum()\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 0, field)\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 1, out)\n    return out\n\n\ndef _distance_transform(mask: np.ndarray, cell_size: float) -> np.ndarray:\n    \"\"\"Khoảng cách xấp xỉ tới điểm gần nhất của mặt nạ, tính bằng mét.\n\n    Dùng thuật toán quét hai lượt theo khoảng cách chamfer, đủ chính xác cho mục\n    đích xây dựng đặc trưng và không phụ thuộc thư viện ngoài.\n    \"\"\"\n    big = 1.0e9\n    dist = np.where(mask, 0.0, big).astype(float)\n    ny, nx = dist.shape\n    d1, d2 = 1.0, np.sqrt(2.0)\n\n    for y in range(ny):\n        for x in range(nx):\n            best = dist[y, x]\n            if y > 0:\n                best = min(best, dist[y - 1, x] + d1)\n                if x > 0:\n                    best = min(best, dist[y - 1, x - 1] + d2)\n                if x < nx - 1:\n                    best = min(best, dist[y - 1, x + 1] + d2)\n            if x > 0:\n                best = min(best, dist[y, x - 1] + d1)\n            dist[y, x] = best\n\n    for y in range(ny - 1, -1, -1):\n        for x in range(nx - 1, -1, -1):\n            best = dist[y, x]\n            if y < ny - 1:\n                best = min(best, dist[y + 1, x] + d1)\n                if x > 0:\n                    best = min(best, dist[y + 1, x - 1] + d2)\n                if x < nx - 1:\n                    best = min(best, dist[y + 1, x + 1] + d2)\n            if x < nx - 1:\n                best = min(best, dist[y, x + 1] + d1)\n            dist[y, x] = best\n\n    return dist * cell_size\n\n\n@dataclass\nclass Terrain:\n    \"\"\"Toàn bộ các lớp nền của vùng nghiên cứu, cùng kích thước với lưới.\"\"\"\n\n    elevation_m: np.ndarray\n    slope_deg: np.ndarray\n    soil_softness: np.ndarray\n    is_farmland: np.ndarray\n    is_forest: np.ndarray\n    dist_village_m: np.ndarray\n    dist_road_m: np.ndarray\n    dist_river_m: np.ndarray\n    village_xy: np.ndarray\n    exposure: np.ndarray\n\n    @property\n    def shape(self):\n        return self.elevation_m.shape\n\n\ndef build_terrain(grid: Grid, cfg: TerrainConfig, rng: np.random.Generator) -> Terrain:\n    ny, nx = grid.shape\n\n    # Địa hình: trường ngẫu nhiên làm trơn ở hai bậc kích thước, tạo ra dạng đồi\n    # thấp xen thung lũng đặc trưng của vùng trung du.\n    base = _smooth(rng.normal(size=(ny, nx)), sigma_cells=14.0)\n    detail = _smooth(rng.normal(size=(ny, nx)), sigma_cells=4.0)\n    elevation = 120.0 * (base / (np.abs(base).max() + 1e-9)) + 25.0 * detail\n    elevation -= elevation.min()\n\n    gy, gx = np.gradient(elevation, grid.cell)\n    slope = np.rad2deg(np.arctan(np.hypot(gx, gy)))\n\n    # Sông: đường gãy khúc chạy qua vùng thấp.\n    river_mask = np.zeros((ny, nx), dtype=bool)\n    for _ in range(cfg.n_rivers):\n        y = rng.integers(ny // 5, 4 * ny // 5)\n        for x in range(nx):\n            y += int(rng.integers(-1, 2))\n            y = int(np.clip(y, 1, ny - 2))\n            river_mask[y - 1 : y + 2, x] = True\n\n    # Đường bộ: đoạn thẳng nối hai biên.\n    road_mask = np.zeros((ny, nx), dtype=bool)\n    for _ in range(cfg.n_roads):\n        y0, y1 = rng.integers(0, ny, size=2)\n        x0, x1 = 0, nx - 1\n        n = max(nx, ny) * 2\n        xs = np.linspace(x0, x1, n).astype(int)\n        ys = np.linspace(y0, y1, n).astype(int)\n        ys = np.clip(ys, 0, ny - 1)\n        road_mask[ys, xs] = True\n\n    # Làng: cụm dân cư đặt ở nơi độ dốc thấp và gần nguồn nước.\n    village_mask = np.zeros((ny, nx), dtype=bool)\n    villages = []\n    dist_river_tmp = _distance_transform(river_mask, grid.cell)\n    favour = np.exp(-dist_river_tmp / 2500.0) * np.exp(-slope / 6.0)\n    flat = favour.ravel() / favour.sum()\n    picks = rng.choice(favour.size, size=cfg.n_villages, replace=False, p=flat)\n    for p in picks:\n        vy, vx = divmod(int(p), nx)\n        village_mask[\n            max(0, vy - 2) : vy + 3, max(0, vx - 2) : vx + 3\n        ] = True\n        villages.append((vx, vy))\n\n    dist_village = _distance_transform(village_mask, grid.cell)\n    dist_road = _distance_transform(road_mask, grid.cell)\n    dist_river = _distance_transform(river_mask, grid.cell)\n\n    # Thổ nhưỡng: đất mềm tập trung ở vùng thấp ven sông và nơi ít dốc.\n    softness = _smooth(rng.normal(size=(ny, nx)), sigma_cells=9.0)\n    softness = (softness - softness.min()) / (np.ptp(softness) + 1e-9)\n    softness = 0.45 * softness + 0.35 * np.exp(-dist_river / 1800.0)\n    softness += 0.20 * np.exp(-slope / 4.0)\n    softness = np.clip(softness, 0.0, 1.0)\n\n    # Đất canh tác: gần làng, ít dốc.\n    suitability = np.exp(-dist_village / 1600.0) * np.exp(-slope / 7.0)\n    threshold = np.quantile(suitability, 1.0 - cfg.farmland_fraction)\n    is_farmland = suitability >= threshold\n    is_forest = (~is_farmland) & (slope > np.quantile(slope, 0.55))\n\n    # Mức độ phơi nhiễm của con người: tổ hợp của việc đất có được canh tác hay\n    # không, khoảng cách tới khu dân cư và tới đường đi lại.\n    exposure = (\n        1.00 * is_farmland.astype(float)\n        + 0.85 * np.exp(-dist_village / 1200.0)\n        + 0.35 * np.exp(-dist_road / 900.0)\n    )\n    exposure = exposure / (exposure.max() + 1e-9)\n\n    return Terrain(\n        elevation_m=elevation,\n        slope_deg=slope,\n        soil_softness=softness,\n        is_farmland=is_farmland,\n        is_forest=is_forest,\n        dist_village_m=dist_village,\n        dist_road_m=dist_road,\n        dist_river_m=dist_river,\n        village_xy=np.asarray(villages, dtype=float),\n        exposure=exposure,\n    )\n", "src/demine/detect/__init__.py": "\"\"\"Tầng hai — phát hiện hố bom trên ảnh vệ tinh lịch sử.\"\"\"\n\nfrom .interface import BaseDetector, DetectionOutput, build_detector, select_backend\n\n__all__ = [\"BaseDetector\", \"DetectionOutput\", \"build_detector\", \"select_backend\"]\n", "src/demine/detect/interface.py": "\"\"\"Giao diện thống nhất cho các bộ phát hiện phương tiện.\n\nHệ thống hỗ trợ hai phương án thực thi để bảo đảm chạy được trong mọi điều kiện\ncủa môi trường Kaggle:\n\n* **Ultralytics YOLO** — phương án chính, đúng như mô tả trong đề xuất. Khai thác\n  được cả hai card T4 thông qua huấn luyện phân tán.\n* **Torchvision Faster R-CNN** — phương án dự phòng, dùng khi không cài đặt được\n  gói bổ sung. Torchvision luôn có sẵn trong ảnh Python của Kaggle nên phương án\n  này không cần kết nối mạng.\n\nCả hai đều phơi bày cùng một giao diện nên các tầng phía sau không cần biết\nphương án nào đang được dùng.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\n\n@dataclass\nclass DetectionOutput:\n    \"\"\"Kết quả phát hiện trên một ảnh.\"\"\"\n\n    image_id: str\n    boxes: np.ndarray   # (N, 4) theo thứ tự x1, y1, x2, y2\n    scores: np.ndarray  # (N,)\n\n    def filter_by_score(self, threshold: float) -> \"DetectionOutput\":\n        keep = self.scores >= threshold\n        return DetectionOutput(self.image_id, self.boxes[keep], self.scores[keep])\n\n\nclass BaseDetector(ABC):\n    \"\"\"Hợp đồng chung cho mọi bộ phát hiện.\"\"\"\n\n    name: str = \"base\"\n    devices: List[int]\n\n    @abstractmethod\n    def train(self, data_yaml: Path, cfg) -> Dict:\n        \"\"\"Huấn luyện mô hình. Trả về từ điển thông tin của lần chạy.\"\"\"\n\n    @abstractmethod\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        \"\"\"Suy luận trên danh sách ảnh.\"\"\"\n\n    @abstractmethod\n    def load(self, weights: Path) -> \"BaseDetector\":\n        \"\"\"Nạp trọng số đã huấn luyện.\"\"\"\n\n\ndef select_backend(preference: str = \"auto\") -> str:\n    \"\"\"Chọn phương án thực thi phù hợp với môi trường hiện tại.\"\"\"\n    from ..utils import package_available\n\n    if preference in (\"ultralytics\", \"torchvision\"):\n        return preference\n    if package_available(\"ultralytics\"):\n        return \"ultralytics\"\n    return \"torchvision\"\n\n\ndef build_detector(cfg, backend: Optional[str] = None) -> BaseDetector:\n    \"\"\"Khởi tạo bộ phát hiện theo cấu hình.\"\"\"\n    from ..utils import get_logger, resolve_devices\n\n    log = get_logger(\"detect\")\n    chosen = select_backend(backend or cfg.detect.backend)\n    devices = resolve_devices(cfg.detect.devices, allow_multi=cfg.multi_gpu)\n\n    if chosen == \"ultralytics\":\n        from .yolo_backend import YOLODetector\n\n        log.info(\"Phương án phát hiện: Ultralytics YOLO | GPU: %s\", devices or \"CPU\")\n        return YOLODetector(devices=devices, cfg=cfg)\n\n    from .torchvision_backend import TorchvisionDetector\n\n    log.info(\n        \"Phương án phát hiện: Torchvision Faster R-CNN | GPU: %s\",\n        devices[:1] or \"CPU\",\n    )\n    return TorchvisionDetector(devices=devices[:1], cfg=cfg)\n", "src/demine/detect/torchvision_backend.py": "\"\"\"Bộ phát hiện hố bom dự phòng dựa trên Faster R-CNN của Torchvision.\n\nPhương án này tồn tại để bảo đảm pipeline chạy được ngay cả khi notebook bị ngắt\nkết nối mạng và không cài đặt được gói bổ sung, vì Torchvision luôn có sẵn trong\nảnh Python của Kaggle. Mô hình được khởi tạo từ đầu, không dùng trọng số huấn\nluyện trước, nên hoàn toàn không phụ thuộc vào việc tải tệp từ Internet.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence, Tuple\n\nimport numpy as np\n\nfrom ..utils import get_logger\nfrom .interface import BaseDetector, DetectionOutput\n\nLOG = get_logger(\"detect.torchvision\")\n\n\nclass YoloFormatDataset:\n    \"\"\"Đọc bộ dữ liệu bố trí theo quy ước YOLO và trả về định dạng Torchvision.\"\"\"\n\n    def __init__(self, root: Path, split: str, size: int) -> None:\n        import torch  # noqa: F401\n\n        self.img_dir = Path(root) / \"images\" / split\n        self.lbl_dir = Path(root) / \"labels\" / split\n        self.size = size\n        self.files = sorted(self.img_dir.glob(\"*.png\"))\n        if not self.files:\n            raise FileNotFoundError(f\"Không tìm thấy ảnh trong {self.img_dir}\")\n\n    def __len__(self) -> int:\n        return len(self.files)\n\n    def __getitem__(self, idx: int):\n        import torch\n\n        from ..data.dataset import load_image\n\n        path = self.files[idx]\n        img = load_image(path).astype(np.float32) / 255.0\n        # Nhân bản kênh xám thành ba kênh cho tương thích với mạng nền tiêu chuẩn\n        tensor = torch.from_numpy(img).unsqueeze(0).repeat(3, 1, 1)\n\n        lbl_path = self.lbl_dir / f\"{path.stem}.txt\"\n        boxes: List[List[float]] = []\n        if lbl_path.exists():\n            for line in lbl_path.read_text(encoding=\"utf-8\").splitlines():\n                parts = line.split()\n                if len(parts) != 5:\n                    continue\n                _, xc, yc, w, h = map(float, parts)\n                x1 = (xc - w / 2) * self.size\n                y1 = (yc - h / 2) * self.size\n                x2 = (xc + w / 2) * self.size\n                y2 = (yc + h / 2) * self.size\n                if x2 - x1 > 1 and y2 - y1 > 1:\n                    boxes.append([x1, y1, x2, y2])\n\n        if boxes:\n            boxes_t = torch.tensor(boxes, dtype=torch.float32)\n            labels_t = torch.ones((len(boxes),), dtype=torch.int64)\n        else:\n            boxes_t = torch.zeros((0, 4), dtype=torch.float32)\n            labels_t = torch.zeros((0,), dtype=torch.int64)\n\n        target = {\"boxes\": boxes_t, \"labels\": labels_t,\n                  \"image_id\": torch.tensor([idx])}\n        return tensor, target, path.stem\n\n\ndef _collate(batch):\n    imgs, targets, ids = zip(*batch)\n    return list(imgs), list(targets), list(ids)\n\n\nclass TorchvisionDetector(BaseDetector):\n    name = \"torchvision\"\n\n    def __init__(self, devices: List[int], cfg) -> None:\n        self.devices = list(devices)\n        self.cfg = cfg\n        self.model = None\n        self.weights_path: Optional[Path] = None\n\n    def _device(self):\n        from ..utils import torch_device\n\n        return torch_device(self.devices[0] if self.devices else 0)\n\n    def _build_model(self):\n        import torchvision\n        from torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n        from torchvision.models.detection.rpn import AnchorGenerator\n\n        # Hố bom chỉ chiếm chừng mười đến hai mươi điểm ảnh nên bộ sinh neo mặc\n        # định là quá lớn. Dải neo dưới đây được thiết kế lại cho đúng kích thước\n        # thực tế của đối tượng, và tỉ lệ cạnh tập trung quanh một vì hố bom gần\n        # tròn.\n        anchor_gen = AnchorGenerator(\n            sizes=((6,), (12,), (20,), (32,), (56,)),\n            aspect_ratios=((0.75, 1.0, 1.33),) * 5,\n        )\n        model = torchvision.models.detection.fasterrcnn_resnet50_fpn(\n            weights=None,\n            weights_backbone=None,\n            num_classes=2,            # nền và hố bom\n            rpn_anchor_generator=anchor_gen,\n            min_size=self.cfg.detect.image_size,\n            max_size=self.cfg.detect.image_size,\n            box_detections_per_img=64,\n        )\n        return model\n\n    # -- Huấn luyện --------------------------------------------------------\n    def train(self, data_yaml: Path, cfg=None) -> Dict:\n        import torch\n        from torch.utils.data import DataLoader\n\n        cfg = cfg or self.cfg\n        root = Path(data_yaml).parent\n        device = self._device()\n\n        train_ds = YoloFormatDataset(root, \"train\", cfg.imagery.tile_size_px)\n\n        # Faster R-CNN tiêu tốn bộ nhớ hơn YOLO nên giảm kích thước lô\n        batch = max(2, cfg.detect.batch_size // 8)\n        train_dl = DataLoader(\n            train_ds, batch_size=batch, shuffle=True,\n            num_workers=2, collate_fn=_collate,\n            pin_memory=(device.type == \"cuda\"),\n        )\n\n        self.model = self._build_model().to(device)\n        params = [p for p in self.model.parameters() if p.requires_grad]\n        optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer, T_max=max(1, cfg.detect.epochs)\n        )\n        # Độ chính xác hỗn hợp chỉ được bật trên CUDA. Metal và CPU chạy ở độ chính\n        # xác đơn; bật cưỡng bức sẽ gây lỗi hoặc làm chậm chứ không nhanh hơn.\n        use_amp = device.type == \"cuda\"\n        scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)\n\n        LOG.info(\n            \"Huấn luyện Faster R-CNN: %d chu kỳ, lô %d, thiết bị %s\",\n            cfg.detect.epochs, batch, device,\n        )\n\n        for epoch in range(cfg.detect.epochs):\n            self.model.train()\n            running = 0.0\n            for imgs, targets, _ in train_dl:\n                imgs = [i.to(device, non_blocking=True) for i in imgs]\n                targets = [\n                    {k: v.to(device) for k, v in t.items()} for t in targets\n                ]\n                optimizer.zero_grad(set_to_none=True)\n                with torch.amp.autocast(\"cuda\", enabled=use_amp):\n                    loss_dict = self.model(imgs, targets)\n                    loss = sum(loss_dict.values())\n                scaler.scale(loss).backward()\n                scaler.step(optimizer)\n                scaler.update()\n                running += float(loss.detach())\n            scheduler.step()\n            LOG.info(\n                \"  chu kỳ %2d/%d — hàm mất mát trung bình %.4f\",\n                epoch + 1, cfg.detect.epochs, running / max(1, len(train_dl)),\n            )\n\n        out_dir = Path(cfg.output_dir) / \"runs\" / \"frcnn_crater\" / \"weights\"\n        out_dir.mkdir(parents=True, exist_ok=True)\n        self.weights_path = out_dir / \"best.pt\"\n        torch.save(self.model.state_dict(), self.weights_path)\n        LOG.info(\"Đã lưu trọng số: %s\", self.weights_path)\n\n        return {\n            \"backend\": self.name,\n            \"weights\": str(self.weights_path),\n            \"devices\": self.devices,\n            \"epochs\": cfg.detect.epochs,\n        }\n\n    # -- Nạp trọng số ------------------------------------------------------\n    def load(self, weights: Path) -> \"TorchvisionDetector\":\n        import torch\n\n        self.model = self._build_model()\n        state = torch.load(str(weights), map_location=\"cpu\")\n        self.model.load_state_dict(state)\n        self.model.to(self._device()).eval()\n        self.weights_path = Path(weights)\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        import torch\n\n        from ..data.dataset import load_image\n\n        if self.model is None:\n            raise RuntimeError(\"Chưa nạp mô hình. Gọi train() hoặc load() trước.\")\n\n        device = self._device()\n        self.model.eval()\n        outs: List[DetectionOutput] = []\n        paths = [Path(p) for p in image_paths]\n        batch = 8\n\n        with torch.no_grad():\n            for i in range(0, len(paths), batch):\n                chunk = paths[i : i + batch]\n                tensors = []\n                for p in chunk:\n                    arr = load_image(p).astype(np.float32) / 255.0\n                    t = torch.from_numpy(arr).unsqueeze(0).repeat(3, 1, 1)\n                    tensors.append(t.to(device))\n                preds = self.model(tensors)\n                for p, pr in zip(chunk, preds):\n                    boxes = pr[\"boxes\"].cpu().numpy().astype(np.float32)\n                    scores = pr[\"scores\"].cpu().numpy().astype(np.float32)\n                    keep = scores >= conf\n                    outs.append(DetectionOutput(p.stem, boxes[keep], scores[keep]))\n        return outs\n\n    def predict_sharded(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        return self.predict(image_paths, conf)\n", "src/demine/detect/yolo_backend.py": "\"\"\"Bộ phát hiện hố bom dựa trên Ultralytics YOLO, khai thác đồng thời hai GPU T4.\n\nHuấn luyện phân tán được kích hoạt bằng cách truyền danh sách nhiều thiết bị cho\ntham số ``device``. Ultralytics tự khởi tạo tiến trình phân tán ở phía sau; để quá\ntrình này ổn định trên Kaggle, việc huấn luyện nên được gọi từ một tiến trình con\nđộc lập thay vì gọi trực tiếp trong nhân của notebook. Tệp\n``scripts/train_detector.py`` đảm nhiệm vai trò đó.\n\nCấu hình tăng cường dữ liệu được điều chỉnh riêng cho ảnh trinh sát lịch sử: ảnh\nđơn sắc nên tắt toàn bộ phép biến đổi màu; hố bom không có hướng ưu tiên nên xoay\ntoàn dải và lật theo cả hai trục đều hợp lệ; hố bom là đối tượng nhỏ nên hạn chế\nbiên độ co giãn để không làm mất đối tượng.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Sequence\n\nimport numpy as np\n\nfrom ..utils import get_logger\nfrom .interface import BaseDetector, DetectionOutput\n\nLOG = get_logger(\"detect.yolo\")\n\n\ndef resolve_model_spec(preferred: str = \"yolo11n.pt\") -> str:\n    \"\"\"Chọn điểm khởi tạo mô hình phù hợp với điều kiện kết nối mạng.\n\n    Trọng số đã huấn luyện trước cần tải về từ máy chủ của Ultralytics. Khi\n    notebook chạy ở chế độ ngắt mạng, hệ thống chuyển sang khởi tạo mô hình từ tệp\n    mô tả kiến trúc — quá trình huấn luyện vẫn diễn ra bình thường, chỉ cần thêm\n    một số chu kỳ để hội tụ.\n    \"\"\"\n    local = Path(preferred)\n    if local.exists():\n        return str(local)\n\n    try:\n        import urllib.request\n\n        urllib.request.urlopen(\"https://github.com\", timeout=6)\n        return preferred\n    except Exception:\n        fallback = preferred.replace(\".pt\", \".yaml\")\n        LOG.warning(\n            \"Không có kết nối mạng: khởi tạo mô hình từ kiến trúc %s thay vì \"\n            \"trọng số huấn luyện trước.\",\n            fallback,\n        )\n        return fallback\n\n\nclass YOLODetector(BaseDetector):\n    name = \"ultralytics\"\n\n    def __init__(self, devices: List[int], cfg) -> None:\n        self.devices = list(devices)\n        self.cfg = cfg\n        self.model = None\n        self.weights_path: Optional[Path] = None\n\n    # -- Huấn luyện --------------------------------------------------------\n    def train(self, data_yaml: Path, cfg=None) -> Dict:\n        from ultralytics import YOLO\n\n        cfg = cfg or self.cfg\n        spec = resolve_model_spec(cfg.detect.model_spec)\n        self.model = YOLO(spec)\n\n        from ..utils import probe_devices\n\n        kind = probe_devices().kind\n        if kind == \"mps\":\n            device_arg = \"mps\"\n        elif len(self.devices) > 1:\n            device_arg = self.devices\n        else:\n            device_arg = self.devices[0] if self.devices else \"cpu\"\n        LOG.info(\"Bắt đầu huấn luyện YOLO trên thiết bị: %s\", device_arg)\n\n        # Ultralytics hiểu tham số project theo đường dẫn tương đối với thư mục\n        # cấu hình riêng của nó, nên nếu truyền đường dẫn tương đối thì kết quả sẽ\n        # rơi vào một chỗ khác hẳn với chỗ ta mong đợi. Luôn truyền đường dẫn tuyệt\n        # đối để trọng số nằm đúng nơi các bước sau đi tìm.\n        runs_dir = Path(cfg.output_dir).resolve() / \"runs\"\n        runs_dir.mkdir(parents=True, exist_ok=True)\n\n        self.model.train(\n            data=str(data_yaml),\n            epochs=cfg.detect.epochs,\n            imgsz=cfg.detect.image_size,\n            batch=cfg.detect.batch_size,\n            workers=cfg.detect.workers,\n            device=device_arg,\n            project=str(runs_dir),\n            name=\"yolo_crater\",\n            exist_ok=True,\n            seed=cfg.detect.seed,\n            pretrained=spec.endswith(\".pt\"),\n            verbose=True,\n            plots=False,\n            # Ảnh trinh sát là ảnh đơn sắc: tắt mọi phép tăng cường theo màu.\n            hsv_h=0.0,\n            hsv_s=0.0,\n            hsv_v=0.30,\n            # Hố bom không có hướng ưu tiên nên xoay và lật tự do.\n            degrees=180.0,\n            fliplr=0.5,\n            flipud=0.5,\n            # Hố bom là đối tượng nhỏ: giữ biên độ co giãn ở mức vừa phải.\n            mosaic=0.5,\n            mixup=0.0,\n            translate=0.06,\n            scale=0.25,\n        )\n\n        best = runs_dir / \"yolo_crater\" / \"weights\" / \"best.pt\"\n        if not best.exists():\n            # Phòng trường hợp phiên bản Ultralytics đặt kết quả ở nơi khác: tìm\n            # tệp trọng số mới nhất thay vì báo là không có.\n            found = sorted(\n                runs_dir.rglob(\"weights/best.pt\"),\n                key=lambda q: q.stat().st_mtime,\n                reverse=True,\n            )\n            if not found:\n                found = sorted(\n                    Path.cwd().rglob(\"yolo_crater*/weights/best.pt\"),\n                    key=lambda q: q.stat().st_mtime,\n                    reverse=True,\n                )\n            best = found[0] if found else best\n        self.weights_path = best if best.exists() else None\n        return {\n            \"backend\": self.name,\n            \"weights\": str(best),\n            \"devices\": self.devices,\n            \"epochs\": cfg.detect.epochs,\n        }\n\n    # -- Nạp trọng số ------------------------------------------------------\n    def load(self, weights: Path) -> \"YOLODetector\":\n        from ultralytics import YOLO\n\n        self.model = YOLO(str(weights))\n        self.weights_path = Path(weights)\n        LOG.info(\"Đã nạp trọng số: %s\", weights)\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def predict(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        if self.model is None:\n            raise RuntimeError(\"Chưa nạp mô hình. Gọi train() hoặc load() trước.\")\n\n        from ..utils import probe_devices\n\n        kind = probe_devices().kind\n        device = \"mps\" if kind == \"mps\" else (self.devices[0] if self.devices else \"cpu\")\n        outs: List[DetectionOutput] = []\n        paths = [Path(p) for p in image_paths]\n\n        for i in range(0, len(paths), 32):\n            chunk = paths[i : i + 32]\n            results = self.model.predict(\n                [str(p) for p in chunk],\n                conf=conf,\n                imgsz=self.cfg.detect.image_size,\n                device=device,\n                verbose=False,\n            )\n            for p, r in zip(chunk, results):\n                outs.append(_to_output(p, r))\n        return outs\n\n    # -- Suy luận phân tán trên hai GPU ------------------------------------\n    def predict_sharded(\n        self, image_paths: Sequence[Path], conf: float = 0.25\n    ) -> List[DetectionOutput]:\n        \"\"\"Chia đôi khối lượng suy luận cho hai GPU nhằm rút ngắn thời gian.\"\"\"\n        if len(self.devices) < 2:\n            return self.predict(image_paths, conf)\n\n        import threading\n\n        paths = [Path(p) for p in image_paths]\n        mid = len(paths) // 2\n        shards = [paths[:mid], paths[mid:]]\n        results: Dict[int, List[DetectionOutput]] = {}\n\n        def worker(idx: int, subset: Sequence[Path], dev: int) -> None:\n            from ultralytics import YOLO\n\n            local = YOLO(str(self.weights_path)) if self.weights_path else self.model\n            outs: List[DetectionOutput] = []\n            for i in range(0, len(subset), 32):\n                chunk = subset[i : i + 32]\n                rs = local.predict(\n                    [str(p) for p in chunk],\n                    conf=conf,\n                    imgsz=self.cfg.detect.image_size,\n                    device=dev,\n                    verbose=False,\n                )\n                for p, r in zip(chunk, rs):\n                    outs.append(_to_output(p, r))\n            results[idx] = outs\n\n        threads = [\n            threading.Thread(target=worker, args=(i, shards[i], self.devices[i]))\n            for i in range(2)\n        ]\n        for t in threads:\n            t.start()\n        for t in threads:\n            t.join()\n\n        return results.get(0, []) + results.get(1, [])\n\n\ndef _to_output(path: Path, result) -> DetectionOutput:\n    if result.boxes is None or len(result.boxes) == 0:\n        return DetectionOutput(\n            path.stem, np.zeros((0, 4), np.float32), np.zeros((0,), np.float32)\n        )\n    return DetectionOutput(\n        path.stem,\n        result.boxes.xyxy.cpu().numpy().astype(np.float32),\n        result.boxes.conf.cpu().numpy().astype(np.float32),\n    )\n", "src/demine/risk/__init__.py": "\"\"\"Tầng ba — mô hình nguy cơ hợp nhất.\"\"\"\n\nfrom .features import FEATURE_NAMES, FeatureTable, build_features\nfrom .model import RiskModel, permutation_importance\n\n__all__ = [\n    \"FEATURE_NAMES\",\n    \"FeatureTable\",\n    \"build_features\",\n    \"RiskModel\",\n    \"permutation_importance\",\n]\n", "src/demine/risk/features.py": "\"\"\"Tầng ba, phần một — xây dựng tập đặc trưng theo từng ô lưới.\n\nBốn nhóm đặc trưng, đúng như trình bày trong đề xuất:\n\n  nhóm lịch sử   — quy chiếu hồ sơ không kích về lưới qua hạt nhân lan toả\n  nhóm quan sát  — mật độ hố bom phát hiện được trên ảnh vệ tinh lịch sử\n  nhóm địa hình  — độ cao, độ dốc, độ mềm của nền đất, khoảng cách tới dòng chảy\n  nhóm hiện trạng— lớp phủ, khoảng cách tới khu dân cư và tới đường giao thông\n\nĐiểm cần lưu ý về phương pháp: hồ sơ không kích ghi một điểm cho mỗi phi vụ, kèm\nsai số định vị hàng trăm mét. Nếu dồn thẳng điểm đó vào một ô lưới thì thông tin\nbị đặt sai chỗ. Vì vậy mỗi bản ghi được lan toả ra chung quanh bằng một hạt nhân\nGauss có độ rộng đúng bằng sai số định vị đã biết. Đây là cách mô hình hoá tường\nminh bất định thay vì giả vờ rằng nó không tồn tại.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import List\n\nimport numpy as np\n\nfrom ..config import RiskConfig\nfrom ..data.geo import Grid\nfrom ..data.sorties import Scene\n\n\ndef gaussian_spread(field: np.ndarray, sigma_cells: float) -> np.ndarray:\n    \"\"\"Lan toả một trường theo hạt nhân Gauss tách được, bảo toàn tổng.\"\"\"\n    if sigma_cells <= 0:\n        return field.copy()\n    radius = max(1, int(3.0 * sigma_cells))\n    offsets = np.arange(-radius, radius + 1, dtype=float)\n    kernel = np.exp(-0.5 * (offsets / sigma_cells) ** 2)\n    kernel /= kernel.sum()\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 0, field)\n    out = np.apply_along_axis(lambda m: np.convolve(m, kernel, mode=\"same\"), 1, out)\n    return out\n\n\ndef neighbourhood_sum(field: np.ndarray, radius_cells: int) -> np.ndarray:\n    \"\"\"Tổng giá trị trong lân cận vuông bán kính cho trước.\"\"\"\n    if radius_cells <= 0:\n        return field.copy()\n    k = 2 * radius_cells + 1\n    ones = np.ones(k, dtype=float)\n    out = np.apply_along_axis(lambda m: np.convolve(m, ones, mode=\"same\"), 0, field)\n    out = np.apply_along_axis(lambda m: np.convolve(m, ones, mode=\"same\"), 1, out)\n    return out\n\n\nFEATURE_NAMES: List[str] = [\n    \"tai_trong_ghi_nhan_lan_toa\",\n    \"tai_trong_lan_can_500m\",\n    \"mat_do_phi_vu_lan_toa\",\n    \"ky_vong_bom_khong_no\",\n    \"ho_bom_mat_do\",\n    \"ho_bom_lan_can_300m\",\n    \"ho_bom_duong_kinh_tb\",\n    \"ho_bom_hieu_chinh_tam_nhin\",\n    \"do_cao\",\n    \"do_doc\",\n    \"do_mem_nen_dat\",\n    \"khoang_cach_song\",\n    \"la_dat_canh_tac\",\n    \"la_rung\",\n    \"khoang_cach_lang\",\n    \"khoang_cach_duong\",\n    \"muc_do_phoi_nhiem\",\n]\n\n\n@dataclass\nclass FeatureTable:\n    \"\"\"Bảng đặc trưng phẳng, mỗi hàng là một ô lưới.\"\"\"\n\n    X: np.ndarray\n    names: List[str]\n    col: np.ndarray\n    row: np.ndarray\n\n    @property\n    def n_rows(self) -> int:\n        return int(self.X.shape[0])\n\n    def column(self, name: str) -> np.ndarray:\n        return self.X[:, self.names.index(name)]\n\n\ndef build_features(\n    scene: Scene,\n    crater_map: np.ndarray,\n    crater_diameter_map: np.ndarray,\n    cfg: RiskConfig,\n    crater_coverage: np.ndarray = None,\n) -> FeatureTable:\n    \"\"\"Dựng bảng đặc trưng từ cảnh mô phỏng và kết quả phát hiện hố bom.\n\n    Tham số ``crater_map`` là mật độ hố bom **do mô hình thị giác phát hiện**, chứ\n    không phải mật độ thật. Đây là điểm quan trọng: tầng ba tiêu thụ đầu ra của\n    tầng hai, kể cả sai sót của nó, đúng như khi vận hành thật.\n    \"\"\"\n    grid: Grid = scene.grid\n    terrain = scene.terrain\n\n    sigma_cells = cfg.record_kernel_radius_m / grid.cell\n\n    tonnage_spread = gaussian_spread(scene.recorded_tonnage, sigma_cells)\n    mission_density = gaussian_spread(\n        grid.accumulate(scene.record_x, scene.record_y), sigma_cells\n    )\n    tonnage_neigh = neighbourhood_sum(tonnage_spread, radius_cells=5)\n\n    crater_spread = gaussian_spread(crater_map, sigma_cells=1.5)\n    crater_neigh = neighbourhood_sum(crater_map, radius_cells=3)\n\n    # Hai đặc trưng dưới đây là nơi việc hợp nhất ba nguồn tạo ra giá trị thật, chứ\n    # không phải chỉ đặt chúng cạnh nhau. Cả hai đều xuất phát từ cơ chế vật lý đã\n    # nêu trong tài liệu, không phải từ việc thử nghiệm mò.\n    softness = terrain.soil_softness\n\n    # Thứ nhất — kỳ vọng số bom không nổ. Cái ta muốn biết không phải là bao nhiêu\n    # bom đã rơi, mà bao nhiêu quả trong số đó đã không nổ. Tỉ lệ không nổ tăng\n    # theo độ mềm của nền đất, vì đầu nổ chạm nổ cần lực cản đủ lớn mới kích hoạt.\n    # Nhân hai đại lượng đó với nhau cho ra ngay đại lượng cần ước lượng.\n    dud_multiplier = 1.0 + (cfg.dud_softness_gain - 1.0) * softness\n    expected_duds = tonnage_spread * dud_multiplier\n\n    # Thứ hai — mật độ hố bom đã hiệu chỉnh thiên lệch quan sát. Hố bom trên nền\n    # mềm bị bồi lấp nhanh nên ít quan sát được hơn, đúng ở nơi nhiều bom không nổ\n    # nhất. Nếu dùng thẳng số hố bom đếm được thì mô hình sẽ đánh giá thấp có hệ\n    # thống chính những vùng nguy hiểm nhất. Chia cho xác suất còn nhìn thấy được\n    # sẽ khử đúng thiên lệch đó.\n    visibility = np.clip(\n        cfg.crater_visibility_hard\n        + (cfg.crater_visibility_soft - cfg.crater_visibility_hard) * softness,\n        0.15,\n        1.0,\n    )\n    crater_corrected = crater_spread / visibility\n\n    layers = [\n        tonnage_spread,\n        tonnage_neigh,\n        mission_density,\n        expected_duds,\n        crater_spread,\n        crater_neigh,\n        crater_diameter_map,\n        crater_corrected,\n        terrain.elevation_m,\n        terrain.slope_deg,\n        terrain.soil_softness,\n        terrain.dist_river_m,\n        terrain.is_farmland.astype(float),\n        terrain.is_forest.astype(float),\n        terrain.dist_village_m,\n        terrain.dist_road_m,\n        terrain.exposure,\n    ]\n    assert len(layers) == len(FEATURE_NAMES)\n\n    X = np.stack([lay.ravel() for lay in layers], axis=1).astype(np.float32)\n\n    # Ở những ô không có ảnh vệ tinh phủ tới, nhóm đặc trưng quan sát được đánh dấu\n    # là **khuyết**, chứ không phải bằng không. Cây quyết định tăng cường gradient\n    # xử lý được giá trị khuyết một cách tự nhiên: nó học riêng một nhánh cho trường\n    # hợp thiếu dữ liệu, thay vì hiểu nhầm rằng nơi đó đã được nhìn và không thấy gì.\n    if crater_coverage is not None:\n        missing = ~np.asarray(crater_coverage, dtype=bool).ravel()\n        crater_cols = [\n            FEATURE_NAMES.index(name)\n            for name in (\n                \"ho_bom_mat_do\",\n                \"ho_bom_lan_can_300m\",\n                \"ho_bom_duong_kinh_tb\",\n                \"ho_bom_hieu_chinh_tam_nhin\",\n            )\n        ]\n        X[np.ix_(missing, crater_cols)] = np.nan\n\n    cols, rows = np.meshgrid(\n        np.arange(grid.cfg.n_cells_x), np.arange(grid.cfg.n_cells_y)\n    )\n    return FeatureTable(X=X, names=list(FEATURE_NAMES), col=cols.ravel(), row=rows.ravel())\n", "src/demine/risk/model.py": "\"\"\"Tầng ba — mô hình nguy cơ hợp nhất.\n\nMô hình dự báo chính là cây quyết định tăng cường gradient. Lựa chọn này có ba lý\ndo, và cả ba đều phục vụ yêu cầu của bài toán chứ không phải sở thích kỹ thuật.\n\nThứ nhất, dữ liệu ở đây là bảng số không đồng nhất — khoảng cách tính bằng mét,\nđộ dốc tính bằng độ, biến nhị phân về lớp phủ — và cây quyết định xử lý loại dữ\nliệu này tốt hơn mạng nơ-ron mà không cần chuẩn hoá cầu kỳ.\n\nThứ hai, kết quả được dùng để phân bổ nguồn lực công, nên phải giải thích được\ntừng ô lưới vì sao được xếp hạng cao. Cây quyết định cho phép đo mức đóng góp của\ntừng đặc trưng một cách trực tiếp.\n\nThứ ba, mô hình chạy trong vài giây trên máy thường, nên toàn bộ quy trình kiểm\nchứng bốn tầng — vốn đòi hỏi huấn luyện lại nhiều lần — vẫn hoàn tất trong thời\ngian chấp nhận được.\n\nĐầu ra thô của mô hình đi qua hai bước hiệu chỉnh trước khi được công bố: hiệu\nchỉnh theo khung học từ dữ liệu chỉ có quan sát dương, rồi hiệu chỉnh xác suất\nbằng hồi quy đẳng hướng trên tập giữ lại riêng.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional\n\nimport numpy as np\n\nfrom ..config import RiskConfig\nfrom ..utils import get_logger\nfrom ..evaluation.calibration import IsotonicCalibrator, fit_isotonic\nfrom .pu_learning import (\n    apply_pu_correction,\n    estimate_label_frequency,\n    fit_propensity,\n    inverse_propensity_weights,\n)\n\nlogger = get_logger(__name__)\n\n\ndef _build_estimator(cfg: RiskConfig):\n    \"\"\"Khởi tạo bộ học, ưu tiên XGBoost và lùi về scikit-learn nếu không có.\"\"\"\n    try:\n        from xgboost import XGBClassifier\n\n        return (\n            \"xgboost\",\n            XGBClassifier(\n                n_estimators=cfg.max_iter,\n                learning_rate=cfg.learning_rate,\n                max_depth=6,\n                min_child_weight=5,\n                subsample=0.85,\n                colsample_bytree=0.85,\n                reg_lambda=cfg.l2_regularization,\n                objective=\"binary:logistic\",\n                eval_metric=\"logloss\",\n                tree_method=\"hist\",\n                random_state=cfg.seed,\n                n_jobs=4,\n            ),\n        )\n    except Exception:\n        from sklearn.ensemble import HistGradientBoostingClassifier\n\n        return (\n            \"sklearn\",\n            HistGradientBoostingClassifier(\n                max_iter=cfg.max_iter,\n                learning_rate=cfg.learning_rate,\n                max_leaf_nodes=cfg.max_leaf_nodes,\n                min_samples_leaf=cfg.min_samples_leaf,\n                l2_regularization=cfg.l2_regularization,\n                random_state=cfg.seed,\n            ),\n        )\n\n\n@dataclass\nclass RiskModel:\n    \"\"\"Mô hình nguy cơ đã huấn luyện, kèm toàn bộ thành phần hiệu chỉnh.\"\"\"\n\n    cfg: RiskConfig\n    estimator: object = None\n    backend: str = \"\"\n    propensity: object = None\n    label_frequency: float = 1.0\n    calibrator: Optional[IsotonicCalibrator] = None\n    feature_names: List[str] = field(default_factory=list)\n    _sample_X: Optional[np.ndarray] = None\n    _sample_y: Optional[np.ndarray] = None\n\n    def _permutation_importance_on_sample(self) -> np.ndarray:\n        \"\"\"Ước lượng mức đóng góp bằng phép hoán vị trên mẫu giữ lại từ lúc học.\"\"\"\n        rng = np.random.default_rng(self.cfg.seed)\n        X, y = self._sample_X, self._sample_y\n        base = self._raw_score(X)\n        base_loss = float(\n            np.mean((base - y) ** 2)\n        )\n        out = np.zeros(X.shape[1], dtype=float)\n        for j in range(X.shape[1]):\n            Xp = X.copy()\n            rng.shuffle(Xp[:, j])\n            loss = float(np.mean((self._raw_score(Xp) - y) ** 2))\n            out[j] = max(0.0, loss - base_loss)\n        return out\n\n    # -- Huấn luyện --------------------------------------------------------\n    def fit(\n        self,\n        X: np.ndarray,\n        y: np.ndarray,\n        was_selected: np.ndarray,\n        X_all: np.ndarray,\n        feature_names: Optional[List[str]] = None,\n        calibration_mask: Optional[np.ndarray] = None,\n    ) -> \"RiskModel\":\n        \"\"\"Huấn luyện trên phần đất đã rà phá.\n\n        ``X`` và ``y`` chỉ gồm các ô nằm trong khoảnh đã rà phá — đó là toàn bộ\n        phần có nhãn thật. ``X_all`` gồm mọi ô của vùng nghiên cứu và được dùng để\n        ước lượng cơ chế chọn mẫu.\n        \"\"\"\n        self.feature_names = list(feature_names or [])\n        self.backend, self.estimator = _build_estimator(self.cfg)\n\n        sample_weight = None\n        if self.cfg.use_propensity_weighting:\n            self.propensity = fit_propensity(X_all, was_selected, seed=self.cfg.seed)\n            p_train = self.propensity.predict(X)\n            sample_weight = inverse_propensity_weights(p_train)\n            logger.info(\n                \"Hiệu chỉnh thiên lệch chọn mẫu: trọng số trong dải [%.2f, %.2f]\",\n                float(sample_weight.min()),\n                float(sample_weight.max()),\n            )\n\n        try:\n            self.estimator.fit(X, y, sample_weight=sample_weight)\n        except TypeError:\n            self.estimator.fit(X, y)\n\n        raw = self._raw_score(X)\n\n        # Giữ lại một mẫu nhỏ của tập huấn luyện để ước lượng mức đóng góp đặc trưng\n        # khi bộ học không tự cung cấp đại lượng đó.\n        n_sample = min(2000, X.shape[0])\n        idx = np.random.default_rng(self.cfg.seed).choice(\n            X.shape[0], size=n_sample, replace=False\n        )\n        self._sample_X = np.asarray(X, dtype=float)[idx]\n        self._sample_y = np.asarray(y, dtype=float)[idx]\n\n        if self.cfg.use_pu_correction:\n            self.label_frequency = estimate_label_frequency(raw[y == 1], raw)\n            logger.info(\n                \"Hệ số tần suất nhãn của khung học chỉ có quan sát dương: c = %.3f\",\n                self.label_frequency,\n            )\n\n        if calibration_mask is not None and calibration_mask.any():\n            scores_cal = apply_pu_correction(\n                self._raw_score(X[calibration_mask]), self.label_frequency\n            )\n            self.calibrator = fit_isotonic(scores_cal, y[calibration_mask])\n\n        logger.info(\n            \"Đã huấn luyện mô hình nguy cơ bằng %s trên %d ô có nhãn (%.1f%% dương).\",\n            self.backend,\n            int(X.shape[0]),\n            100.0 * float(np.mean(y)),\n        )\n        return self\n\n    # -- Suy luận ----------------------------------------------------------\n    def _raw_score(self, X: np.ndarray) -> np.ndarray:\n        proba = self.estimator.predict_proba(X)\n        return np.asarray(proba[:, 1], dtype=float)\n\n    def predict_probability(self, X: np.ndarray) -> np.ndarray:\n        \"\"\"Xác suất còn tồn tại vật nổ, đã qua toàn bộ các bước hiệu chỉnh.\"\"\"\n        scores = self._raw_score(X)\n        if self.cfg.use_pu_correction:\n            scores = apply_pu_correction(scores, self.label_frequency)\n        if self.calibrator is not None:\n            scores = self.calibrator.predict(scores)\n        return np.clip(scores, 0.0, 1.0)\n\n    # -- Giải thích --------------------------------------------------------\n    def feature_importance(self) -> Dict[str, float]:\n        \"\"\"Mức đóng góp của từng đặc trưng, chuẩn hoá về tổng bằng một.\n\n        XGBoost cung cấp sẵn mức đóng góp nội tại. HistGradientBoosting của\n        scikit-learn thì không, nên trong trường hợp đó mức đóng góp được ước lượng\n        bằng phép hoán vị trên một mẫu nhỏ của chính tập huấn luyện.\n        \"\"\"\n        values = None\n        if hasattr(self.estimator, \"feature_importances_\"):\n            values = np.asarray(self.estimator.feature_importances_, dtype=float)\n\n        if (values is None or values.size != len(self.feature_names)) and (\n            self._sample_X is not None\n        ):\n            values = self._permutation_importance_on_sample()\n\n        if values is None or values.size != len(self.feature_names):\n            return {}\n        total = values.sum()\n        if total <= 0:\n            return {}\n        values = values / total\n        pairs = sorted(\n            zip(self.feature_names, values), key=lambda kv: -kv[1]\n        )\n        return {k: round(float(v), 4) for k, v in pairs}\n\n\ndef permutation_importance(\n    model: RiskModel,\n    X: np.ndarray,\n    items: np.ndarray,\n    n_repeats: int = 3,\n    seed: int = 0,\n) -> Dict[str, float]:\n    \"\"\"Mức đóng góp theo phép hoán vị, đo trên chính chỉ tiêu ưu tiên rà phá.\n\n    Mức đóng góp nội tại của cây quyết định đo bằng mức giảm tạp chất, vốn không\n    phải là thứ mà người lập kế hoạch quan tâm. Phép hoán vị đo trực tiếp trên chỉ\n    tiêu mà đề tài cam kết: xáo trộn một đặc trưng rồi xem hiệu quả xếp thứ tự ưu\n    tiên sụt bao nhiêu.\n    \"\"\"\n    from ..evaluation.prioritisation import clearance_efficiency_curve\n\n    rng = np.random.default_rng(seed)\n    base = clearance_efficiency_curve(\n        model.predict_probability(X), items\n    ).recovered_at_20pct\n\n    out: Dict[str, float] = {}\n    for j, name in enumerate(model.feature_names):\n        drops = []\n        for _ in range(n_repeats):\n            X_perm = X.copy()\n            rng.shuffle(X_perm[:, j])\n            score = clearance_efficiency_curve(\n                model.predict_probability(X_perm), items\n            ).recovered_at_20pct\n            drops.append(base - score)\n        out[name] = round(float(np.mean(drops)), 4)\n\n    return dict(sorted(out.items(), key=lambda kv: -kv[1]))\n", "src/demine/risk/pu_learning.py": "\"\"\"Học từ dữ liệu chỉ có quan sát dương, và hiệu chỉnh thiên lệch chọn mẫu.\n\nĐây là phần phương pháp luận cốt lõi của tầng ba, và là chỗ dễ làm sai nhất.\n\n**Vấn đề.** Nhãn thật chỉ tồn tại ở những khoảnh đất đã được rà phá. Nhưng các\nkhoảnh đó không được chọn ngẫu nhiên — cơ quan chuyên môn chọn chúng dựa trên hồ\nsơ không kích, mức độ gần khu dân cư và nhu cầu sử dụng đất. Nếu huấn luyện thẳng\ntrên tập này rồi đánh giá cũng trên tập này, mô hình sẽ học lại đúng phán đoán của\nnhững người đi trước, và mọi chỉ tiêu đều bị thổi phồng.\n\n**Hai biện pháp được áp dụng song song.**\n\n1. *Hiệu chỉnh trọng số theo nghịch đảo xác suất được chọn.* Ước lượng xác suất\n   một ô lưới được đưa vào rà phá, rồi gán cho mỗi mẫu trong tập huấn luyện một\n   trọng số bằng nghịch đảo xác suất đó. Ô nào ít có khả năng được chọn mà vẫn lọt\n   vào tập thì đại diện cho nhiều ô tương tự chưa ai tới, nên đáng được coi trọng\n   hơn. Trọng số được cắt ngọn để tránh một vài mẫu hiếm chi phối toàn bộ.\n\n2. *Hiệu chỉnh theo khung học từ dữ liệu chỉ có quan sát dương.* Ngay trong phần\n   đất đã rà phá, việc rà phá cũng không hoàn hảo: một tỉ lệ nhỏ vật nổ bị bỏ sót\n   nên ô đó bị gán nhãn âm trong khi thực tế là dương. Theo cách tiếp cận của\n   Elkan và Noto, nếu ước lượng được xác suất ``c`` mà một ô thực sự dương được\n   ghi nhận là dương, thì xác suất thật xấp xỉ bằng xác suất mô hình chia cho\n   ``c``. Hằng số ``c`` được ước lượng ngay từ dữ liệu, trên tập giữ lại.\n\nCả hai biện pháp đều làm cho ước lượng xác suất thận trọng hơn — tức là nghiêng\nvề phía cho rằng còn nhiều vật nổ hơn những gì đã quan sát được. Đó là chiều\nnghiêng đúng cho bài toán này, nơi bỏ sót là sai lầm phải tránh bằng mọi giá.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass\nclass PropensityModel:\n    \"\"\"Mô hình xác suất một ô lưới được đưa vào rà phá.\"\"\"\n\n    coefficients: np.ndarray\n    intercept: float\n    feature_mean: np.ndarray\n    feature_std: np.ndarray\n    fill_values: np.ndarray\n\n    def predict(self, X: np.ndarray) -> np.ndarray:\n        Xf = _impute(np.asarray(X, dtype=np.float64), self.fill_values)\n        Z = (Xf - self.feature_mean) / self.feature_std\n        logit = Z @ self.coefficients + self.intercept\n        return 1.0 / (1.0 + np.exp(-np.clip(logit, -30.0, 30.0)))\n\n\ndef _impute(X: np.ndarray, fill: np.ndarray) -> np.ndarray:\n    \"\"\"Điền giá trị khuyết bằng trung vị của cột.\n\n    Nhóm đặc trưng quan sát mang giá trị khuyết ở những ô không có ảnh vệ tinh phủ\n    tới. Mô hình dự báo chính là cây quyết định nên xử lý được giá trị khuyết một\n    cách tự nhiên, nhưng mô hình xác suất được chọn ở đây là hồi quy tuyến tính nên\n    không. Vì đây chỉ là mô hình phụ trợ dùng để ước lượng trọng số, việc điền bằng\n    trung vị là đủ và không ảnh hưởng tới kết luận.\n    \"\"\"\n    if not np.isnan(X).any():\n        return X\n    out = X.copy()\n    idx = np.where(np.isnan(out))\n    out[idx] = np.take(fill, idx[1])\n    return out\n\n\ndef fit_propensity(X: np.ndarray, was_selected: np.ndarray, seed: int = 0) -> PropensityModel:\n    \"\"\"Ước lượng xác suất được chọn đưa vào rà phá bằng hồi quy logistic.\n\n    Cài đặt bằng phương pháp giảm gradient có phạt chuẩn bậc hai, không phụ thuộc\n    thư viện ngoài, để phần phương pháp luận này hoàn toàn tự chứa và kiểm tra\n    được.\n    \"\"\"\n    X = np.asarray(X, dtype=np.float64)\n    y = np.asarray(was_selected, dtype=np.float64)\n\n    with np.errstate(invalid=\"ignore\"):\n        fill = np.nanmedian(X, axis=0)\n    fill = np.where(np.isfinite(fill), fill, 0.0)\n    X = _impute(X, fill)\n\n    mean = X.mean(axis=0)\n    std = X.std(axis=0)\n    std[std < 1e-9] = 1.0\n    Z = (X - mean) / std\n\n    rng = np.random.default_rng(seed)\n    w = rng.normal(0.0, 0.01, size=Z.shape[1])\n    b = 0.0\n\n    lr = 0.25\n    n = Z.shape[0]\n    for step in range(400):\n        logit = np.clip(Z @ w + b, -30.0, 30.0)\n        p = 1.0 / (1.0 + np.exp(-logit))\n        diff = p - y\n        grad_w = Z.T @ diff / n + 1e-3 * w\n        grad_b = float(diff.mean())\n        w -= lr * grad_w\n        b -= lr * grad_b\n        if step == 250:\n            lr *= 0.4\n\n    return PropensityModel(\n        coefficients=w,\n        intercept=b,\n        feature_mean=mean,\n        feature_std=std,\n        fill_values=fill,\n    )\n\n\ndef inverse_propensity_weights(\n    propensity: np.ndarray, clip_quantile: float = 0.98\n) -> np.ndarray:\n    \"\"\"Trọng số nghịch đảo xác suất được chọn, có cắt ngọn.\n\n    Cắt ngọn là bắt buộc: khi xác suất được chọn tiến gần không, nghịch đảo của nó\n    bùng nổ và một vài mẫu hiếm sẽ chi phối toàn bộ quá trình huấn luyện. Ngưỡng\n    cắt được lấy theo phân vị chứ không phải một hằng số cố định, để phương pháp\n    không phụ thuộc thang đo của bộ dữ liệu cụ thể.\n    \"\"\"\n    p = np.clip(np.asarray(propensity, dtype=float), 1e-4, 1.0)\n    w = 1.0 / p\n    cap = float(np.quantile(w, clip_quantile))\n    w = np.minimum(w, cap)\n    return w / w.mean()\n\n\ndef estimate_label_frequency(\n    scores_positive: np.ndarray, scores_all: np.ndarray\n) -> float:\n    \"\"\"Ước lượng hằng số ``c`` của khung Elkan–Noto.\n\n    ``c`` là xác suất một ô thực sự dương được ghi nhận là dương trong dữ liệu. Ước\n    lượng bằng điểm trung bình mà mô hình gán cho các mẫu đã được ghi nhận dương;\n    đây là ước lượng chuẩn và không thiên lệch khi giả thiết chọn mẫu ngẫu nhiên\n    trong nhóm dương được thoả mãn xấp xỉ.\n    \"\"\"\n    if scores_positive.size == 0:\n        return 1.0\n    c = float(np.mean(scores_positive))\n    # Chặn dưới để tránh chia cho số rất nhỏ làm xác suất vượt quá một cách vô lý.\n    return float(np.clip(c, 0.25, 1.0))\n\n\ndef apply_pu_correction(scores: np.ndarray, c: float) -> np.ndarray:\n    \"\"\"Quy đổi điểm của mô hình thành xác suất thật theo khung Elkan–Noto.\"\"\"\n    if c <= 0:\n        return scores\n    return np.clip(np.asarray(scores, dtype=float) / c, 0.0, 1.0)\n", "src/demine/evaluation/__init__.py": "\"\"\"Khung kiểm chứng bốn tầng.\"\"\"\n\nfrom .calibration import calibration_report, reliability_curve\nfrom .consistency import check_consistency\nfrom .detection_metrics import evaluate_detections\nfrom .prioritisation import (\n    accident_coverage,\n    clearance_efficiency_curve,\n    priority_index,\n)\nfrom .spatial_cv import block_kfold, make_spatial_blocks, spatial_holdout, transfer_split\n\n__all__ = [\n    \"calibration_report\",\n    \"reliability_curve\",\n    \"check_consistency\",\n    \"evaluate_detections\",\n    \"accident_coverage\",\n    \"clearance_efficiency_curve\",\n    \"priority_index\",\n    \"block_kfold\",\n    \"make_spatial_blocks\",\n    \"spatial_holdout\",\n    \"transfer_split\",\n]\n", "src/demine/evaluation/ablation.py": "\"\"\"Phân tích đóng góp thành phần.\n\nBốn cấu hình được so sánh, đúng như cam kết trong đề xuất. Mục đích không phải để\nchứng minh hệ thống đầy đủ là tốt nhất — điều đó gần như chắc chắn — mà để trả lời\ncâu hỏi mà hội đồng sẽ đặt ra: **từng nguồn dữ liệu đóng góp được bao nhiêu, và có\nnguồn nào thừa không**.\n\n  A — chỉ hồ sơ không kích\n  B — chỉ hố bom phát hiện trên ảnh vệ tinh\n  C — hợp nhất hai nguồn trên, không có đặc trưng địa hình và hiện trạng\n  D — hệ thống đầy đủ\n\nSo sánh A với B đặc biệt đáng chú ý. Hồ sơ không kích có sai số định vị lớn nhưng\nphủ khắp; hố bom thì chính xác về vị trí nhưng thiếu hụt có hệ thống đúng ở nơi nền\nđất mềm — tức là đúng nơi nhiều vật nổ còn sót nhất. Hai nguồn sai theo hai kiểu\nkhác nhau, nên hợp nhất lại có giá trị thật chứ không phải cộng thêm cho đẹp.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, List\n\nimport numpy as np\n\nfrom ..config import RiskConfig\nfrom ..risk.features import FEATURE_NAMES\nfrom ..risk.model import RiskModel\nfrom ..utils import get_logger\nfrom .prioritisation import accident_coverage, clearance_efficiency_curve\n\nlogger = get_logger(__name__)\n\nFEATURE_GROUPS: Dict[str, List[str]] = {\n    \"ho_so_khong_kich\": [\n        \"tai_trong_ghi_nhan_lan_toa\",\n        \"tai_trong_lan_can_500m\",\n        \"mat_do_phi_vu_lan_toa\",\n        \"ky_vong_bom_khong_no\",\n    ],\n    \"ho_bom_anh_ve_tinh\": [\n        \"ho_bom_mat_do\",\n        \"ho_bom_lan_can_300m\",\n        \"ho_bom_duong_kinh_tb\",\n        \"ho_bom_hieu_chinh_tam_nhin\",\n    ],\n    \"dia_hinh_hien_trang\": [\n        \"do_cao\",\n        \"do_doc\",\n        \"do_mem_nen_dat\",\n        \"khoang_cach_song\",\n        \"la_dat_canh_tac\",\n        \"la_rung\",\n        \"khoang_cach_lang\",\n        \"khoang_cach_duong\",\n        \"muc_do_phoi_nhiem\",\n    ],\n}\n\nCONFIGURATIONS: Dict[str, List[str]] = {\n    \"A — chỉ hồ sơ không kích\": [\"ho_so_khong_kich\"],\n    \"B — chỉ hố bom từ ảnh vệ tinh\": [\"ho_bom_anh_ve_tinh\"],\n    \"C — hợp nhất hai nguồn, không địa hình\": [\n        \"ho_so_khong_kich\",\n        \"ho_bom_anh_ve_tinh\",\n    ],\n    \"D — hệ thống đầy đủ\": [\n        \"ho_so_khong_kich\",\n        \"ho_bom_anh_ve_tinh\",\n        \"dia_hinh_hien_trang\",\n    ],\n}\n\n\n@dataclass\nclass AblationRow:\n    name: str\n    n_features: int\n    recovered_at_20pct: float\n    gain_over_uniform: float\n    accident_capture_at_20pct: float\n\n    def as_dict(self) -> Dict[str, object]:\n        return {\n            \"cau_hinh\": self.name,\n            \"so_dac_trung\": self.n_features,\n            \"thu_hoi_tai_20pct\": round(self.recovered_at_20pct, 4),\n            \"loi_the_so_voi_quet_deu\": round(self.gain_over_uniform, 4),\n            \"bao_phu_tai_nan_tai_20pct\": round(self.accident_capture_at_20pct, 4),\n        }\n\n\ndef _column_indices(groups: List[str]) -> List[int]:\n    wanted = []\n    for g in groups:\n        wanted.extend(FEATURE_GROUPS[g])\n    return [FEATURE_NAMES.index(name) for name in wanted]\n\n\ndef run_ablation(\n    X_train: np.ndarray,\n    y_train: np.ndarray,\n    was_selected_train: np.ndarray,\n    X_all: np.ndarray,\n    X_test: np.ndarray,\n    items_test: np.ndarray,\n    accident_index_test: np.ndarray,\n    cfg: RiskConfig,\n) -> List[AblationRow]:\n    \"\"\"Chạy cả bốn cấu hình trên cùng một phép chia tập.\"\"\"\n    rows: List[AblationRow] = []\n\n    for name, groups in CONFIGURATIONS.items():\n        cols = _column_indices(groups)\n        names = [FEATURE_NAMES[i] for i in cols]\n\n        model = RiskModel(cfg=cfg)\n        model.fit(\n            X_train[:, cols],\n            y_train,\n            was_selected_train,\n            X_all[:, cols],\n            feature_names=names,\n        )\n        prob = model.predict_probability(X_test[:, cols])\n\n        curve = clearance_efficiency_curve(prob, items_test)\n        coverage = accident_coverage(prob, accident_index_test, name)\n\n        rows.append(\n            AblationRow(\n                name=name,\n                n_features=len(cols),\n                recovered_at_20pct=curve.recovered_at_20pct,\n                gain_over_uniform=curve.gain_over_uniform,\n                accident_capture_at_20pct=coverage.capture_at_20pct,\n            )\n        )\n        logger.info(\n            \"  %-38s thu hồi tại 20%% diện tích: %.3f | bao phủ tai nạn: %.3f\",\n            name,\n            curve.recovered_at_20pct,\n            coverage.capture_at_20pct,\n        )\n\n    return rows\n", "src/demine/evaluation/calibration.py": "\"\"\"Hiệu chỉnh xác suất và các chỉ tiêu đo chất lượng hiệu chỉnh.\n\nĐầu ra của hệ thống được dùng để phân bổ nguồn lực công, nên xếp hạng đúng thôi\nchưa đủ. Nếu mô hình nói một ô có xác suất ba mươi phần trăm thì trong thực tế,\ntrong số các ô được nói như vậy, phải có khoảng ba mươi phần trăm thực sự chứa vật\nnổ. Một mô hình xếp hạng hoàn hảo nhưng ước lượng lệch mức độ vẫn dẫn tới chia\nngân sách sai giữa các địa bàn.\n\nBa đại lượng được báo cáo: biểu đồ tin cậy, sai số hiệu chỉnh kỳ vọng và điểm\nBrier. Phép hiệu chỉnh dùng hồi quy đẳng hướng, cài đặt bằng thuật toán gộp các\nvi phạm liền kề, không phụ thuộc thư viện ngoài.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Tuple\n\nimport numpy as np\n\n\n@dataclass\nclass IsotonicCalibrator:\n    \"\"\"Hiệu chỉnh xác suất bằng hồi quy đẳng hướng.\"\"\"\n\n    x_thresholds: np.ndarray\n    y_values: np.ndarray\n\n    def predict(self, scores: np.ndarray) -> np.ndarray:\n        s = np.asarray(scores, dtype=float)\n        if self.x_thresholds.size == 0:\n            return np.clip(s, 0.0, 1.0)\n        return np.clip(\n            np.interp(s, self.x_thresholds, self.y_values), 0.0, 1.0\n        )\n\n\ndef fit_isotonic(scores: np.ndarray, labels: np.ndarray) -> IsotonicCalibrator:\n    \"\"\"Khớp hồi quy đẳng hướng bằng thuật toán gộp các vi phạm liền kề.\"\"\"\n    s = np.asarray(scores, dtype=float)\n    y = np.asarray(labels, dtype=float)\n    if s.size == 0:\n        return IsotonicCalibrator(np.array([]), np.array([]))\n\n    order = np.argsort(s, kind=\"mergesort\")\n    s_sorted = s[order]\n    y_sorted = y[order]\n\n    values = list(y_sorted)\n    weights = [1.0] * len(values)\n    positions = list(range(len(values)))\n\n    i = 0\n    while i < len(values) - 1:\n        if values[i] <= values[i + 1] + 1e-12:\n            i += 1\n            continue\n        total_w = weights[i] + weights[i + 1]\n        merged = (values[i] * weights[i] + values[i + 1] * weights[i + 1]) / total_w\n        values[i] = merged\n        weights[i] = total_w\n        del values[i + 1]\n        del weights[i + 1]\n        del positions[i + 1]\n        if i > 0:\n            i -= 1\n\n    x_out, y_out = [], []\n    idx = 0\n    for value, weight in zip(values, weights):\n        n = int(round(weight))\n        x_out.append(s_sorted[min(idx + n - 1, s_sorted.size - 1)])\n        y_out.append(value)\n        idx += n\n\n    return IsotonicCalibrator(\n        x_thresholds=np.asarray(x_out, dtype=float),\n        y_values=np.asarray(y_out, dtype=float),\n    )\n\n\ndef reliability_curve(\n    probability: np.ndarray, labels: np.ndarray, n_bins: int = 12\n) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:\n    \"\"\"Biểu đồ tin cậy: xác suất dự báo trung bình so với tần suất thực tế.\"\"\"\n    p = np.asarray(probability, dtype=float)\n    y = np.asarray(labels, dtype=float)\n\n    edges = np.linspace(0.0, 1.0, n_bins + 1)\n    edges[-1] += 1e-9\n    bin_idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)\n\n    mean_pred, mean_true, counts = [], [], []\n    for b in range(n_bins):\n        sel = bin_idx == b\n        counts.append(int(sel.sum()))\n        if sel.any():\n            mean_pred.append(float(p[sel].mean()))\n            mean_true.append(float(y[sel].mean()))\n        else:\n            mean_pred.append(np.nan)\n            mean_true.append(np.nan)\n\n    return (\n        np.asarray(mean_pred),\n        np.asarray(mean_true),\n        np.asarray(counts, dtype=float),\n    )\n\n\ndef expected_calibration_error(\n    probability: np.ndarray, labels: np.ndarray, n_bins: int = 12\n) -> float:\n    \"\"\"Sai số hiệu chỉnh kỳ vọng, trung bình có trọng số theo số mẫu mỗi khoảng.\"\"\"\n    pred, true, counts = reliability_curve(probability, labels, n_bins)\n    valid = counts > 0\n    if not valid.any():\n        return 0.0\n    w = counts[valid] / counts[valid].sum()\n    return float(np.sum(w * np.abs(pred[valid] - true[valid])))\n\n\ndef brier_score(probability: np.ndarray, labels: np.ndarray) -> float:\n    p = np.asarray(probability, dtype=float)\n    y = np.asarray(labels, dtype=float)\n    if p.size == 0:\n        return 0.0\n    return float(np.mean((p - y) ** 2))\n\n\ndef calibration_report(\n    probability: np.ndarray, labels: np.ndarray, n_bins: int = 12\n) -> Dict[str, float]:\n    return {\n        \"sai_so_hieu_chinh_ky_vong\": round(\n            expected_calibration_error(probability, labels, n_bins), 4\n        ),\n        \"diem_brier\": round(brier_score(probability, labels), 4),\n        \"ty_le_duong_thuc_te\": round(float(np.mean(labels)), 4),\n        \"xac_suat_du_bao_trung_binh\": round(float(np.mean(probability)), 4),\n    }\n", "src/demine/evaluation/consistency.py": "\"\"\"Kiểm tra nhất quán giữa hai nguồn dữ liệu độc lập.\n\nHồ sơ không kích và ảnh vệ tinh lịch sử không liên quan gì đến nhau về xuất xứ:\nmột bên là sổ sách tác chiến của phi đội, một bên là dấu vết vật lý còn lại trên\nmặt đất và được ghi lại bởi một hệ thống hoàn toàn khác. Nếu hai nguồn này khớp\nnhau về mặt không gian, độ tin cậy của cả hai cùng được củng cố mà không cần viện\nđến bất kỳ nhãn đối chứng nào.\n\nPhép kiểm tra thứ hai ở bậc độ lớn: từ tổng lượng bom đạn theo hồ sơ và tỉ lệ bom\nkhông nổ đã biết trong tài liệu kỹ thuật, ước lượng khối lượng vật nổ còn sót rồi\nso với con số mà mô hình đưa ra. Không đòi hỏi trùng khít, chỉ đòi hỏi cùng bậc.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict\n\nimport numpy as np\n\n\ndef spearman_correlation(a: np.ndarray, b: np.ndarray) -> float:\n    \"\"\"Hệ số tương quan hạng Spearman, cài đặt trực tiếp.\n\n    Dùng tương quan hạng chứ không phải tương quan tuyến tính vì quan hệ giữa tải\n    trọng bom và số hố bom quan sát được là quan hệ đồng biến nhưng không tuyến\n    tính, và cả hai phân bố đều lệch mạnh về phía giá trị nhỏ.\n    \"\"\"\n    a = np.asarray(a, dtype=float).ravel()\n    b = np.asarray(b, dtype=float).ravel()\n    if a.size < 3:\n        return 0.0\n\n    ra = _rank_average(a)\n    rb = _rank_average(b)\n    ra = ra - ra.mean()\n    rb = rb - rb.mean()\n    denom = np.sqrt((ra ** 2).sum() * (rb ** 2).sum())\n    if denom < 1e-12:\n        return 0.0\n    return float((ra * rb).sum() / denom)\n\n\ndef _rank_average(x: np.ndarray) -> np.ndarray:\n    \"\"\"Hạng của từng phần tử, các giá trị bằng nhau nhận hạng trung bình.\"\"\"\n    order = np.argsort(x, kind=\"mergesort\")\n    ranks = np.empty(x.size, dtype=float)\n    ranks[order] = np.arange(1, x.size + 1, dtype=float)\n\n    x_sorted = x[order]\n    i = 0\n    while i < x_sorted.size:\n        j = i\n        while j + 1 < x_sorted.size and x_sorted[j + 1] == x_sorted[i]:\n            j += 1\n        if j > i:\n            mean_rank = (i + j + 2) / 2.0\n            ranks[order[i : j + 1]] = mean_rank\n        i = j + 1\n    return ranks\n\n\n@dataclass\nclass ConsistencyReport:\n    spearman_crater_vs_tonnage: float\n    n_cells_compared: int\n    estimated_uxo_from_records: float\n    model_expected_uxo: float\n    order_of_magnitude_ratio: float\n\n    def as_dict(self) -> Dict[str, float]:\n        return {\n            \"spearman_ho_bom_vs_tai_trong\": round(self.spearman_crater_vs_tonnage, 4),\n            \"so_o_duoc_so_sanh\": self.n_cells_compared,\n            \"uoc_luong_vat_no_tu_ho_so\": round(self.estimated_uxo_from_records, 1),\n            \"ky_vong_vat_no_tu_mo_hinh\": round(self.model_expected_uxo, 1),\n            \"ty_so_bac_do_lon\": round(self.order_of_magnitude_ratio, 3),\n        }\n\n\ndef check_consistency(\n    crater_density: np.ndarray,\n    tonnage_spread: np.ndarray,\n    n_recorded_bombs: int,\n    base_dud_rate: float,\n    model_probability: np.ndarray,\n) -> ConsistencyReport:\n    \"\"\"Chạy cả hai phép kiểm tra nhất quán và gộp kết quả.\"\"\"\n    a = np.asarray(crater_density, dtype=float).ravel()\n    b = np.asarray(tonnage_spread, dtype=float).ravel()\n\n    # Chỉ so sánh trên các ô mà ít nhất một trong hai nguồn có tín hiệu; các ô\n    # trống ở cả hai nguồn không mang thông tin và chỉ làm loãng hệ số.\n    mask = (a > 1e-9) | (b > 1e-9)\n    rho = spearman_correlation(a[mask], b[mask])\n\n    expected_from_records = float(n_recorded_bombs) * base_dud_rate\n    expected_from_model = float(np.sum(model_probability))\n    ratio = expected_from_model / max(expected_from_records, 1e-9)\n\n    return ConsistencyReport(\n        spearman_crater_vs_tonnage=rho,\n        n_cells_compared=int(mask.sum()),\n        estimated_uxo_from_records=expected_from_records,\n        model_expected_uxo=expected_from_model,\n        order_of_magnitude_ratio=ratio,\n    )\n", "src/demine/evaluation/detection_metrics.py": "\"\"\"Chỉ tiêu đánh giá tầng phát hiện hố bom.\n\nCài đặt mAP theo đúng quy ước PASCAL VOC từ năm 2010 trở đi: đường cong độ chính\nxác theo độ bao phủ được nội suy đơn điệu trước khi lấy tích phân. Việc tự cài đặt\nthay vì gọi thư viện là có chủ ý — đội thi phải hiểu và giải thích được từng bước\ncủa chỉ tiêu mà mình công bố, và cách này cũng loại bỏ một phụ thuộc có thể không\ncài được khi notebook chạy ngắt mạng.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, List, Sequence\n\nimport numpy as np\n\n\ndef iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:\n    \"\"\"Ma trận tỉ số giao trên hợp giữa hai tập khung bao.\"\"\"\n    if boxes_a.size == 0 or boxes_b.size == 0:\n        return np.zeros((boxes_a.shape[0], boxes_b.shape[0]), dtype=float)\n\n    ax1, ay1, ax2, ay2 = [boxes_a[:, i][:, None] for i in range(4)]\n    bx1, by1, bx2, by2 = [boxes_b[:, i][None, :] for i in range(4)]\n\n    inter_w = np.clip(np.minimum(ax2, bx2) - np.maximum(ax1, bx1), 0.0, None)\n    inter_h = np.clip(np.minimum(ay2, by2) - np.maximum(ay1, by1), 0.0, None)\n    inter = inter_w * inter_h\n\n    area_a = np.clip(ax2 - ax1, 0.0, None) * np.clip(ay2 - ay1, 0.0, None)\n    area_b = np.clip(bx2 - bx1, 0.0, None) * np.clip(by2 - by1, 0.0, None)\n\n    union = area_a + area_b - inter\n    return np.where(union > 0, inter / union, 0.0)\n\n\n@dataclass\nclass DetectionMetrics:\n    precision: float\n    recall: float\n    f1: float\n    ap50: float\n    ap50_95: float\n    n_true: int\n    n_pred: int\n\n    def as_dict(self) -> Dict[str, float]:\n        return {\n            \"precision\": round(self.precision, 4),\n            \"recall\": round(self.recall, 4),\n            \"f1\": round(self.f1, 4),\n            \"mAP@0.5\": round(self.ap50, 4),\n            \"mAP@0.5:0.95\": round(self.ap50_95, 4),\n            \"so_ho_bom_that\": self.n_true,\n            \"so_ho_bom_du_bao\": self.n_pred,\n        }\n\n\ndef _average_precision(\n    all_scores: np.ndarray, all_tp: np.ndarray, n_true: int\n) -> float:\n    \"\"\"Độ chính xác trung bình theo quy ước nội suy đơn điệu.\"\"\"\n    if n_true == 0:\n        return 0.0\n    if all_scores.size == 0:\n        return 0.0\n\n    order = np.argsort(-all_scores, kind=\"mergesort\")\n    tp = all_tp[order].astype(float)\n    fp = 1.0 - tp\n\n    cum_tp = np.cumsum(tp)\n    cum_fp = np.cumsum(fp)\n\n    recall = cum_tp / n_true\n    precision = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)\n\n    # Nội suy đơn điệu: độ chính xác tại mỗi mức bao phủ lấy bằng giá trị lớn nhất\n    # ở các mức bao phủ từ đó trở đi.\n    precision = np.maximum.accumulate(precision[::-1])[::-1]\n\n    recall = np.concatenate([[0.0], recall])\n    precision = np.concatenate([[precision[0] if precision.size else 0.0], precision])\n    return float(np.trapezoid(precision, recall))\n\n\ndef _match_at_threshold(\n    predictions: Sequence, ground_truth: Dict[str, np.ndarray], iou_threshold: float\n):\n    scores_all: List[float] = []\n    tp_all: List[float] = []\n    n_true = 0\n\n    for det in predictions:\n        gt = ground_truth.get(det.image_id, np.zeros((0, 4), dtype=float))\n        n_true += int(gt.shape[0])\n\n        if det.boxes.shape[0] == 0:\n            continue\n\n        order = np.argsort(-det.scores, kind=\"mergesort\")\n        boxes = det.boxes[order]\n        scores = det.scores[order]\n\n        matched = np.zeros(gt.shape[0], dtype=bool)\n        ious = iou_matrix(boxes, gt)\n\n        for i in range(boxes.shape[0]):\n            scores_all.append(float(scores[i]))\n            if gt.shape[0] == 0:\n                tp_all.append(0.0)\n                continue\n            candidates = ious[i].copy()\n            candidates[matched] = -1.0\n            best = int(np.argmax(candidates))\n            if candidates[best] >= iou_threshold:\n                matched[best] = True\n                tp_all.append(1.0)\n            else:\n                tp_all.append(0.0)\n\n    return np.asarray(scores_all), np.asarray(tp_all), n_true\n\n\ndef evaluate_detections(\n    predictions: Sequence,\n    ground_truth: Dict[str, np.ndarray],\n    score_threshold: float = 0.25,\n) -> DetectionMetrics:\n    \"\"\"Tính toàn bộ chỉ tiêu của tầng phát hiện.\"\"\"\n    scores, tp, n_true = _match_at_threshold(predictions, ground_truth, 0.50)\n    ap50 = _average_precision(scores, tp, n_true)\n\n    aps = []\n    for thr in np.arange(0.50, 0.951, 0.05):\n        s, t, n = _match_at_threshold(predictions, ground_truth, float(thr))\n        aps.append(_average_precision(s, t, n))\n    ap50_95 = float(np.mean(aps)) if aps else 0.0\n\n    keep = scores >= score_threshold\n    n_pred = int(keep.sum())\n    tp_count = float(tp[keep].sum()) if n_pred else 0.0\n    precision = tp_count / n_pred if n_pred else 0.0\n    recall = tp_count / n_true if n_true else 0.0\n    f1 = (\n        2.0 * precision * recall / (precision + recall)\n        if (precision + recall) > 0\n        else 0.0\n    )\n\n    return DetectionMetrics(\n        precision=precision,\n        recall=recall,\n        f1=f1,\n        ap50=ap50,\n        ap50_95=ap50_95,\n        n_true=n_true,\n        n_pred=n_pred,\n    )\n", "src/demine/evaluation/prioritisation.py": "\"\"\"Chỉ tiêu đánh giá hiệu quả xếp thứ tự ưu tiên rà phá.\n\nĐây là nhóm chỉ tiêu quan trọng nhất của đề tài, vì nó là đại lượng duy nhất\nchuyển thẳng được thành ý nghĩa thực tiễn. Độ chính xác phân loại không nói lên\nđiều gì cho người lập kế hoạch rà phá; câu hỏi của họ là: nếu đi theo thứ tự này\nthì sau khi làm xong một phần diện tích, đã xử lý được bao nhiêu phần nguy cơ.\n\nHai chỉ tiêu được cài đặt:\n\n*Đường cong hiệu quả rà phá.* Sắp các khoảnh đất theo thứ tự mô hình đề xuất, rồi\nvẽ tỉ lệ vật nổ đã thu hồi theo tỉ lệ diện tích đã rà. Đường chéo là phương án\nquét trải đều. Diện tích nằm giữa đường cong và đường chéo là phần giá trị mà mô\nhình tạo ra.\n\n*Tỉ lệ bao phủ tai nạn.* Tỉ lệ các vụ tai nạn đã thực sự xảy ra rơi vào phần diện\ntích được mô hình xếp nguy cơ cao nhất. Vì vị trí tai nạn không do ai chọn, đây là\nphép kiểm chứng độc lập với mọi phán đoán chuyên môn đã có trước.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List\n\nimport numpy as np\n\n\n@dataclass\nclass ClearanceCurve:\n    \"\"\"Đường cong hiệu quả rà phá.\"\"\"\n\n    area_fraction: np.ndarray\n    recovered_fraction: np.ndarray\n    recovered_at_20pct: float\n    recovered_at_10pct: float\n    recovered_at_50pct: float\n    gain_over_uniform: float\n\n    def as_dict(self) -> Dict[str, float]:\n        return {\n            \"thu_hoi_tai_10pct_dien_tich\": round(self.recovered_at_10pct, 4),\n            \"thu_hoi_tai_20pct_dien_tich\": round(self.recovered_at_20pct, 4),\n            \"thu_hoi_tai_50pct_dien_tich\": round(self.recovered_at_50pct, 4),\n            \"loi_the_so_voi_quet_deu\": round(self.gain_over_uniform, 4),\n        }\n\n\ndef clearance_efficiency_curve(\n    scores: np.ndarray, items: np.ndarray, cell_weight: np.ndarray = None\n) -> ClearanceCurve:\n    \"\"\"Dựng đường cong hiệu quả rà phá.\n\n    Tham số ``items`` là số vật nổ thực sự có trên mỗi ô, ``scores`` là điểm nguy\n    cơ do mô hình gán. Nếu các ô có diện tích khác nhau thì truyền ``cell_weight``;\n    ở đây mọi ô đều bằng nhau nên tham số này thường bỏ trống.\n    \"\"\"\n    scores = np.asarray(scores, dtype=float)\n    items = np.asarray(items, dtype=float)\n    weight = (\n        np.ones_like(items) if cell_weight is None else np.asarray(cell_weight, float)\n    )\n\n    order = np.argsort(-scores, kind=\"mergesort\")\n    items_sorted = items[order]\n    weight_sorted = weight[order]\n\n    cum_area = np.cumsum(weight_sorted) / max(weight_sorted.sum(), 1e-12)\n    total_items = items_sorted.sum()\n    cum_items = np.cumsum(items_sorted) / (total_items if total_items > 0 else 1.0)\n\n    cum_area = np.concatenate([[0.0], cum_area])\n    cum_items = np.concatenate([[0.0], cum_items])\n\n    def at(fraction: float) -> float:\n        return float(np.interp(fraction, cum_area, cum_items))\n\n    # Diện tích dưới đường cong, trừ đi 0,5 của đường chéo, rồi chuẩn hoá về [0, 1].\n    auc = float(np.trapezoid(cum_items, cum_area))\n    gain = (auc - 0.5) / 0.5\n\n    return ClearanceCurve(\n        area_fraction=cum_area,\n        recovered_fraction=cum_items,\n        recovered_at_10pct=at(0.10),\n        recovered_at_20pct=at(0.20),\n        recovered_at_50pct=at(0.50),\n        gain_over_uniform=gain,\n    )\n\n\n@dataclass\nclass AccidentCoverage:\n    \"\"\"Kết quả kiểm chứng bằng hồ sơ tai nạn.\"\"\"\n\n    n_accidents: int\n    capture_at_10pct: float\n    capture_at_20pct: float\n    capture_at_30pct: float\n    lift_at_20pct: float\n    split_label: str = \"toàn bộ\"\n    detail: Dict[str, float] = field(default_factory=dict)\n\n    def as_dict(self) -> Dict[str, float]:\n        return {\n            \"tap_kiem_chung\": self.split_label,\n            \"so_vu_tai_nan\": self.n_accidents,\n            \"bao_phu_tai_10pct\": round(self.capture_at_10pct, 4),\n            \"bao_phu_tai_20pct\": round(self.capture_at_20pct, 4),\n            \"bao_phu_tai_30pct\": round(self.capture_at_30pct, 4),\n            \"he_so_vuot_ngau_nhien_tai_20pct\": round(self.lift_at_20pct, 3),\n        }\n\n\ndef accident_coverage(\n    scores: np.ndarray,\n    accident_cell_index: np.ndarray,\n    split_label: str = \"toàn bộ\",\n) -> AccidentCoverage:\n    \"\"\"Đo tỉ lệ vụ tai nạn rơi vào phần diện tích nguy cơ cao nhất.\n\n    ``accident_cell_index`` là chỉ số phẳng của ô lưới nơi xảy ra từng vụ tai nạn,\n    theo đúng thứ tự hàng của ``scores``.\n    \"\"\"\n    scores = np.asarray(scores, dtype=float)\n    idx = np.asarray(accident_cell_index, dtype=int)\n    n = int(idx.size)\n\n    if n == 0:\n        return AccidentCoverage(0, 0.0, 0.0, 0.0, 0.0, split_label)\n\n    order = np.argsort(-scores, kind=\"mergesort\")\n    rank = np.empty_like(order)\n    rank[order] = np.arange(order.size)\n    accident_rank = rank[idx] / float(order.size)\n\n    def capture(fraction: float) -> float:\n        return float(np.mean(accident_rank <= fraction))\n\n    c20 = capture(0.20)\n    return AccidentCoverage(\n        n_accidents=n,\n        capture_at_10pct=capture(0.10),\n        capture_at_20pct=c20,\n        capture_at_30pct=capture(0.30),\n        lift_at_20pct=c20 / 0.20,\n        split_label=split_label,\n    )\n\n\ndef priority_index(\n    probability: np.ndarray, exposure: np.ndarray, exposure_weight: float = 0.45\n) -> np.ndarray:\n    \"\"\"Chỉ số ưu tiên tổng hợp giữa xác suất và mức độ phơi nhiễm của cộng đồng.\n\n    Xác suất còn vật nổ không phải là tất cả. Một ô nằm giữa rừng, xác suất cao,\n    nhưng nhiều năm không ai đặt chân tới, thì nguy cơ gây thương vong thấp hơn một\n    ô xác suất trung bình nằm ngay cạnh trường học. Chỉ số này kết hợp hai vế đó\n    để phục vụ việc phân bổ nguồn lực, trong khi vẫn giữ nguyên xác suất gốc để báo\n    cáo riêng.\n    \"\"\"\n    p = np.asarray(probability, dtype=float)\n    e = np.asarray(exposure, dtype=float)\n    e = (e - e.min()) / (np.ptp(e) + 1e-12)\n    return (1.0 - exposure_weight) * p + exposure_weight * p * e\n", "src/demine/evaluation/spatial_cv.py": "\"\"\"Chia tập theo khối không gian.\n\nĐây là chi tiết phương pháp quan trọng nhất của toàn bộ phần đánh giá, và cũng là\nchi tiết dễ bị bỏ qua nhất.\n\nPhân bố vật nổ còn sót có tương quan không gian rất mạnh: hai ô lưới cạnh nhau gần\nnhư luôn cùng có hoặc cùng không có vật nổ, vì chúng nằm trong cùng một loạt bom.\nNếu chia tập huấn luyện và tập kiểm tra một cách ngẫu nhiên theo từng ô, thì mỗi ô\ntrong tập kiểm tra sẽ có hàng xóm nằm trong tập huấn luyện. Mô hình chỉ cần nội\nsuy từ hàng xóm là đã đạt chỉ tiêu rất cao, mà không hề học được quy luật nào.\nKết quả thu được sẽ đẹp và vô nghĩa.\n\nCách làm đúng là chia theo khối không gian liền mạch, đủ lớn để ranh giới giữa các\nkhối vượt quá tầm tương quan. Toàn bộ phần đánh giá của đề tài dùng cách chia này.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Iterator, List, Tuple\n\nimport numpy as np\n\n\ndef make_spatial_blocks(\n    col: np.ndarray, row: np.ndarray, n_blocks: int, grid_shape: Tuple[int, int]\n) -> np.ndarray:\n    \"\"\"Gán mỗi ô lưới vào một khối không gian hình chữ nhật.\n\n    Số khối theo mỗi chiều được chọn sao cho khối gần vuông nhất có thể, để không\n    có chiều nào bị chia quá mảnh làm mất tác dụng của việc chia khối.\n    \"\"\"\n    ny, nx = grid_shape\n    side = max(1, int(round(np.sqrt(n_blocks))))\n    n_bx = side\n    n_by = max(1, int(np.ceil(n_blocks / side)))\n\n    bx = np.minimum((np.asarray(col) * n_bx) // nx, n_bx - 1)\n    by = np.minimum((np.asarray(row) * n_by) // ny, n_by - 1)\n    return (by * n_bx + bx).astype(int)\n\n\ndef block_kfold(\n    blocks: np.ndarray, n_folds: int, seed: int = 0\n) -> Iterator[Tuple[np.ndarray, np.ndarray]]:\n    \"\"\"Sinh các lần chia huấn luyện và kiểm tra theo nhóm khối.\n\n    Mỗi lần chia giữ lại toàn bộ một nhóm khối làm tập kiểm tra. Không có khối nào\n    bị chia đôi giữa hai tập, nên không có rò rỉ thông tin qua ranh giới.\n    \"\"\"\n    unique = np.unique(blocks)\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(unique)\n    folds: List[np.ndarray] = np.array_split(order, max(1, n_folds))\n\n    for held in folds:\n        if held.size == 0:\n            continue\n        test_mask = np.isin(blocks, held)\n        yield ~test_mask, test_mask\n\n\ndef spatial_holdout(\n    blocks: np.ndarray, test_fraction: float = 0.3, seed: int = 0\n) -> Tuple[np.ndarray, np.ndarray]:\n    \"\"\"Một lần chia duy nhất theo khối, dùng cho các bước cần tập giữ lại cố định.\"\"\"\n    unique = np.unique(blocks)\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(unique)\n    n_test = max(1, int(round(order.size * test_fraction)))\n    held = order[:n_test]\n    test_mask = np.isin(blocks, held)\n    return ~test_mask, test_mask\n\n\ndef transfer_split(col: np.ndarray, grid_shape: Tuple[int, int]) -> Tuple[np.ndarray, np.ndarray]:\n    \"\"\"Chia vùng nghiên cứu thành hai nửa đông và tây.\n\n    Dùng cho phép thử chuyển vùng: huấn luyện trên một nửa, áp dụng cho nửa còn\n    lại như thể đó là một địa bàn hoàn toàn mới chưa từng khảo sát.\n    \"\"\"\n    _, nx = grid_shape\n    west = np.asarray(col) < nx // 2\n    return west, ~west\n", "src/demine/viz/__init__.py": "\"\"\"Tầng bốn — kết xuất hình minh hoạ và bản đồ.\"\"\"\n\nfrom .figures import make_all_figures\nfrom .maps import make_priority_map\n\n__all__ = [\"make_all_figures\", \"make_priority_map\"]\n", "src/demine/viz/figures.py": "\"\"\"Hình minh hoạ phục vụ báo cáo và trình diễn trước hội đồng.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import List\n\nimport numpy as np\n\nfrom ..utils import get_logger\n\nlogger = get_logger(__name__)\n\n# Bảng màu pastel thống nhất với bản đề xuất.\nINK = \"#1C3557\"\nBLUE = \"#4C7FB0\"\nMINT = \"#5FA383\"\nBLUSH = \"#C9705C\"\nCREAM = \"#DFC58A\"\n\n\ndef _setup():\n    import matplotlib\n\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    plt.rcParams.update(\n        {\n            \"figure.dpi\": 130,\n            \"savefig.dpi\": 130,\n            \"font.size\": 9.5,\n            \"axes.edgecolor\": \"#9DBCDA\",\n            \"axes.labelcolor\": INK,\n            \"text.color\": INK,\n            \"xtick.color\": INK,\n            \"ytick.color\": INK,\n            \"axes.grid\": True,\n            \"grid.color\": \"#E4ECF4\",\n            \"grid.linewidth\": 0.8,\n            \"figure.facecolor\": \"white\",\n        }\n    )\n    return plt\n\n\ndef fig_clearance_curve(pipeline, path: Path) -> None:\n    \"\"\"Đường cong hiệu quả rà phá — hình quan trọng nhất của đề tài.\"\"\"\n    plt = _setup()\n    curve = pipeline.curve\n\n    fig, ax = plt.subplots(figsize=(6.2, 4.4))\n    ax.plot(\n        curve.area_fraction * 100,\n        curve.recovered_fraction * 100,\n        color=INK,\n        linewidth=2.2,\n        label=\"Rà phá theo thứ tự mô hình đề xuất\",\n    )\n    ax.plot([0, 100], [0, 100], color=BLUSH, linestyle=\"--\", linewidth=1.5,\n            label=\"Quét trải đều\")\n\n    ax.axvline(20, color=CREAM, linewidth=1.2, linestyle=\":\")\n    y20 = curve.recovered_at_20pct * 100\n    ax.plot([20], [y20], marker=\"o\", color=MINT, markersize=7, zorder=5)\n    ax.annotate(\n        f\"tại 20% diện tích\\nthu hồi {y20:.0f}% vật nổ\",\n        xy=(20, y20),\n        xytext=(28, max(12.0, y20 - 26)),\n        color=INK,\n        arrowprops=dict(arrowstyle=\"->\", color=MINT, linewidth=1.2),\n    )\n\n    ax.set_xlabel(\"Tỉ lệ diện tích đã rà phá (%)\")\n    ax.set_ylabel(\"Tỉ lệ vật nổ đã thu hồi (%)\")\n    ax.set_title(\"Hiệu quả xếp thứ tự ưu tiên rà phá\", color=INK, fontsize=11.5)\n    ax.set_xlim(0, 100)\n    ax.set_ylim(0, 100)\n    ax.legend(frameon=False, loc=\"lower right\", fontsize=8.5)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_risk_map(pipeline, path: Path) -> None:\n    \"\"\"Bản đồ nguy cơ theo ô lưới, kèm vị trí tai nạn đã ghi nhận.\"\"\"\n    plt = _setup()\n    prob = pipeline.probability.reshape(pipeline.grid.shape)\n\n    fig, ax = plt.subplots(figsize=(7.2, 5.6))\n    im = ax.imshow(prob, origin=\"lower\", cmap=\"YlOrRd\", vmin=0.0,\n                   vmax=float(np.quantile(prob, 0.995)))\n    if pipeline.accidents.n:\n        ax.scatter(\n            pipeline.accidents.col,\n            pipeline.accidents.row,\n            s=16,\n            facecolor=\"none\",\n            edgecolor=\"#1A1A1A\",\n            linewidth=0.9,\n            label=\"Tai nạn đã ghi nhận\",\n        )\n        ax.legend(frameon=False, loc=\"upper right\", fontsize=8.5)\n\n    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)\n    cb.set_label(\"Xác suất còn tồn tại vật nổ\", color=INK)\n    ax.set_title(\n        \"Bản đồ nguy cơ và vị trí tai nạn thực tế\", color=INK, fontsize=11.5\n    )\n    ax.set_xlabel(\"Ô lưới theo hướng đông\")\n    ax.set_ylabel(\"Ô lưới theo hướng bắc\")\n    ax.grid(False)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_sources(pipeline, path: Path) -> None:\n    \"\"\"Ba lớp dữ liệu đặt cạnh nhau: hồ sơ, hố bom, vật nổ thật.\"\"\"\n    plt = _setup()\n    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9))\n\n    layers = [\n        (pipeline.tonnage_spread, \"Hồ sơ không kích\\n(đã lan toả theo sai số)\", \"Blues\"),\n        (pipeline.crater_map, \"Hố bom phát hiện\\ntrên ảnh vệ tinh\", \"Greens\"),\n        (pipeline.scene.uxo_count, \"Vật nổ còn sót\\n(nhãn đối chứng mô phỏng)\", \"Reds\"),\n    ]\n    for ax, (layer, title, cmap) in zip(axes, layers):\n        vmax = float(np.quantile(layer, 0.995)) or 1.0\n        ax.imshow(layer, origin=\"lower\", cmap=cmap, vmin=0, vmax=vmax)\n        ax.set_title(title, color=INK, fontsize=9.5)\n        ax.set_xticks([])\n        ax.set_yticks([])\n        ax.grid(False)\n\n    fig.suptitle(\n        \"Ba nguồn dữ liệu độc lập trên cùng một vùng nghiên cứu\",\n        color=INK,\n        fontsize=11.5,\n    )\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_calibration(pipeline, path: Path) -> None:\n    \"\"\"Biểu đồ tin cậy.\"\"\"\n    from ..evaluation.calibration import reliability_curve\n\n    plt = _setup()\n    pred, true, counts = reliability_curve(\n        pipeline.probability[pipeline.is_cleared],\n        pipeline.y_observed[pipeline.is_cleared],\n        pipeline.cfg.risk.n_calibration_bins,\n    )\n    valid = counts > 0\n\n    fig, ax = plt.subplots(figsize=(5.4, 4.6))\n    ax.plot([0, 1], [0, 1], linestyle=\"--\", color=BLUSH, linewidth=1.4,\n            label=\"Hiệu chỉnh hoàn hảo\")\n    ax.plot(pred[valid], true[valid], marker=\"o\", color=INK, linewidth=1.8,\n            markersize=5, label=\"Mô hình\")\n    ax.set_xlabel(\"Xác suất mô hình dự báo\")\n    ax.set_ylabel(\"Tần suất thực tế\")\n    ax.set_title(\"Biểu đồ tin cậy\", color=INK, fontsize=11.5)\n    ax.legend(frameon=False, fontsize=8.5)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_ablation(pipeline, path: Path) -> None:\n    \"\"\"So sánh bốn cấu hình của phân tích đóng góp thành phần.\"\"\"\n    plt = _setup()\n    rows = pipeline.results.get(\"phan_tich_dong_gop_thanh_phan\", [])\n    if not rows:\n        return\n\n    names = [r[\"cau_hinh\"].split(\"—\")[0].strip() for r in rows]\n    recovered = [r[\"thu_hoi_tai_20pct\"] * 100 for r in rows]\n    coverage = [r[\"bao_phu_tai_nan_tai_20pct\"] * 100 for r in rows]\n\n    x = np.arange(len(names))\n    width = 0.36\n\n    fig, ax = plt.subplots(figsize=(6.6, 4.2))\n    ax.bar(x - width / 2, recovered, width, color=BLUE, label=\"Thu hồi vật nổ tại 20% diện tích\")\n    ax.bar(x + width / 2, coverage, width, color=MINT, label=\"Bao phủ tai nạn tại 20% diện tích\")\n    ax.axhline(20, color=BLUSH, linestyle=\"--\", linewidth=1.2)\n    ax.text(len(names) - 0.55, 21.5, \"mức quét trải đều\", color=BLUSH, fontsize=8)\n\n    ax.set_xticks(x)\n    ax.set_xticklabels(names)\n    ax.set_ylabel(\"Phần trăm\")\n    ax.set_title(\"Đóng góp của từng nguồn dữ liệu\", color=INK, fontsize=11.5)\n    ax.legend(frameon=False, fontsize=8.5)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_importance(pipeline, path: Path) -> None:\n    \"\"\"Mức đóng góp của từng đặc trưng theo phép hoán vị.\"\"\"\n    plt = _setup()\n    imp = pipeline.results.get(\"muc_dong_gop_theo_hoan_vi\", {})\n    if not imp:\n        return\n\n    items = list(imp.items())[:10][::-1]\n    names = [k for k, _ in items]\n    values = [v for _, v in items]\n\n    fig, ax = plt.subplots(figsize=(6.6, 4.4))\n    ax.barh(names, values, color=BLUE)\n    ax.set_xlabel(\"Mức sụt giảm hiệu quả ưu tiên khi xáo trộn đặc trưng\")\n    ax.set_title(\"Đóng góp của từng đặc trưng\", color=INK, fontsize=11.5)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef fig_sample_tiles(pipeline, path: Path) -> None:\n    \"\"\"Vài ảnh vệ tinh lịch sử kèm nhãn hố bom.\"\"\"\n    plt = _setup()\n    tiles = sorted(pipeline.tiles, key=lambda t: -t.boxes.shape[0])[:4]\n    if not tiles:\n        return\n\n    fig, axes = plt.subplots(1, len(tiles), figsize=(3.1 * len(tiles), 3.4))\n    if len(tiles) == 1:\n        axes = [axes]\n    for ax, tile in zip(axes, tiles):\n        ax.imshow(tile.image, cmap=\"gray\")\n        for b in tile.boxes:\n            ax.add_patch(\n                plt.Rectangle(\n                    (b[0], b[1]),\n                    b[2] - b[0],\n                    b[3] - b[1],\n                    fill=False,\n                    edgecolor=\"#FFD166\",\n                    linewidth=1.0,\n                )\n            )\n        ax.set_title(f\"{tile.boxes.shape[0]} hố bom\", fontsize=9)\n        ax.set_xticks([])\n        ax.set_yticks([])\n        ax.grid(False)\n\n    fig.suptitle(\"Ảnh vệ tinh lịch sử mô phỏng và nhãn hố bom\", color=INK, fontsize=11)\n    fig.tight_layout()\n    fig.savefig(path)\n    plt.close(fig)\n\n\ndef make_all_figures(pipeline, out_dir: Path) -> List[Path]:\n    \"\"\"Sinh toàn bộ hình minh hoạ, bỏ qua hình nào lỗi mà không chặn quy trình.\"\"\"\n    jobs = [\n        (\"01_duong_cong_hieu_qua_ra_pha.png\", fig_clearance_curve),\n        (\"02_ban_do_nguy_co.png\", fig_risk_map),\n        (\"03_ba_nguon_du_lieu.png\", fig_sources),\n        (\"04_bieu_do_tin_cay.png\", fig_calibration),\n        (\"05_dong_gop_thanh_phan.png\", fig_ablation),\n        (\"06_dong_gop_dac_trung.png\", fig_importance),\n        (\"07_anh_ve_tinh_mau.png\", fig_sample_tiles),\n    ]\n    made: List[Path] = []\n    for name, fn in jobs:\n        path = Path(out_dir) / name\n        try:\n            fn(pipeline, path)\n            if path.exists():\n                made.append(path)\n        except Exception as exc:  # pragma: no cover\n            logger.warning(\"Không dựng được hình %s: %s\", name, exc)\n    logger.info(\"  Đã dựng %d hình minh hoạ\", len(made))\n    return made\n", "src/demine/viz/maps.py": "\"\"\"Bản đồ ưu tiên rà phá, kết xuất thành một tệp web tự chứa.\n\nBản đồ được dựng bằng Folium khi có sẵn, và lùi về một trang web tự vẽ bằng thẻ\ncanvas khi không có — để bước này không bao giờ làm gãy quy trình vì thiếu gói.\n\nNguyên tắc an toàn được in cố định ngay trên giao diện, không thể tắt: hệ thống\nxếp thứ tự ưu tiên, không xác nhận an toàn. Bảng màu cũng được chọn theo nguyên\ntắc đó — không có màu xanh lá cho vùng nguy cơ thấp, vì xanh lá đọc thành an toàn.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .. import SAFETY_NOTICE\nfrom ..utils import get_logger\n\nlogger = get_logger(__name__)\n\n# Thang màu từ vàng nhạt tới đỏ sẫm. Cố ý không dùng xanh lá ở đầu thang.\nCOLORS = [\"#FBF3E3\", \"#FBE2B4\", \"#F6BE7E\", \"#E8895C\", \"#C9452F\", \"#8E1B12\"]\n\n\ndef _top_cells(pipeline, n_top: int = 400):\n    priority = pipeline.priority\n    order = np.argsort(-priority)[:n_top]\n    lon, lat = pipeline.grid.cell_centers_lonlat()\n    return order, lon, lat\n\n\ndef _color_for(value: float, breaks: np.ndarray) -> str:\n    idx = int(np.searchsorted(breaks, value, side=\"right\"))\n    return COLORS[min(idx, len(COLORS) - 1)]\n\n\ndef make_priority_map(pipeline, path: Path, n_top: int = 400) -> Path:\n    path = Path(path)\n    order, lon, lat = _top_cells(pipeline, n_top)\n    priority = pipeline.priority\n    probability = pipeline.probability\n    breaks = np.quantile(priority[order], [0.2, 0.4, 0.6, 0.8, 0.95])\n\n    try:\n        import folium\n\n        centre = [float(np.mean(lat)), float(np.mean(lon))]\n        fmap = folium.Map(location=centre, zoom_start=12, tiles=\"CartoDB positron\")\n\n        half = pipeline.grid.cell / 111_320.0 / 2.0\n        for i in order:\n            color = _color_for(priority[i], breaks)\n            folium.Rectangle(\n                bounds=[\n                    [lat[i] - half, lon[i] - half],\n                    [lat[i] + half, lon[i] + half],\n                ],\n                color=color,\n                weight=0.4,\n                fill=True,\n                fill_color=color,\n                fill_opacity=0.72,\n                popup=folium.Popup(\n                    f\"<b>Ô lưới {int(i)}</b><br>\"\n                    f\"Xác suất còn vật nổ: {probability[i]:.3f}<br>\"\n                    f\"Chỉ số ưu tiên: {priority[i]:.3f}<br>\"\n                    f\"<i>Chưa được rà phá — không phải xác nhận an toàn.</i>\",\n                    max_width=280,\n                ),\n            ).add_to(fmap)\n\n        for c, r in zip(pipeline.accidents.col, pipeline.accidents.row):\n            idx = int(r) * pipeline.grid.cfg.n_cells_x + int(c)\n            folium.CircleMarker(\n                location=[lat[idx], lon[idx]],\n                radius=3.2,\n                color=\"#1A1A1A\",\n                weight=1.2,\n                fill=False,\n                popup=\"Vị trí tai nạn đã ghi nhận\",\n            ).add_to(fmap)\n\n        banner = f\"\"\"\n        <div style=\"position: fixed; bottom: 18px; left: 18px; z-index: 9999;\n                    background: #FAEAE6; border-left: 5px solid #C9705C;\n                    padding: 10px 14px; max-width: 430px; border-radius: 4px;\n                    font-family: Georgia, serif; font-size: 12.5px; color: #1C3557;\">\n          <b>Nguyên tắc an toàn bắt buộc.</b><br>{SAFETY_NOTICE}\n        </div>\n        \"\"\"\n        fmap.get_root().html.add_child(folium.Element(banner))\n        fmap.save(str(path))\n        logger.info(\"  Đã dựng bản đồ ưu tiên bằng Folium: %s\", path.name)\n        return path\n\n    except Exception as exc:\n        logger.warning(\"Không dùng được Folium (%s), chuyển sang bản đồ tự vẽ.\", exc)\n\n    cells = [\n        {\n            \"x\": int(i % pipeline.grid.cfg.n_cells_x),\n            \"y\": int(i // pipeline.grid.cfg.n_cells_x),\n            \"p\": round(float(probability[i]), 4),\n            \"u\": round(float(priority[i]), 4),\n            \"c\": _color_for(priority[i], breaks),\n        }\n        for i in order\n    ]\n    payload = json.dumps(\n        {\n            \"cells\": cells,\n            \"nx\": pipeline.grid.cfg.n_cells_x,\n            \"ny\": pipeline.grid.cfg.n_cells_y,\n            \"accidents\": [\n                {\"x\": int(c), \"y\": int(r)}\n                for c, r in zip(pipeline.accidents.col, pipeline.accidents.row)\n            ],\n        },\n        ensure_ascii=False,\n    )\n\n    html = f\"\"\"<!doctype html>\n<html lang=\"vi\"><head><meta charset=\"utf-8\">\n<title>DeMine-VN — Bản đồ ưu tiên rà phá</title>\n<style>\n body {{ margin:0; font-family: Georgia, serif; background:#FBF9F5; color:#1C3557; }}\n header {{ padding:16px 22px; border-bottom:1px solid #9DBCDA; }}\n h1 {{ margin:0; font-size:19px; }}\n p.sub {{ margin:6px 0 0; font-size:13px; color:#3F576F; }}\n #wrap {{ padding:18px 22px; }}\n canvas {{ border:1px solid #9DBCDA; background:#fff; max-width:100%; }}\n .notice {{ margin-top:16px; background:#FAEAE6; border-left:5px solid #C9705C;\n            padding:12px 16px; font-size:13px; border-radius:4px; max-width:760px; }}\n .legend {{ margin-top:12px; font-size:12.5px; color:#3F576F; }}\n .sw {{ display:inline-block; width:22px; height:11px; margin:0 4px 0 12px;\n        vertical-align:middle; border:1px solid #ccc; }}\n</style></head><body>\n<header>\n  <h1>DeMine-VN — Bản đồ ưu tiên rà phá</h1>\n  <p class=\"sub\">Ô càng sẫm màu, thứ tự ưu tiên rà phá càng cao. Vòng tròn đen là vị trí tai nạn đã ghi nhận.</p>\n</header>\n<div id=\"wrap\">\n  <canvas id=\"cv\" width=\"1100\" height=\"760\"></canvas>\n  <div class=\"legend\">Mức ưu tiên:\n    <span class=\"sw\" style=\"background:{COLORS[0]}\"></span>thấp\n    <span class=\"sw\" style=\"background:{COLORS[2]}\"></span>trung bình\n    <span class=\"sw\" style=\"background:{COLORS[5]}\"></span>cao nhất\n  </div>\n  <div class=\"notice\"><b>Nguyên tắc an toàn bắt buộc.</b> {SAFETY_NOTICE}</div>\n</div>\n<script>\nconst D = {payload};\nconst cv = document.getElementById('cv'), ctx = cv.getContext('2d');\nconst sx = cv.width / D.nx, sy = cv.height / D.ny;\nfor (const c of D.cells) {{\n  ctx.fillStyle = c.c;\n  ctx.fillRect(c.x*sx, cv.height - (c.y+1)*sy, Math.max(sx,1.5), Math.max(sy,1.5));\n}}\nctx.strokeStyle = '#1A1A1A'; ctx.lineWidth = 1.1;\nfor (const a of D.accidents) {{\n  ctx.beginPath();\n  ctx.arc(a.x*sx + sx/2, cv.height - (a.y+0.5)*sy, 3.2, 0, 6.2832);\n  ctx.stroke();\n}}\n</script></body></html>\n\"\"\"\n    path.write_text(html, encoding=\"utf-8\")\n    logger.info(\"  Đã dựng bản đồ ưu tiên tự vẽ: %s\", path.name)\n    return path\n", "scripts/train_detector.py": "#!/usr/bin/env python\n\"\"\"Huấn luyện riêng tầng phát hiện hố bom, trong một tiến trình độc lập.\n\nLý do tách riêng: khi huấn luyện phân tán trên hai GPU, Ultralytics khởi tạo nhóm\ntiến trình ở phía sau. Việc này không ổn định nếu gọi thẳng trong nhân của\nnotebook — nhân có thể treo hoặc giữ lại tiến trình con sau khi chạy xong. Gọi qua\nmột tiến trình độc lập thì mọi tài nguyên được giải phóng sạch khi kết thúc.\n\nVí dụ::\n\n    python scripts/train_detector.py --data data/imagery/data.yaml --epochs 40\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / \"src\"))\n\nfrom demine.config import RunConfig  # noqa: E402\nfrom demine.detect import build_detector  # noqa: E402\nfrom demine.utils import gpu_utilisation, setup_logging  # noqa: E402\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=\"Huấn luyện tầng phát hiện hố bom.\")\n    ap.add_argument(\"--data\", required=True, help=\"Đường dẫn tới data.yaml\")\n    ap.add_argument(\"--epochs\", type=int, default=40)\n    ap.add_argument(\"--batch\", type=int, default=16)\n    ap.add_argument(\"--imgsz\", type=int, default=640)\n    ap.add_argument(\"--output-dir\", default=\"outputs\")\n    ap.add_argument(\n        \"--backend\", choices=[\"auto\", \"ultralytics\", \"torchvision\"], default=\"auto\"\n    )\n    ap.add_argument(\n        \"--devices\",\n        default=\"0,1\",\n        help=\"Danh sách GPU, phân tách bằng dấu phẩy. Để trống nghĩa là chạy trên CPU.\",\n    )\n    args = ap.parse_args()\n\n    setup_logging()\n\n    cfg = RunConfig()\n    cfg.detect.epochs = args.epochs\n    cfg.detect.batch_size = args.batch\n    cfg.detect.image_size = args.imgsz\n    cfg.detect.backend = args.backend\n    cfg.output_dir = args.output_dir\n    cfg.detect.devices = (\n        [int(d) for d in args.devices.split(\",\") if d.strip() != \"\"]\n        if args.devices\n        else []\n    )\n    cfg.imagery.tile_size_px = args.imgsz\n\n    gpu = gpu_utilisation()\n    if gpu:\n        print(\"Tình trạng GPU trước khi huấn luyện:\")\n        print(gpu)\n\n    detector = build_detector(cfg)\n    info = detector.train(Path(args.data), cfg)\n\n    print()\n    print(\"Đã huấn luyện xong.\")\n    for k, v in info.items():\n        print(f\"  {k}: {v}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/run_pipeline.py": "#!/usr/bin/env python\n\"\"\"Chạy toàn bộ quy trình DeMine-VN từ dòng lệnh.\n\nVí dụ::\n\n    python scripts/run_pipeline.py                     # chạy đầy đủ\n    python scripts/run_pipeline.py --quick             # chế độ rút gọn\n    python scripts/run_pipeline.py --no-train-detector # dùng lại trọng số đã có\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / \"src\"))\n\nfrom demine.config import RunConfig  # noqa: E402\nfrom demine.pipeline import Pipeline  # noqa: E402\nfrom demine.utils import gpu_utilisation, setup_logging  # noqa: E402\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=\"Chạy quy trình DeMine-VN.\")\n    ap.add_argument(\"--quick\", action=\"store_true\", help=\"Chế độ rút gọn, chạy nhanh\")\n    ap.add_argument(\"--epochs\", type=int, default=None, help=\"Số chu kỳ huấn luyện\")\n    ap.add_argument(\"--tiles\", type=int, default=None, help=\"Số ảnh con\")\n    ap.add_argument(\"--missions\", type=int, default=None, help=\"Số phi vụ mô phỏng\")\n    ap.add_argument(\"--seed\", type=int, default=None, help=\"Hạt giống ngẫu nhiên\")\n    ap.add_argument(\"--output-dir\", default=\"outputs\")\n    ap.add_argument(\"--data-dir\", default=\"data\")\n    ap.add_argument(\n        \"--backend\",\n        choices=[\"auto\", \"ultralytics\", \"torchvision\"],\n        default=\"auto\",\n        help=\"Phương án phát hiện\",\n    )\n    ap.add_argument(\n        \"--no-train-detector\",\n        action=\"store_true\",\n        help=\"Bỏ qua huấn luyện, dùng trọng số đã có\",\n    )\n    args = ap.parse_args()\n\n    setup_logging()\n\n    cfg = RunConfig()\n    if args.quick:\n        cfg.apply_quick_mode()\n    if args.epochs is not None:\n        cfg.detect.epochs = args.epochs\n    if args.tiles is not None:\n        cfg.imagery.n_tiles = args.tiles\n    if args.missions is not None:\n        cfg.sortie.n_missions = args.missions\n    if args.seed is not None:\n        cfg.seed = args.seed\n        cfg.detect.seed = args.seed\n        cfg.risk.seed = args.seed\n    cfg.detect.backend = args.backend\n    cfg.output_dir = args.output_dir\n    cfg.data_dir = args.data_dir\n\n    gpu = gpu_utilisation()\n    if gpu:\n        print(\"Tình trạng GPU:\")\n        print(gpu)\n\n    Pipeline(cfg).run(train_detector=not args.no_train_detector)\n\n    out = Path(cfg.output_dir)\n    print()\n    print(\"=\" * 62)\n    print(\"HOÀN TẤT. Các tệp kết quả:\")\n    print(f\"  {out / 'bao_cao_tong_hop.md'}\")\n    print(f\"  {out / 'ket_qua.json'}\")\n    print(f\"  {out / 'ban_do_uu_tien.html'}\")\n    print(f\"  {out / 'danh_muc_uu_tien_ra_pha.csv'}\")\n    print(f\"  {out / 'figures'}/\")\n    print(\"=\" * 62)\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"}''')

BASE = pathlib.Path('/kaggle/working/demine-vn')
if not str(BASE).startswith('/kaggle'):
    BASE = pathlib.Path.cwd() / 'demine-vn'

for rel, content in _FILES.items():
    target = BASE / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')

sys.path.insert(0, str(BASE / 'src'))
os.chdir(BASE)
print(f'Đã ghi {len(_FILES)} tệp mã nguồn vào {BASE}')
print('Thư mục làm việc hiện tại:', os.getcwd())


## Bước 1 — Kiểm tra môi trường và GPU

In [ ]:
import subprocess, sys
print('Python', sys.version.split()[0])
try:
    import torch
    print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}:', torch.cuda.get_device_name(i))
except Exception as e:
    print('Không nạp được PyTorch:', e)

try:
    print(subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total',
                          '--format=csv,noheader'],
                         capture_output=True, text=True, timeout=20).stdout)
except Exception:
    print('Không gọi được nvidia-smi — có thể đang chạy trên CPU.')


## Bước 2 — Cài các gói còn thiếu

Chỉ cài những gói chưa có. Nếu Internet tắt, ô này bỏ qua và hệ thống dùng phương án dự phòng.

In [ ]:
import importlib.util, subprocess, sys

def have(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False

for pkg, mod in [('ultralytics','ultralytics'), ('xgboost','xgboost'),
                 ('folium','folium')]:
    if have(mod):
        print(f'{pkg}: đã có')
        continue
    print(f'{pkg}: đang cài…')
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg],
                       capture_output=True, text=True)
    print(f'{pkg}:', 'xong' if r.returncode == 0 else 'không cài được, dùng phương án dự phòng')


## Bước 3 — Dựng vùng nghiên cứu

Địa hình, thổ nhưỡng, sông ngòi, đường sá và khu dân cư. Các lớp này chi phối cả cơ chế vật lý (nền đất mềm làm tăng tỉ lệ bom không nổ) lẫn mức độ phơi nhiễm của con người.

In [ ]:
import sys, numpy as np
from demine.config import RunConfig
from demine.utils import setup_logging
from demine.pipeline import Pipeline

setup_logging()

cfg = RunConfig()
# Đặt QUICK = True để chạy thử nhanh trong khoảng 6 đến 10 phút.
QUICK = False
if QUICK:
    cfg.apply_quick_mode()

pipe = Pipeline(cfg)
pipe.step_1_build_area()


## Bước 4 — Mô phỏng phi vụ không kích và vật nổ còn sót

Chuỗi nhân quả: hồ sơ ghi chép → điểm rơi thật → nổ hoặc không nổ → hố bom hoặc vật nổ còn sót. Điểm ngắm bám theo tuyến giao thông và sông ngòi, đúng như không kích thực tế, nên ô nhiễm tập trung thành hành lang.

In [ ]:
pipe.step_2_simulate()


## Bước 5 — Kết xuất ảnh vệ tinh trinh sát lịch sử

Ảnh đơn sắc có hạt phim, chiếu sáng không đều và tương phản thấp, đúng đặc điểm của ảnh phim trinh sát quét lại. Hố bom được dựng theo hình thái thật: lòng tối, viền sáng.

In [ ]:
pipe.step_3_imagery()


In [ ]:
import matplotlib.pyplot as plt

tiles = sorted(pipe.tiles, key=lambda t: -t.boxes.shape[0])[:3]
fig, axes = plt.subplots(1, len(tiles), figsize=(4.2*len(tiles), 4.4))
if len(tiles) == 1:
    axes = [axes]
for ax, tile in zip(axes, tiles):
    ax.imshow(tile.image, cmap='gray')
    for b in tile.boxes:
        ax.add_patch(plt.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                   fill=False, edgecolor='#FFD166', linewidth=1.1))
    ax.set_title(f'{tile.boxes.shape[0]} hố bom')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Ảnh vệ tinh lịch sử mô phỏng và nhãn hố bom')
plt.tight_layout(); plt.show()


## Bước 6 — Huấn luyện tầng phát hiện hố bom

Huấn luyện được gọi qua một tiến trình con độc lập. Lý do: khi chạy phân tán trên hai GPU, Ultralytics khởi tạo nhóm tiến trình ở phía sau, và việc này không ổn định nếu gọi thẳng trong nhân của notebook.

Nếu không có Ultralytics, hệ thống tự chuyển sang Faster R-CNN của Torchvision với bộ sinh neo được thiết kế lại cho đối tượng nhỏ.

In [ ]:
import subprocess, sys
from demine.detect import select_backend

backend = select_backend(cfg.detect.backend)
print('Phương án phát hiện:', backend)

cmd = [sys.executable, 'scripts/train_detector.py',
       '--data', str(pipe.data_yaml),
       '--epochs', str(cfg.detect.epochs),
       '--batch', str(cfg.detect.batch_size),
       '--imgsz', str(cfg.detect.image_size),
       '--backend', backend,
       '--devices', '0,1']
print(' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print('--- lỗi ---')
    print(proc.stderr[-4000:])


## Bước 7 — Đánh giá tầng phát hiện và quy hố bom về lưới

Chỉ tiêu mAP được cài đặt trực tiếp theo quy ước PASCAL VOC, không gọi thư viện ngoài, để đội thi giải thích được từng bước của con số mình công bố.

In [ ]:
from pathlib import Path
from demine.detect import build_detector

detector = build_detector(cfg)
weights = None
for cand in [Path(cfg.output_dir)/'runs'/'yolo_crater'/'weights'/'best.pt',
             Path(cfg.output_dir)/'runs'/'frcnn_crater'/'weights'/'best.pt']:
    if cand.exists():
        weights = cand
        break

if weights is not None:
    print('Nạp trọng số:', weights)
    detector.load(weights)
    pipe.step_4_detect(detector=detector, train=False)
else:
    print('Không tìm thấy trọng số — huấn luyện lại ngay trong nhân.')
    pipe.step_4_detect(detector=detector, train=True)


## Bước 8 — Dựng đặc trưng và mô phỏng dữ liệu rà phá

Hai đặc trưng đáng chú ý nhất là nơi việc hợp nhất tạo ra giá trị thật:

- **kỳ vọng bom không nổ** — tải trọng bom nhân với tỉ lệ không nổ tăng theo độ mềm của nền đất;
- **hố bom đã hiệu chỉnh tầm nhìn** — chia mật độ hố bom cho xác suất còn quan sát được, để khử thiên lệch ở nơi hố bom bị bồi lấp nhanh.

Phần mô phỏng rà phá tái hiện đúng cơ chế **chọn mẫu có chủ đích** của cơ quan chuyên môn, để tầng ba có cái mà hiệu chỉnh.

In [ ]:
pipe.step_5_features()
pipe.step_6_clearance()


## Bước 9 — Huấn luyện mô hình nguy cơ

Hai biện pháp hiệu chỉnh chạy song song: trọng số nghịch đảo xác suất được chọn đưa vào rà phá, và khung học từ dữ liệu chỉ có quan sát dương. Cả hai đều làm ước lượng thận trọng hơn — nghiêng về phía cho rằng còn nhiều vật nổ hơn những gì đã quan sát. Đó là chiều nghiêng đúng cho bài toán này.

In [ ]:
pipe.step_7_risk_model()


## Bước 10 — Kiểm chứng bốn tầng

Đây là phần trọng tâm của đề tài. Chi tiết phương pháp ở `docs/VALIDATION.md`.

1. **Đất đã rà phá** — chia tập theo khối không gian, đo bằng đường cong hiệu quả rà phá.
2. **Hồ sơ tai nạn** — tập kiểm chứng độc lập, có thêm phép chia theo thời gian.
3. **Nhất quán hai nguồn** — hồ sơ không kích và ảnh vệ tinh phải khớp nhau.
4. **Chuyển vùng và hiệu chỉnh** — huấn luyện nửa tây, áp dụng nửa đông.

In [ ]:
pipe.step_8_validate()


## Bước 11 — Phân tích đóng góp thành phần

Bốn cấu hình: chỉ hồ sơ không kích; chỉ hố bom; hợp nhất hai nguồn; hệ thống đầy đủ. Mục đích là trả lời câu hỏi từng nguồn đóng góp bao nhiêu, và có nguồn nào thừa không.

In [ ]:
pipe.step_9_ablation()


## Bước 12 — Bản đồ, hình minh hoạ và báo cáo

In [ ]:
pipe.step_10_outputs()


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

report = Path(cfg.output_dir) / 'bao_cao_tong_hop.md'
display(Markdown(report.read_text(encoding='utf-8')))


### Hình minh hoạ

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for p in sorted((Path(cfg.output_dir) / 'figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))


### Bản đồ ưu tiên rà phá

In [ ]:
from IPython.display import IFrame, display
from pathlib import Path

m = Path(cfg.output_dir) / 'ban_do_uu_tien.html'
print('Tệp bản đồ:', m)
display(IFrame(src=str(m), width='100%', height=640))


### Danh mục ưu tiên rà phá

Đây là sản phẩm mà đơn vị lập kế hoạch dùng trực tiếp. Mỗi dòng có toạ độ, xác suất đã hiệu chỉnh, chỉ số ưu tiên và các yếu tố giải thích.

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path(cfg.output_dir) / 'danh_muc_uu_tien_ra_pha.csv')
print(f'Tổng {len(df)} khoảnh. Hai mươi khoảnh ưu tiên cao nhất:')
display(df.head(20))


---

## Hoàn tất

Các tệp kết quả nằm trong thư mục `outputs/`:

- `bao_cao_tong_hop.md` — báo cáo đầy đủ
- `ket_qua.json` — toàn bộ chỉ tiêu ở dạng máy đọc được
- `ban_do_uu_tien.html` — bản đồ tương tác
- `danh_muc_uu_tien_ra_pha.csv` — danh mục cho đơn vị lập kế hoạch
- `figures/` — hình minh hoạ

> **Nguyên tắc an toàn bắt buộc.** Hệ thống chỉ xếp thứ tự ưu tiên rà phá. Hệ thống **không bao giờ** tuyên bố một khu đất là an toàn. Mọi khu đất vẫn phải được rà phá đầy đủ theo quy trình kỹ thuật hiện hành trước khi đưa vào sử dụng.